# Loading libraries and dataset

In [1]:
#Load libraries
import os
import datetime

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt

import pandas as pd
#import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
#import seaborn as sns
#import folium
!pip install tensorflow
import tensorflow as tf
import sklearn as sk
import sklearn.preprocessing as skprep
from sklearn.model_selection import train_test_split
from collections import Counter
!pip install spektral==1.0.3 #For GCN https://github.com/danielegrattarola/spektral/issues/453
import spektral
!pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 28.7 MB/s eta 0:00:00


In [2]:
#Connect drive to Collab

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os
# List files in the directory
file_list = os.listdir('/content/drive/MyDrive/PhD data analysis/Paper1/')
print(file_list)

['Fire_NIS.csv', 'Water_NIS.csv', 'Fire_TAUW.csv', 'Water_TAUW.csv', 'Raw datasets', 'preprocessed_df.csv', 'LSTM.ipynb', '.ipynb_checkpoints', 'x3.csv', 'x4.csv', 'x5.csv', 'df_pred.csv', 'Copy of LSTM_2.ipynb', 'spatial_dist_30000.csv', 'spatial_dist_10000.csv', 'Models', 'GCNLSTM_1616.csv', 'spatial_dist_suburban.csv', 'spatial_dist_ams_ut.csv', 'spatial_dist_dh_rot.csv', 'spatial_dist_outlier.csv', 'spatial_dist_limburg.csv', 'test_gcn5.csv', 'test_gcn4.csv', 'test_gcn3.csv', 'test_gcn2.csv', 'gcnblstm_training_log3.csv', 'test3_gcnblstm.csv', 'gcnblstm_training_log4.csv', 'test4_gcnblstm.csv', 'gcnblstm_training_log5.csv', 'gat_training_log1.csv', 'test1_gat.csv', 'test3_gat.csv', 'gat_training_log3.csv', 'gat_training_log4.csv', 'test2_gat.csv', 'test4_gat.csv', 'gat_training_log5.csv', 'test5_gat.csv', 'gcnblstm_training_log2.csv', 'test2_gcnblstm.csv', 'gcnblstm_training_log1.csv', 'test1_gcnblstm.csv', 'test5_gcnblstm.csv', 'test5_gcnblstm1.csv', 'blstm_training_log1.csv', 'te

In [4]:
#Load dataset

df = pd.read_csv("/content/drive/MyDrive/PhD data analysis/Paper1/preprocessed_df.csv")  #Preprocessing data - left at 'Overig' incident type to redo the labels
df.head()

,OBJECTID,dataset,start_date,start_time,end_date,end_time,incident_duration,incident_type,incident_description,traffic_jam,...,sum_prec (1h),max_prec (5min),max_temp,round_max_temp,low_veg_%,mid_high_veg_%,misc_%,pavement_%,water_%,total_acreage_%
0,1,UDLS,1/21/2014,12:46:46,1/21/2014,13:44:12,00:57:25,Overig,modder op oprit,NaN,...,0.0,0.0,5.753218,6,35.0,NaN,1.0,33.0,3.0,72
1,2,UDLS,1/27/2014,11:52:00,1/27/2014,12:48:00,00:56:00,Wateroverlast,wateroverlast geldrop,NaN,...,0.0,0.0,5.988402,6,40.0,4.0,6.0,34.0,1.0,85
2,3,UDLS,2-6-2014,08:35:41,2-7-2014,15:38:36,07:02:54,Schade aan infra,uitspoeling in de mb,NaN,...,0.0,0.0,9.551981,10,64.0,NaN,0.0,30.0,2.0,96
3,4,UDLS,3-5-2014,00:56:27,3-5-2014,04:07:14,03:10:47,Zichtbaarheid,mist everdingen - deil,NaN,...,0.0,0.0,12.243471,12,41.0,17.0,1.0,26.0,7.0,92
4,5,UDLS,3/14/2014,07:00:14,3/14/2014,12:10:04,05:09:50,Zichtbaarheid,gesloten sps i.v.m. mist,NaN,...,0.0,0.0,11.787109,12,61.0,9.0,NaN,24.0,6.0,100


In [5]:
#Preprocessing step 1 (Handling NAs)

#Check missing values
print('Columns with missing values:', df.isnull().any())
print('Observations of original df:', len(df))

#Remove observations with a missing end time/end date
df_1 = df.dropna(subset=['end_date', 'end_time', 'incident_type'])

print('--------------------------')

#Check missing values
print('After removal:', df_1.isnull().any())
print('Observations of cleaned df:', len(df_1))

#Replacing all NAs in traffic and areal variables with 0
df_1 = df_1.fillna(0)

print('--------------------------')

#Check missing values again
print('After filling NAs with 0:', df_1.isnull().any())

Columns with missing values: OBJECTID                False
dataset                 False
start_date              False
start_time              False
end_date                 True
end_time                 True
incident_duration       False
incident_type            True
incident_description    False
traffic_jam              True
road_level              False
lon                     False
lat                     False
sum_prec (1h)           False
max_prec (5min)         False
max_temp                False
round_max_temp          False
low_veg_%                True
mid_high_veg_%           True
misc_%                   True
pavement_%               True
water_%                  True
total_acreage_%         False
dtype: bool
Observations of original df: 7591
--------------------------
After removal: OBJECTID                False
dataset                 False
start_date              False
start_time              False
end_date                False
end_time                False
incident_dura

In [6]:
df_1['incident_type'].unique()

array(['Overig', 'Wateroverlast', 'Schade aan infra', 'Zichtbaarheid',
       'Brand', 'Gladheid', 'Voorwerp', 'Ongeval', 'Gladheidsbestrijding',
       'Storing', 'Invalid - Gladheidsbestrijding', 'Onbekend',
       'Weergerelateerd wegprobleem'], dtype=object)

In [7]:
incidents_all = df_1['incident_type'].value_counts()
print(incidents_all)
incidents_filtered = df_1[df_1['incident_type'].isin(['Brand', 'Gladheid',
       'Schade aan infra', 'Wateroverlast','Zichtbaarheid', 'Onbekend', 'Overig'])]['incident_type'].value_counts()
print(incidents_filtered)

incident_type
Wateroverlast                     2129
Voorwerp                          2053
Brand                             1095
Schade aan infra                   769
Overig                             341
Ongeval                            332
Gladheid                           294
Gladheidsbestrijding               155
Zichtbaarheid                      104
Storing                             37
Onbekend                            30
Weergerelateerd wegprobleem         20
Invalid - Gladheidsbestrijding      18
Name: count, dtype: int64
incident_type
Wateroverlast       2129
Brand               1095
Schade aan infra     769
Overig               341
Gladheid             294
Zichtbaarheid        104
Onbekend              30
Name: count, dtype: int64


In [8]:
#Leave out some labels from incident_type

filtered_df = df_1[df_1['incident_type'].isin(['Brand', 'Gladheid',
       'Schade aan infra', 'Wateroverlast', 'Onbekend', 'Overig'])] #'Onbekend', 'Overig'
print(filtered_df)

      OBJECTID       dataset start_date start_time   end_date  end_time  \
0            1          UDLS  1/21/2014   12:46:46  1/21/2014  13:44:12   
1            2          UDLS  1/27/2014   11:52:00  1/27/2014  12:48:00   
2            3          UDLS   2-6-2014   08:35:41   2-7-2014  15:38:36   
5          830          UDLS  2/28/2018   14:54:58  2/28/2018  15:08:49   
6            7          UDLS  4/24/2014   19:50:36  4/25/2014  14:22:17   
...        ...           ...        ...        ...        ...       ...   
7567      5445          UDLS  7/24/2019   10:15:06  7/24/2019  13:47:34   
7568      5452          UDLS  7/25/2019   14:19:02  7/25/2019  16:05:04   
7569      5442          UDLS  7/25/2019   12:21:45  7/25/2019  12:32:45   
7570      5446          UDLS  7/25/2019   12:35:06  7/25/2019  12:40:32   
7571      7338  Dexter files  7/25/2019   14:26:00  7/25/2019  14:57:13   

     incident_duration     incident_type     incident_description  \
0             00:57:25        

In [9]:
#Translate incident labels from Dutch to English

#'Brand', 'Gladheid', 'Schade aan infra', 'Wateroverlast','Zichtbaarheid', 'Onbekend', 'Overig'
#Roadside fire, Slippery pavement, Infrastructure damage, Road ponding, Poor visbility, Unknown, Miscellaneous
filtered_df = filtered_df.replace({'incident_type': {'Brand': 'Roadside fire',
                                                     'Gladheid': 'Slippery pavement', #'Slippery pavement', (Reduce the amount of classes to 3)
                                                     'Schade aan infra': 'Infrastructure damage',#Infrastructure damage',
                                                     'Wateroverlast': 'Road ponding',
                                         #            'Zichtbaarheid':'Poor visbility',
                                                     'Onbekend':'Other',
                                                     'Overig':'Other'}})

In [10]:
filtered_df

,OBJECTID,dataset,start_date,start_time,end_date,end_time,incident_duration,incident_type,incident_description,traffic_jam,...,sum_prec (1h),max_prec (5min),max_temp,round_max_temp,low_veg_%,mid_high_veg_%,misc_%,pavement_%,water_%,total_acreage_%
0,1,UDLS,1/21/2014,12:46:46,1/21/2014,13:44:12,00:57:25,Other,modder op oprit,0.0,...,0.00,0.00,5.753218,6,35.0,0.0,1.0,33.0,3.0,72
1,2,UDLS,1/27/2014,11:52:00,1/27/2014,12:48:00,00:56:00,Road ponding,wateroverlast geldrop,0.0,...,0.00,0.00,5.988402,6,40.0,4.0,6.0,34.0,1.0,85
2,3,UDLS,2-6-2014,08:35:41,2-7-2014,15:38:36,07:02:54,Infrastructure damage,uitspoeling in de mb,0.0,...,0.00,0.00,9.551981,10,64.0,0.0,0.0,30.0,2.0,96
5,830,UDLS,2/28/2018,14:54:58,2/28/2018,15:08:49,00:13:51,Roadside fire,bermbrand,0.0,...,0.00,0.00,-5.431953,-5,76.0,0.0,0.0,23.0,0.0,99
6,7,UDLS,4/24/2014,19:50:36,4/25/2014,14:22:17,18:31:41,Road ponding,wateroverlast,0.0,...,1.52,0.34,20.173279,20,42.0,8.0,5.0,37.0,3.0,95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7567,5445,UDLS,7/24/2019,10:15:06,7/24/2019,13:47:34,03:32:28,Roadside fire,bermbrand,0.0,...,0.00,0.00,38.947773,39,49.0,19.0,2.0,28.0,2.0,100
7568,5452,UDLS,7/25/2019,14:19:02,7/25/2019,16:05:04,01:46:02,Roadside fire,bermbrand + va klapband,0.0,...,0.00,0.00,39.705345,40,9.0,6.0,2.0,15.0,0.0,32
7569,5442,UDLS,7/25/2019,12:21:45,7/25/2019,12:32:45,00:11:00,Roadside fire,bermbrand,0.0,...,0.00,0.00,39.760983,40,42.0,0.0,4.0,27.0,2.0,75
7570,5446,UDLS,7/25/2019,12:35:06,7/25/2019,12:40:32,00:05:26,Roadside fire,bermbrand,0.0,...,0.00,0.00,39.760983,40,45.0,1.0,5.0,28.0,2.0,81


In [11]:
filtered_df.shape

(4658, 23)

In [12]:
incidents_all = filtered_df['incident_type'].value_counts()
print(incidents_all)
print(len(filtered_df['incident_type']))

incident_type
Road ponding             2129
Roadside fire            1095
Infrastructure damage     769
Other                     371
Slippery pavement         294
Name: count, dtype: int64
4658


In [13]:
#Preprocessing step 2 - Dropping variables, date_time transformation


#Dropping unnecessary columns and all the temporal columns (Incident type to be preprocessed and added later)
df1 = filtered_df.drop(columns=['OBJECTID',
                       'dataset',
                       'incident_description',
                       'traffic_jam',
                      # 'lat',
                     #  'lon',
                       'round_max_temp',
                       'total_acreage_%'])


#Removing missing values
#print('Columns with missing values:', df.isnull().any())

#Extracting all temporal variables and converting these into datetime[ns64] data
#The temporal information is added to the train, validation, and test datasets later

df1['start_date'] = pd.to_datetime(df1['start_date'], format='mixed', dayfirst=True)
df1['end_date'] = pd.to_datetime(df1['end_date'], format='mixed', dayfirst=True)
df1['incident_duration'] = pd.to_datetime(df1['incident_duration'], format='%H:%M:%S')
df1['start_time'] = pd.to_datetime(df1['start_time'], format='%H:%M:%S')
df1['end_time'] = pd.to_datetime(df1['end_time'], format='%H:%M:%S')

#Label encoding the target variable:

#Defining the encoder
label_enc1 = skprep.LabelEncoder()
label_enc2 = skprep.LabelEncoder()
label_enc1.fit(df1['incident_type'])
label_enc2.fit(df1['road_level'])

#Transforming the target variable
df1['incident_type'] = label_enc1.transform(df1['incident_type'])
df1['road_level'] = label_enc2.transform(df1['road_level'])

df1.head()
df1.dtypes

,0
start_date,datetime64[ns]
start_time,datetime64[ns]
end_date,datetime64[ns]
end_time,datetime64[ns]
incident_duration,datetime64[ns]
incident_type,int64
road_level,int64
lon,float64
lat,float64
sum_prec (1h),float64


In [14]:
df1

,start_date,start_time,end_date,end_time,incident_duration,incident_type,road_level,lon,lat,sum_prec (1h),max_prec (5min),max_temp,low_veg_%,mid_high_veg_%,misc_%,pavement_%,water_%
0,2014-01-21,1900-01-01 12:46:46,2014-01-21,1900-01-01 13:44:12,1900-01-01 00:57:25,1,0,5.186706,51.903114,0.00,0.00,5.753218,35.0,0.0,1.0,33.0,3.0
1,2014-01-27,1900-01-01 11:52:00,2014-01-27,1900-01-01 12:48:00,1900-01-01 00:56:00,2,1,5.254379,51.829996,0.00,0.00,5.988402,40.0,4.0,6.0,34.0,1.0
2,2014-06-02,1900-01-01 08:35:41,2014-07-02,1900-01-01 15:38:36,1900-01-01 07:02:54,0,0,5.146372,52.118044,0.00,0.00,9.551981,64.0,0.0,0.0,30.0,2.0
5,2018-02-28,1900-01-01 14:54:58,2018-02-28,1900-01-01 15:08:49,1900-01-01 00:13:51,3,0,6.890518,53.170201,0.00,0.00,-5.431953,76.0,0.0,0.0,23.0,0.0
6,2014-04-24,1900-01-01 19:50:36,2014-04-25,1900-01-01 14:22:17,1900-01-01 18:31:41,2,0,5.157969,52.140820,1.52,0.34,20.173279,42.0,8.0,5.0,37.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7567,2019-07-24,1900-01-01 10:15:06,2019-07-24,1900-01-01 13:47:34,1900-01-01 03:32:28,3,0,5.606413,51.312105,0.00,0.00,38.947773,49.0,19.0,2.0,28.0,2.0
7568,2019-07-25,1900-01-01 14:19:02,2019-07-25,1900-01-01 16:05:04,1900-01-01 01:46:02,3,0,5.591747,51.416806,0.00,0.00,39.705345,9.0,6.0,2.0,15.0,0.0
7569,2019-07-25,1900-01-01 12:21:45,2019-07-25,1900-01-01 12:32:45,1900-01-01 00:11:00,3,0,4.740278,51.545056,0.00,0.00,39.760983,42.0,0.0,4.0,27.0,2.0
7570,2019-07-25,1900-01-01 12:35:06,2019-07-25,1900-01-01 12:40:32,1900-01-01 00:05:26,3,0,4.740424,51.542346,0.00,0.00,39.760983,45.0,1.0,5.0,28.0,2.0


In [15]:
#Mapping of labels

label_enc1_mapping = dict(zip(label_enc1.classes_, label_enc1.transform(label_enc1.classes_)))
print(label_enc1_mapping)
label_enc2_mapping = dict(zip(label_enc2.classes_, label_enc2.transform(label_enc2.classes_)))
print(label_enc2_mapping)

{'Infrastructure damage': np.int64(0), 'Other': np.int64(1), 'Road ponding': np.int64(2), 'Roadside fire': np.int64(3), 'Slippery pavement': np.int64(4)}
{'maaiveld': np.int64(0), 'overbrugging': np.int64(1), 'tunneldeel': np.int64(2)}


In [16]:
#Preprocessing step 3 - Converting latitude and longitude as inputs

lat = df1['lat']
lon = df1['lon']

lat_radians = np.deg2rad(lat)
lon_radians = np.deg2rad(lon)

x = np.cos(lat_radians) * np.cos(lon_radians)
y = np.cos(lat_radians) * np.sin(lon_radians)
z = np.sin(lat_radians)

df1['x_coor'] = x
df1['y_coor'] = y
df1['z_coor'] = z

In [17]:
df1[['x_coor', 'y_coor', 'z_coor']]

,x_coor,y_coor,z_coor
0,0.614467,0.055777,0.786969
1,0.615400,0.056595,0.786181
2,0.611561,0.055079,0.789278
5,0.595110,0.071916,0.800420
6,0.611238,0.055175,0.789522
...,...,...,...
7567,0.622088,0.061067,0.780562
7568,0.620683,0.060768,0.781703
7569,0.619772,0.051393,0.783097
7570,0.619809,0.051398,0.783068


# Data augmentation (bootstrapping) & preprocessing

Original dataset splitting and some continued preprocessing

In [18]:
#Indexing and ordering the data

df1['start'] = df1['start_date']

df1 = df1.sort_values(by='start_date').set_index('start_date')

In [19]:
#Remapping ALREADY DONE PREVIOUSLY
df1 = df1.replace({'incident_type': {0: 'Infrastructure damage',
                                                     1: 'Other',
                                                     2: 'Road ponding',
                                                     3: 'Roadside fire',
                                                     4: 'Slippery pavement'}})

label_enc21 = skprep.LabelEncoder()
label_enc21.fit(df1['incident_type'])
df1['incident_type'] = label_enc21.transform(df1['incident_type'])

In [20]:
df1

,start_time,end_date,end_time,incident_duration,incident_type,road_level,lon,lat,sum_prec (1h),max_prec (5min),max_temp,low_veg_%,mid_high_veg_%,misc_%,pavement_%,water_%,x_coor,y_coor,z_coor,start
start_date,,,,,,,,,,,,,,,,,,,,
2014-01-04,1900-01-01 09:26:41,2014-02-04,1900-01-01 04:21:48,1900-01-01 18:55:07,0,0,6.036875,52.193743,0.00,0.00,19.866745,62.0,12.0,6.0,19.0,2.0,0.609594,0.064468,0.790088,2014-01-04
2014-01-05,1900-01-01 18:20:28,2014-01-05,1900-01-01 20:34:11,1900-01-01 02:13:43,2,0,4.368744,51.875982,3.27,0.64,17.798628,46.0,0.0,1.0,46.0,1.0,0.615572,0.047028,0.786676,2014-01-05
2014-01-07,1900-01-01 13:20:56,2014-01-07,1900-01-01 13:41:23,1900-01-01 00:20:27,3,0,4.680794,51.849466,0.00,0.00,19.561977,52.0,3.0,2.0,40.0,5.0,0.615669,0.050409,0.786391,2014-01-07
2014-01-16,1900-01-01 05:51:00,2014-01-16,1900-01-01 06:30:10,1900-01-01 00:39:10,2,0,4.842503,52.353076,0.00,0.00,8.958661,11.0,5.0,7.0,31.0,0.0,0.608614,0.051561,0.791790,2014-01-16
2014-01-21,1900-01-01 12:46:46,2014-01-21,1900-01-01 13:44:12,1900-01-01 00:57:25,1,0,5.186706,51.903114,0.00,0.00,5.753218,35.0,0.0,1.0,33.0,3.0,0.614467,0.055777,0.786969,2014-01-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-31,1900-01-01 10:09:06,2022-12-31,1900-01-01 18:05:29,1900-01-01 07:56:23,2,0,4.342121,52.073279,0.51,0.42,14.974349,2.0,2.0,2.0,21.0,0.0,0.612889,0.046536,0.788798,2022-12-31
2022-12-31,1900-01-01 09:39:29,2022-12-31,1900-01-01 18:20:19,1900-01-01 08:40:50,2,1,4.416950,51.950147,0.12,0.08,15.047128,41.0,2.0,5.0,51.0,1.0,0.614516,0.047467,0.787475,2022-12-31
2022-12-31,1900-01-01 12:32:36,2022-12-31,1900-01-01 13:11:59,1900-01-01 00:39:23,2,0,4.387591,52.070766,3.16,0.72,15.006067,35.0,0.0,11.0,39.0,15.0,0.612886,0.047026,0.788771,2022-12-31


In [21]:
#Normalization and preprocessing

from sklearn import preprocessing

infra_col = ['low_veg_%',
               'mid_high_veg_%',
               'pavement_%',
               'water_%']

temp = ['max_temp']

scaler = preprocessing.MinMaxScaler()

df1[infra_col] = scaler.fit_transform(df1[infra_col])
df1[temp] = scaler.fit_transform(df1[temp])

#Log 1-p scaling of precipitation data

sum1 = df1['sum_prec (1h)']
logsum1 = np.log1p(sum1)
df1['sum_prec (1h)'] = logsum1

In [22]:
df1

,start_time,end_date,end_time,incident_duration,incident_type,road_level,lon,lat,sum_prec (1h),max_prec (5min),max_temp,low_veg_%,mid_high_veg_%,misc_%,pavement_%,water_%,x_coor,y_coor,z_coor,start
start_date,,,,,,,,,,,,,,,,,,,,
2014-01-04,1900-01-01 09:26:41,2014-02-04,1900-01-01 04:21:48,1900-01-01 18:55:07,0,0,6.036875,52.193743,0.000000,0.00,0.559042,0.446043,0.141176,6.0,0.174312,0.02,0.609594,0.064468,0.790088,2014-01-04
2014-01-05,1900-01-01 18:20:28,2014-01-05,1900-01-01 20:34:11,1900-01-01 02:13:43,2,0,4.368744,51.875982,1.451614,0.64,0.513341,0.330935,0.000000,1.0,0.422018,0.01,0.615572,0.047028,0.786676,2014-01-05
2014-01-07,1900-01-01 13:20:56,2014-01-07,1900-01-01 13:41:23,1900-01-01 00:20:27,3,0,4.680794,51.849466,0.000000,0.00,0.552307,0.374101,0.035294,2.0,0.366972,0.05,0.615669,0.050409,0.786391,2014-01-07
2014-01-16,1900-01-01 05:51:00,2014-01-16,1900-01-01 06:30:10,1900-01-01 00:39:10,2,0,4.842503,52.353076,0.000000,0.00,0.317999,0.079137,0.058824,7.0,0.284404,0.00,0.608614,0.051561,0.791790,2014-01-16
2014-01-21,1900-01-01 12:46:46,2014-01-21,1900-01-01 13:44:12,1900-01-01 00:57:25,1,0,5.186706,51.903114,0.000000,0.00,0.247166,0.251799,0.000000,1.0,0.302752,0.03,0.614467,0.055777,0.786969,2014-01-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-31,1900-01-01 10:09:06,2022-12-31,1900-01-01 18:05:29,1900-01-01 07:56:23,2,0,4.342121,52.073279,0.412110,0.42,0.450931,0.014388,0.023529,2.0,0.192661,0.00,0.612889,0.046536,0.788798,2022-12-31
2022-12-31,1900-01-01 09:39:29,2022-12-31,1900-01-01 18:20:19,1900-01-01 08:40:50,2,1,4.416950,51.950147,0.113329,0.08,0.452540,0.294964,0.023529,5.0,0.467890,0.01,0.614516,0.047467,0.787475,2022-12-31
2022-12-31,1900-01-01 12:32:36,2022-12-31,1900-01-01 13:11:59,1900-01-01 00:39:23,2,0,4.387591,52.070766,1.425515,0.72,0.451632,0.251799,0.000000,11.0,0.357798,0.15,0.612886,0.047026,0.788771,2022-12-31


In [36]:
#df1.to_csv('drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/df1.csv', header=True, index=True)

In [23]:
#Data splitting

train1 = df1.loc['2014-01-01':'2020-07-01']
val1 = df1.loc['2020-07-02':'2021-07-01']
test1 = df1.loc['2021-07-02':]

print("Train set shape:", train1.shape)
print("Validation set shape:", val1.shape)
print("Test set shape:", test1.shape)

Train set shape: (2999, 20)
Validation set shape: (559, 20)
Test set shape: (1100, 20)


In [24]:
#Splitting attributes

weather = ['max_temp','sum_prec (1h)']#.to_numpy().tolist()
infra = ['low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']
features = ['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']
spatial = ['x_coor', 'y_coor', 'z_coor']#.to_numpy().tolist()
coords = ['lat', 'lon']
output = ['incident_type']#tolist()

attributes1 = df1[['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']]

X_train_w = train1[weather]
X_train_i = train1[infra]
X_train1 = train1[features]
y_train1 = train1[output]
y_train1 = y_train1.values.flatten()

X_val_w = val1[weather]
X_val_i = val1[infra]
X_val1 = val1[features]
y_val1 = val1[output]
y_val1 = y_val1.values.flatten()

X_test_w = test1[weather]
X_test_i = test1[infra]
X_test1 = test1[features]
y_test1 = test1[output]
y_test1 = y_test1.values.flatten()

print("X train weather, infra, total, and y train shapes:", X_train_w.shape, X_train_i.shape, X_train1.shape, y_train1.shape)
print("X val weather, infra, total, and y val shapes:", X_val_w.shape, X_val_i.shape, X_val1.shape, y_val1.shape)
print("X test weather, infra, total, and y test shapes:", X_test_w.shape, X_test_i.shape, X_test1.shape, y_test1.shape)

X train weather, infra, total, and y train shapes: (2999, 2) (2999, 4) (2999, 6) (2999,)
X val weather, infra, total, and y val shapes: (559, 2) (559, 4) (559, 6) (559,)
X test weather, infra, total, and y test shapes: (1100, 2) (1100, 4) (1100, 6) (1100,)


In [25]:
print(df1.shape)

#{'Infrastructure damage': np.int64(0), 'Other': np.int64(1), 'Road ponding': np.int64(2), 'Roadside fire': np.int64(3), 'Slippery pavement': np.int64(4)}

print("Incident types distribution", df1['incident_type'].value_counts())

(4658, 20)
Incident types distribution incident_type
2    2129
3    1095
0     769
1     371
4     294
Name: count, dtype: int64


Bootstrapping data construction, preprocessing + splitting

In [26]:
#Bootstrapping

from sklearn.utils import resample

def bootstrapping_data(df, n_samples=None):
    """
    Bootstraps a dataset.

    Parameters:
        df (pd.DataFrame): The dataset to bootstrap.
        n_samples (int): How many rows to sample. Defaults to len(df).

    Returns:
        pd.DataFrame: Bootstrapped dataset.
    """
    if n_samples is None:
        n_samples = len(df)
    return resample(df, replace=True, n_samples=n_samples, random_state=27)

In [27]:
import random

random.seed(27)
bootstrapped_df = bootstrapping_data(df1, n_samples=10000)

# Sort all bootstrapped dataframe by 'start_date' to maintain time-series order
bootstrapped_df = bootstrapped_df.sort_values(by='start_date')

In [28]:
print("Bootstrapped outlier cluster URC distribution", bootstrapped_df['incident_type'].value_counts())
print("Original sample size - cluster outlier", df1.shape, "Bootstrapped sample size", bootstrapped_df.shape)


#'Infrastructure damage': np.int64(0), 'Other': np.int64(1), 'Road ponding': np.int64(2), #'Roadside fire': np.int64(3), 'Slippery pavement': np.int64(4), 'Zichtbaarheid': np.int64(5)

Bootstrapped outlier cluster URC distribution incident_type
2    4627
3    2310
0    1639
1     768
4     656
Name: count, dtype: int64
Original sample size - cluster outlier (4658, 20) Bootstrapped sample size (10000, 20)


In [29]:
#Preprocessing bootstrapped df - Standardization of infrastructure and temperature data

from sklearn import preprocessing

infra_col = ['low_veg_%',
               'mid_high_veg_%',
               'pavement_%',
               'water_%']

temp = ['max_temp']

scaler = preprocessing.MinMaxScaler()

bootstrapped_df[infra_col] = scaler.fit_transform(bootstrapped_df[infra_col])
bootstrapped_df[temp] = scaler.fit_transform(bootstrapped_df[temp])

#Log 1-p scaling of precipitation data

sum2 = bootstrapped_df['sum_prec (1h)']
logsum2 = np.log1p(sum2)

bootstrapped_df['sum_prec (1h)'] = logsum2

In [30]:
attributes2 = bootstrapped_df[['max_temp',
               'sum_prec (1h)',
               'low_veg_%',
               'mid_high_veg_%',
               'pavement_%',
               'water_%']]#.to_numpy().tolist()
spatial2 = bootstrapped_df[['x_coor', 'y_coor', 'z_coor']]#.to_numpy().tolist()
coords2 = bootstrapped_df[['lat', 'lon']]
output2 = bootstrapped_df['incident_type']#tolist()

In [31]:
coords2

,lat,lon
start_date,,
2014-01-04,52.193743,6.036875
2014-01-07,51.849466,4.680794
2014-01-16,52.353076,4.842503
2014-01-21,51.903114,5.186706
2014-01-25,52.041733,4.352465
...,...,...
2022-12-31,51.665756,5.300320
2022-12-31,51.668908,5.994987
2022-12-31,52.070766,4.387591


In [32]:
#Data splitting

train2 = bootstrapped_df.loc['2014-01-01':'2020-07-01']
val2 = bootstrapped_df.loc['2020-07-02':'2021-07-01']
test2 = bootstrapped_df.loc['2021-07-02':]

print("Train set shape:", train2.shape)
print("Validation set shape:", val2.shape)
print("Test set shape:", test2.shape)

Train set shape: (6471, 20)
Validation set shape: (1156, 20)
Test set shape: (2373, 20)


In [33]:
#Splitting attributes across train, val, test

weather = ['max_temp','sum_prec (1h)']#.to_numpy().tolist()
infra = ['low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']
features = ['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']
spatial = ['x_coor', 'y_coor', 'z_coor']#.to_numpy().tolist()
coords = ['lat', 'lon']
output = ['incident_type']#tolist()

X_train_w2 = train2[weather]
X_train_i2 = train2[infra]
X_train2 = train2[features]
y_train2 = train2[output]
y_train2 = y_train2.values.flatten()

X_val_w2 = val2[weather]
X_val_i2 = val2[infra]
X_val2 = val2[features]
y_val2 = val2[output]
y_val2 = y_val2.values.flatten()

X_test_w2 = test2[weather]
X_test_i2 = test2[infra]
X_test2 = test2[features]
y_test2 = test2[output]
y_test2 = y_test2.values.flatten()

print("X train weather, infra, total, and y train shapes:", X_train_w2.shape, X_train_i2.shape, X_train2.shape, y_train2.shape)
print("X val weather, infra, total, and y val shapes:", X_val_w2.shape, X_val_i2.shape, X_val2.shape, y_val2.shape)
print("X test weather, infra, total, and y test shapes:", X_test_w2.shape, X_test_i2.shape, X_test2.shape, y_test2.shape)

X train weather, infra, total, and y train shapes: (6471, 2) (6471, 4) (6471, 6) (6471,)
X val weather, infra, total, and y val shapes: (1156, 2) (1156, 4) (1156, 6) (1156,)
X test weather, infra, total, and y test shapes: (2373, 2) (2373, 4) (2373, 6) (2373,)


# RF, LR, XGBoost (Original dataset) - change feature sets

RF training

In [34]:
#Defining model

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, f1_score

# Initialize the Random Forest model with base parameters
rf_model = RandomForestClassifier(random_state=42)

#X_train_w = train1[weather]
#X_train_i = train1[infra]
#X_train1 = train1[features]
#y_train1 = train1[output]
#y_train1 = y_train1.values.flatten()

#X_val_w = val1[weather]
#X_val_i = val1[infra]
#X_val1 = val1[features]
#y_val1 = val1[output]
#y_val1 = y_val1.values.flatten()

#X_test_w = test1[weather]
#X_test_i = test1[infra]
#X_test1 = test1[features]
#y_test1 = test1[output]
#y_test1 = y_test1.values.flatten()


In [35]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score

def objective_rf(trial):
    # --- Define search space ---
    n_estimators = trial.suggest_int('n_estimators', 50, 500, step=50)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    # Build model using selected hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )

    # Fit on training data
    model.fit(X_train_i, y_train1) #CHANGE

    # Evaluate validation loss and accuracy
    y_val_pred_probs = model.predict_proba(X_val_i) #CHANGE
    y_val_pred = model.predict(X_val_i) #CHANGE
    val_loss = log_loss(y_val1, y_val_pred_probs) #CHANGE
    val_accuracy = accuracy_score(y_val1, y_val_pred) #CHANGE

    # Log accuracy as a user attribute so we can track it
    trial.set_user_attr('accuracy', val_accuracy)
    print(f"Trial {trial.number} -> val_loss: {val_loss:.4f}, val_accuracy: {val_accuracy:.4f}")

    return val_loss

# --- Run Bayesian Optimization Study to MINIMIZE loss ---
study_rf = optuna.create_study(direction='minimize')
study_rf.optimize(objective_rf, n_trials=20) # Adjust n_trials as needed

print("Optimization finished.")
print("Best trial (lowest loss):", study_rf.best_trial.params)
print("Best validation loss:", study_rf.best_value)
print("Best validation accuracy:", study_rf.best_trial.user_attrs['accuracy'])

[I 2026-07-29 11:16:10,410] A new study created in memory with name: no-name-e340e195-f1e3-4877-8c50-d7aef3a2926f
[I 2026-07-29 11:16:10,612] Trial 0 finished with value: 1.2947280651732374 and parameters: {'n_estimators': 50, 'max_depth': 20, 'min_samples_split': 9, 'min_samples_leaf': 9}. Best is trial 0 with value: 1.2947280651732374.


Trial 0 -> val_loss: 1.2947, val_accuracy: 0.5116


[I 2026-07-29 11:16:11,236] Trial 1 finished with value: 1.3024394093663747 and parameters: {'n_estimators': 200, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 7}. Best is trial 0 with value: 1.2947280651732374.


Trial 1 -> val_loss: 1.3024, val_accuracy: 0.5081


[I 2026-07-29 11:16:11,879] Trial 2 finished with value: 1.310385946384535 and parameters: {'n_estimators': 200, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 4}. Best is trial 0 with value: 1.2947280651732374.


...

Trial 6 -> val_loss: 1.3015, val_accuracy: 0.5242


[I 2026-07-29 11:16:17,821] Trial 7 finished with value: 1.3301436952017838 and parameters: {'n_estimators': 350, 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 7}. Best is trial 5 with value: 1.2940059623676359.


Trial 7 -> val_loss: 1.3301, val_accuracy: 0.4866


[W 2026-07-29 11:16:18,251] Trial 8 failed with parameters: {'n_estimators': 500, 'max_depth': 16, 'min_samples_split': 7, 'min_samples_leaf': 9} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_3139/4013914911.py", line 23, in objective_rf
    model.fit(X_train_i, y_train1) #CHANGE
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_forest.py", line 487, in fit
    trees = Parallel(
            ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/parallel.py", line 77, in __call__
    return super().__call__

KeyboardInterrupt: 

In [ ]:
#Best parameters

#Original df
#Best trial (lowest loss): {'n_estimators': 450, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 9}
#Best validation loss: 0.9638736544824301
#Best validation accuracy: 0.6064400715563506

#Bootstrapped df

#Original - w
#Best trial (lowest loss): {'n_estimators': 100, 'max_depth': 4, 'min_samples_split': 9, 'min_samples_leaf': 9}
#Best validation loss: 1.009183733833864
#Best validation accuracy: 0.5813953488372093

#Bootstrapped - w

#Original - i
#Best trial (lowest loss): {'n_estimators': 350, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 10}
#Best validation loss: 1.2927044763641955
#Best validation accuracy: 0.5134168157423972

#Bootstrapped - i

In [ ]:
#Compile model with tuned parameters

rf_model = RandomForestClassifier(
        n_estimators=350,
        max_depth=17,
        min_samples_split=6,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )



# Train the model
print('Training Random Forest model...')
rf_model.fit(X_train_i, y_train1) #CHANGE

# Validate the model
print('\nEvaluating on Validation Set:')
y_val_pred = rf_model.predict(X_val_i) #CHANGE
print(f'Validation Accuracy: {accuracy_score(y_val1, y_val_pred):.4f}') #CHANGE

RF testing + evaluation metrics

In [ ]:
# Test the model

from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class_names = list(label_enc1_mapping.keys())

print('\nEvaluating on Test Set:')
y_true = y_test1 #CHANGE
y_pred = rf_model.predict(X_test_i) #CHANGE
print(f'Test Accuracy: {accuracy_score(y_true, y_pred):.4f}')
print('\nClassification Report (Test Set):')
print(classification_report(y_true, y_pred, target_names=class_names))
#Balanced Accuracy
balanced_acc = balanced_accuracy_score(y_true, y_pred)
print(f"Balanced Accuracy on Test Set: {balanced_acc:.4f}")

#Weighted F1 Score
w_f1_score = f1_score(y_true, y_pred, average='weighted')
print(f"Weighted F1 Score on Test Set: {w_f1_score:.4f}")

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix Random Forest - Original dataset (IP features)") #CHANGE THIS ACCORDING TO CLUSTER
plt.show()

print("\nFeature Coefficients (Mean Absolute weight across classes):")
rf_model_importances = pd.Series(np.mean(np.abs(rf_model.feature_importances_), axis=0), index=features).sort_values(ascending=False)
display(rf_model_importances)

In [ ]:
#Bootstrapped evaluation metrics

from sklearn.utils import resample
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, balanced_accuracy_score
import pandas as pd

# Initialize lists to store metrics
metrics_list = []

n_iterations = 10
sample_size = 4*len(X_test_i) #CHANGE

# Ensure class_names is available, fallback to model classes if not
try:
    c_names = class_names
except NameError:
    c_names = [f"Class {c}" for c in rf_model.classes_]

for i in range(n_iterations):
    # Bootstrap the test set
    X_test_boot, y_test_boot = resample(X_test_i, y_test1, n_samples=sample_size, random_state=42 + i, replace=True) #CHANGE

    # Predict using the trained rf_model
    y_pred_boot = rf_model.predict(X_test_boot)

    # Calculate general metrics
    precision = precision_score(y_test_boot, y_pred_boot, average='weighted', zero_division=0)
    recall = recall_score(y_test_boot, y_pred_boot, average='weighted', zero_division=0)
    f1 = f1_score(y_test_boot, y_pred_boot, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test_boot, y_pred_boot)
    balanced_acc = balanced_accuracy_score(y_test_boot, y_pred_boot)

    # Calculate per-class metrics
    precision_per_class = precision_score(y_test_boot, y_pred_boot, average=None, zero_division=0, labels=rf_model.classes_)
    recall_per_class = recall_score(y_test_boot, y_pred_boot, average=None, zero_division=0, labels=rf_model.classes_)

    # Build iteration dictionary
    iter_metrics = {
       'iteration': i + 1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }

    # Add per-class metrics to the dictionary
    for idx, c_name in enumerate(c_names):
        iter_metrics[f'precision_{c_name}'] = precision_per_class[idx]
        iter_metrics[f'recall_{c_name}'] = recall_per_class[idx]

    # Append to list
    metrics_list.append(iter_metrics)

# Create a DataFrame to view and save the metrics
bootstrap_metrics_df = pd.DataFrame(metrics_list)
display(bootstrap_metrics_df)

# Report the mean and standard deviation of all evaluation metrics
mean_metrics = bootstrap_metrics_df.drop(columns=['iteration']).mean()
std_metrics = bootstrap_metrics_df.drop(columns=['iteration']).std()

summary_rf_df = pd.DataFrame({
    'Mean': mean_metrics,
    'Standard Deviation': std_metrics
})

print("\nMean and Standard Deviation of Evaluation Metrics (10 iterations):")
display(summary_rf_df)


In [ ]:
# Extract prediction probabilities for the test set
y_pred_probs_rf = rf_model.predict_proba(X_test_i) #CHANGE

# Convert to a DataFrame for better readability
#rf_probs_df = pd.DataFrame(y_pred_probs_rf, columns=[f'Class {c}' for c in rf_model.classes_])
rf_probs_df = pd.DataFrame(y_pred_probs_rf, columns=[c for c in class_names])

rf_probs_df[100:110]

rf_probs_df.to_csv('drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/pred_df1_i_rf.csv', header=True, index=True)

summary_rf_df.to_csv('drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/rf_df1_i_summary_metrics.csv', index=True)

import joblib
joblib.dump(rf_model, '/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/Ablation_models/rf_df1_i.pkl')

LR training

In [ ]:
#LR trained on SGD, 100 epochs, Softmax probabilities as class outputs

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score, f1_score

# Initialize the SGDClassifier with logistic regression loss (log_loss)
lr_model = SGDClassifier(loss='log_loss', max_iter=100,         # 100 epochs
        learning_rate="optimal",
        verbose=0,
        random_state=42)

In [ ]:
import optuna
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss, accuracy_score

def objective_lr(trial):
    # --- Define search space ---
    penalty = trial.suggest_categorical('penalty', ['l2', 'l1', 'elasticnet'])
    alpha = trial.suggest_float('alpha', 1e-6, 1e-1, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0) # Used if penalty='elasticnet'
    learning_rate = trial.suggest_categorical('learning_rate', ['constant', 'optimal', 'invscaling', 'adaptive'])
    eta0 = trial.suggest_float('eta0', 1e-5, 1e-1, log=True) # Required for constant, invscaling, adaptive

    # Build model using selected hyperparameters
    model = SGDClassifier(
        loss='log_loss',
        max_iter=100,
        penalty=penalty,
        alpha=alpha,
        l1_ratio=l1_ratio,
        learning_rate=learning_rate,
        eta0=eta0,
        random_state=42,
        verbose=0
    )

    # Fit on training data
    model.fit(X_train_i, y_train1) #CHANGE

    # Return the validation log loss achieved in this trial
    y_val_pred_probs = model.predict_proba(X_val_i) #CHANGE
    y_val_pred = model.predict(X_val_i) #CHANGE
    val_loss = log_loss(y_val1, y_val_pred_probs)
    val_accuracy = accuracy_score(y_val1, y_val_pred)

    trial.set_user_attr('accuracy', val_accuracy)
    print(f"Trial {trial.number} -> val_loss: {val_loss:.4f}, val_accuracy: {val_accuracy:.4f}")

    return val_loss

# --- Run Bayesian Optimization Study to MINIMIZE loss ---
study_lr = optuna.create_study(direction='minimize')
study_lr.optimize(objective_lr, n_trials=20) # Adjust n_trials as needed

print("Optimization finished.")
print("Best trial (lowest loss):", study_lr.best_trial.params)
print("Best validation loss:", study_lr.best_value)
print("Best validation accuracy:", study_lr.best_trial.user_attrs['accuracy'])

In [ ]:
#Best parameters

#Original df
#Best trial (lowest loss): {'penalty': 'l1', 'alpha': 1.2966824341784752e-06, 'l1_ratio': 0.7399410945520057, 'learning_rate': 'constant', 'eta0': 0.07863901493969765}
#Best validation loss: 0.9788361403226371
#Best validation accuracy: 0.6118067978533095

#Bootstrapped df

#Original - w
#Best trial (lowest loss): {'penalty': 'l1', 'alpha': 3.210096732284539e-05, 'l1_ratio': 0.6556522027336379, 'learning_rate': 'adaptive', 'eta0': 0.09820780548592228}
#Best validation loss: 1.018364387752707
#Best validation accuracy: 0.556350626118068

#Bootstrapped - w

#Original - i

#Bootstrapped - i
#Best trial (lowest loss): {'penalty': 'l1', 'alpha': 1.1923084322693968e-06, 'l1_ratio': 0.3384945381066028, 'learning_rate': 'adaptive', 'eta0': 0.09263365387651848}
#Best validation loss: 1.3364966027251404
#Best validation accuracy: 0.4847942754919499

In [ ]:
#Compile model with new parameters

lr_model = SGDClassifier(loss='log_loss',
        max_iter=100,
        penalty='l1',
        alpha=0,
        l1_ratio=0.3384945381066028,
        learning_rate='adaptive',
        eta0=0.09263365387651848)

# Train the model
print('Training Logistic Regression (SGD) model...')
lr_model.fit(X_train_i, y_train1) #CHANGE

# Validate the model
print('\nEvaluating on Validation Set:')
y_val_pred_lr = lr_model.predict(X_val_i) #CHANGE
print(f'Validation Accuracy: {accuracy_score(y_val1, y_val_pred_lr):.4f}') #CHANGE

LR testing + evaluation metrics

In [ ]:
# Test the model
print('\nEvaluating on Test Set:')
y_true_lr = y_test1 #CHANGE
y_pred_lr = lr_model.predict(X_test_i) #CHANGE

print(f'Test Accuracy: {accuracy_score(y_true_lr, y_pred_lr):.4f}')
print('\nClassification Report (Test Set):')
print(classification_report(y_true_lr, y_pred_lr, target_names=class_names, zero_division=0))

# Balanced Accuracy
balanced_acc_lr = balanced_accuracy_score(y_true_lr, y_pred_lr)
print(f"Balanced Accuracy on Test Set: {balanced_acc_lr:.4f}")

# Weighted F1 Score
w_f1_score_lr = f1_score(y_true_lr, y_pred_lr, average='weighted', zero_division=0)
print(f"Weighted F1 Score on Test Set: {w_f1_score_lr:.4f}")

cm = confusion_matrix(y_true_lr, y_pred_lr)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix Logistic Regression - Original dataset (IP features)") #CHANGE THIS ACCORDING TO CLUSTER
plt.show()

In [ ]:
#Bootstrapped evaluation metrics

from sklearn.utils import resample
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, balanced_accuracy_score
import pandas as pd

# Initialize lists to store metrics
metrics_list_lr = []

n_iterations = 10
sample_size = 4*len(X_test_i) #CHANGE

# Ensure class_names is available, fallback to model classes if not
try:
    c_names = class_names
except NameError:
    c_names = [f"Class {c}" for c in lr_model.classes_]

for i in range(n_iterations):
    # Bootstrap the test set
    X_test_boot, y_test_boot = resample(X_test_i, y_test1, n_samples=sample_size, random_state=42 + i, replace=True) #CHANGE

    # Predict using the trained lr_model
    y_pred_boot = lr_model.predict(X_test_boot)

    # Calculate general metrics
    precision = precision_score(y_test_boot, y_pred_boot, average='weighted', zero_division=0)
    recall = recall_score(y_test_boot, y_pred_boot, average='weighted', zero_division=0)
    f1 = f1_score(y_test_boot, y_pred_boot, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test_boot, y_pred_boot)
    balanced_acc = balanced_accuracy_score(y_test_boot, y_pred_boot)

    # Calculate per-class metrics
    precision_per_class = precision_score(y_test_boot, y_pred_boot, average=None, zero_division=0, labels=lr_model.classes_)
    recall_per_class = recall_score(y_test_boot, y_pred_boot, average=None, zero_division=0, labels=lr_model.classes_)

    # Build iteration dictionary
    iter_metrics = {
        'iteration': i + 1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }

    # Add per-class metrics to the dictionary
    for idx, c_name in enumerate(c_names):
        iter_metrics[f'precision_{c_name}'] = precision_per_class[idx]
        iter_metrics[f'recall_{c_name}'] = recall_per_class[idx]

    # Append to list
    metrics_list_lr.append(iter_metrics)

# Create a DataFrame to view and save the metrics
bootstrap_metrics_lr_df = pd.DataFrame(metrics_list_lr)
display(bootstrap_metrics_lr_df)

# Report the mean and standard deviation of all evaluation metrics
mean_metrics = bootstrap_metrics_lr_df.drop(columns=['iteration']).mean()
std_metrics = bootstrap_metrics_lr_df.drop(columns=['iteration']).std()

summary_lr_df = pd.DataFrame({
    'Mean': mean_metrics,
    'Standard Deviation': std_metrics
})

print("\nMean and Standard Deviation of Evaluation Metrics (10 iterations):")
display(summary_lr_df)

In [ ]:
# Extract prediction probabilities for the test set (Logistic Regression)
y_pred_probs_lr = lr_model.predict_proba(X_test_i)

# Convert to a DataFrame for better readability
lr_probs_df = pd.DataFrame(y_pred_probs_lr, columns=class_names)

# Display the first 10 rows
display(lr_probs_df.head(10))

# Optional: Save to CSV
lr_probs_df.to_csv('drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/pred_df1_i_lr.csv', header=True, index=True)

bootstrap_metrics_lr_df.to_csv('drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/lr_df1_i_summary_metrics.csv', index=False)

import joblib
joblib.dump(rf_model, '/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/Ablation_models/lr_df1_i.pkl')

XGBoost training

In [ ]:
#Model definition

!pip install xgboost
import xgboost as xgb
from sklearn.metrics import accuracy_score

# Initialize the XGBoost model with base parameters for multiclass classification
xgb_model = xgb.XGBClassifier(random_state=42, eval_metric='mlogloss',
                              verbosity=2)

In [ ]:
import optuna
import xgboost as xgb
from sklearn.metrics import log_loss, accuracy_score

def objective_xgb(trial):
    # --- Define search space ---
    param = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100, 250, 500]),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42,
        'eval_metric': 'mlogloss',
        'verbosity': 0
    }

    # Build model using selected hyperparameters
    model = xgb.XGBClassifier(**param)

    # Fit on training data with early stopping based on val_loss
    model.fit(
        X_train_i, y_train1, #CHANGE
        eval_set=[(X_val_i, y_val1)], #CHANGE
        verbose=False
    )

    # Return the validation log loss achieved in this trial
    y_val_pred_probs = model.predict_proba(X_val_i) #CHANGE
    y_val_pred = model.predict(X_val_i) #CHANGE
    val_loss = log_loss(y_val1, y_val_pred_probs)
    val_accuracy = accuracy_score(y_val1, y_val_pred)

    trial.set_user_attr('accuracy', val_accuracy)
    print(f"Trial {trial.number} -> val_loss: {val_loss:.4f}, val_accuracy: {val_accuracy:.4f}")

    return val_loss

# --- Run Bayesian Optimization Study to MINIMIZE loss ---
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=20) # Adjust n_trials as needed based on compute limits

print("Optimization finished.")
print("Best trial (lowest loss):", study_xgb.best_trial.params)
print("Best validation loss:", study_xgb.best_value)
print("Best validation accuracy:", study_xgb.best_trial.user_attrs['accuracy'])

In [ ]:
#Best parameters

#Original df
#Best trial (lowest loss): {'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.01870683573665569, 'subsample': 0.5017198734678123, 'colsample_bytree': 0.8302734821239883}
#Best validation loss: 0.9577331934514111
#Best validation accuracy: 0.6100178890876565

#Bootstrapped df

#Original - w
#Best trial (lowest loss): {'n_estimators': 250, 'max_depth': 5, 'learning_rate': 0.021246315041165902, 'subsample': 0.6350353338161683, 'colsample_bytree': 0.8379108525995116}
#Best validation loss: 1.0522794809816156
#Best validation accuracy: 0.5670840787119857

#Bootstrapped - w

#Original - i
#Best trial (lowest loss): {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.021189415346917848, 'subsample': 0.7881216968570179, 'colsample_bytree': 0.995588174218036}
#Best validation loss: 1.292640252193927
#Best validation accuracy: 0.5098389982110912

#Bootstrapped - i

In [ ]:
#Compile model with new parameters
xgb_model = xgb.XGBClassifier(random_state=42, eval_metric='mlogloss',
                              verbosity=2,
                              n_estimators=100,
                              max_depth=10,
                              learning_rate=0.021189415346917848,
                              subsample=0.7881216968570179,
                              colsample_bytree=0.995588174218036)
# Train the model
print('Training XGBoost model...')
xgb_model.fit(X_train_i, y_train1) #CHANGE

# Validate the model
print('\nEvaluating on Validation Set:')
y_val_pred_xgb = xgb_model.predict(X_val_i) #CHANGE
print(f'Validation Accuracy: {accuracy_score(y_val1, y_val_pred_xgb):.4f}')

XGBoost testing + evaluation metrics

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score

print('\nEvaluating on Test Set:')
y_true_xgb = y_test1 #CHANGE
y_pred_xgb = xgb_model.predict(X_test_i) #CHANGE

print(f'Test Accuracy: {accuracy_score(y_true_xgb, y_pred_xgb):.4f}')
print('\nClassification Report (Test Set):')
print(classification_report(y_true_xgb, y_pred_xgb, target_names=class_names))

# Balanced Accuracy
balanced_acc_xgb = balanced_accuracy_score(y_true_xgb, y_pred_xgb)
print(f"Balanced Accuracy on Test Set: {balanced_acc_xgb:.4f}")

# Weighted F1 Score
w_f1_score_xgb = f1_score(y_true_xgb, y_pred_xgb, average='weighted')
print(f"Weighted F1 Score on Test Set: {w_f1_score_xgb:.4f}")

cm = confusion_matrix(y_true_xgb, y_pred_xgb)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix XGBoost - Original dataset (IP features)") #CHANGE THIS ACCORDING TO CLUSTER
plt.show()

In [ ]:
from sklearn.utils import resample
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, balanced_accuracy_score
import pandas as pd

# Initialize lists to store metrics
metrics_list_xgb = []

n_iterations = 10
sample_size = 4*len(X_test_i) #CHANGE

# Ensure class_names is available, fallback to model classes if not
try:
    c_names = class_names
except NameError:
    c_names = [f"Class {c}" for c in xgb_model.classes_]

for i in range(n_iterations):
    # Bootstrap the test set
    X_test_boot, y_test_boot = resample(X_test_i, y_test1, n_samples=sample_size, random_state=42 + i, replace=True) #CHANGE

    # Predict using the trained xgb_model
    y_pred_boot = xgb_model.predict(X_test_boot)

    # Calculate general metrics
    precision = precision_score(y_test_boot, y_pred_boot, average='weighted', zero_division=0)
    recall = recall_score(y_test_boot, y_pred_boot, average='weighted', zero_division=0)
    f1 = f1_score(y_test_boot, y_pred_boot, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test_boot, y_pred_boot)
    balanced_acc = balanced_accuracy_score(y_test_boot, y_pred_boot)

    # Calculate per-class metrics
    precision_per_class = precision_score(y_test_boot, y_pred_boot, average=None, zero_division=0, labels=xgb_model.classes_)
    recall_per_class = recall_score(y_test_boot, y_pred_boot, average=None, zero_division=0, labels=xgb_model.classes_)

    # Build iteration dictionary
    iter_metrics = {
       'iteration': i + 1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }

    # Add per-class metrics to the dictionary
    for idx, c_name in enumerate(c_names):
        iter_metrics[f'precision_{c_name}'] = precision_per_class[idx]
        iter_metrics[f'recall_{c_name}'] = recall_per_class[idx]

    # Append to list
    metrics_list_xgb.append(iter_metrics)

# Create a DataFrame to view and save the metrics
bootstrap_metrics_xgb_df = pd.DataFrame(metrics_list_xgb)
display(bootstrap_metrics_xgb_df)

# Report the mean and standard deviation of all evaluation metrics
mean_metrics = bootstrap_metrics_xgb_df.drop(columns=['iteration']).mean()
std_metrics = bootstrap_metrics_xgb_df.drop(columns=['iteration']).std()

summary_xgb_df = pd.DataFrame({
    'Mean': mean_metrics,
    'Standard Deviation': std_metrics
})

print("\nMean and Standard Deviation of Evaluation Metrics (10 iterations):")
display(summary_xgb_df)


In [ ]:
import pandas as pd

# Extract prediction probabilities for the test set
y_pred_probs_xgb = xgb_model.predict_proba(X_test_i) #CHANGE

# Convert to a DataFrame for better readability using class names
xgb_probs_df = pd.DataFrame(y_pred_probs_xgb, columns=class_names)

# Display the first 10 rows
display(xgb_probs_df.head(10))

# Optional: Save to CSV
xgb_probs_df.to_csv('drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/pred_df1_i_xgb.csv', header=True, index=True)

bootstrap_metrics_xgb_df.to_csv('drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/xgb_df1_i_summary_metrics.csv', index=False)

import joblib
joblib.dump(rf_model, '/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/Ablation_models/xgb_df1_i.pkl')

# GCN-BLSTM, GAT-BLSTM, GCN, GAT

Preprocessing original dataset (graph construction + time-windowing)

Graph Construction

In [ ]:
#Calculate spatial distance using Haversine #LOAD THEM FOR NEXT TIME

#!pip install haversine
#from haversine import haversine, Unit

#original dataset

#coords = df1[['lat', 'lon']]
#n1 = len(coords)
#spatial_dist1 = np.zeros((n1, n1))

#for i in range(n1):
#    for j in range(n1):
#        if i != j:
#            spatial_dist1[i, j] = haversine(tuple(coords.iloc[i]), tuple(coords.iloc[j]), unit=Unit.KILOMETERS)

#adj_df1 = pd.DataFrame(spatial_dist1)
#adj_df1.to_csv('drive/MyDrive/PhD data analysis/Paper1/adj_df1.csv', header=True, index=True)

#coord2 = bootstrapped_df[['lat', 'lon']]
#n2 = len(coord2)
#spatial_dist2 = np.zeros((n2, n2))

#for i in range(n2):
#    for j in range(n2):
#        if i != j:
#            spatial_dist2[i, j] = haversine(tuple(coord2.iloc[i]), tuple(coord2.iloc[j]), unit=Unit.KILOMETERS)

#adj_bootstrap = pd.DataFrame(spatial_dist2)
#adj_bootstrap.to_csv('drive/MyDrive/PhD data analysis/Paper1/adj_bootstrap.csv', header=True, index=True)

In [ ]:
#Load for next time

adj_df1 = pd.read_csv("/content/drive/MyDrive/PhD data analysis/Paper1/adj_df1.csv")
adj_df1 = adj_df1.drop(columns=['Unnamed: 0'])

adj_bootstrap = pd.read_csv("/content/drive/MyDrive/PhD data analysis/Paper1/adj_bootstrap.csv")
adj_bootstrap = adj_bootstrap.drop(columns=['Unnamed: 0'])

print("Original data spatial distance matrix shape:", adj_df1.shape)
print("Bootstrapped data spatial distance matrix shape:", adj_bootstrap.shape)

In [ ]:
#Create diff input matrices for graph construction

#Original data

weather1 = df1[['max_temp','sum_prec (1h)']]#.to_numpy().tolist()
infra1 = df1[['low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']]
features1 = df1[['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']]
spatial1 = df1[['x_coor', 'y_coor', 'z_coor']]#.to_numpy().tolist()
coords1 = df1[['lat', 'lon']]
output1 = df1[['incident_type']]#tolist()
output1 = output1.values.flatten()

attributes1 = df1[['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']]

#Bootstrap

weather2 = bootstrapped_df[['max_temp','sum_prec (1h)']]#.to_numpy().tolist()
infra2 = bootstrapped_df[['low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']]
features2 = bootstrapped_df[['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']]
spatial2 = bootstrapped_df[['x_coor', 'y_coor', 'z_coor']]#.to_numpy().tolist()
coords2 = bootstrapped_df[['lat', 'lon']]
output2 = bootstrapped_df[['incident_type']]#tolist()
output2 = output2.values.flatten()

attributes2 = bootstrapped_df[['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']]

In [ ]:
#Convert to similarity (Exponential decay)

similarity_df = np.exp(-adj_df1 / adj_df1.std())
similarity_bootstrap = np.exp(-adj_bootstrap / adj_bootstrap.std())

In [ ]:
#Attribute similarity using Cosine similarity

from sklearn.metrics.pairwise import cosine_similarity

#All features

attr_sim_df = cosine_similarity(attributes1)
attr_sim_bootstrap = cosine_similarity(attributes2)

#Weather features

w_attr_sim_df = cosine_similarity(weather1)
w_attr_sim_bootstrap = cosine_similarity(weather2)

#Infrastructure features

i_attr_sim_df = cosine_similarity(infra1)
i_attr_sim_bootstrap = cosine_similarity(infra2)

In [ ]:
#Check for negative values - No negative values, so no opposities, only similarities (good news)
print(attr_sim_df[attr_sim_df<0])
print(attr_sim_bootstrap[attr_sim_bootstrap<0])
print(w_attr_sim_df[w_attr_sim_df<0])
print(w_attr_sim_bootstrap[w_attr_sim_bootstrap<0])
print(i_attr_sim_df[i_attr_sim_df<0])
print(i_attr_sim_bootstrap[i_attr_sim_bootstrap<0])
print("Attr. matrix original data shape:", attr_sim_df.shape)
print("Attr. matrix bootstrapped shape:", attr_sim_bootstrap.shape)
print("Weather attr. matrix original data shape:", w_attr_sim_df.shape)
print("Weather attr. matrix bootstrapped shape:", w_attr_sim_bootstrap.shape)
print("Infrastructure attr. matrix original data shape:", i_attr_sim_df.shape)
print("Infrastructure attr. matrix bootstrapped shape:", i_attr_sim_bootstrap.shape)

In [ ]:
#Combine spatial + attribute similarities - Weighted sum or product (tunable hyperparameter alpha):

alpha = 0.5  # weight for spatial vs attribute

#All attributes

combined_sim_df = alpha * similarity_df + (1 - alpha) * attr_sim_df
combined_sim_boot = alpha * similarity_bootstrap + (1 - alpha) * attr_sim_bootstrap

#Weather attributes

w_combined_sim_df = alpha * similarity_df + (1 - alpha) * w_attr_sim_df
w_combined_sim_boot = alpha * similarity_bootstrap + (1 - alpha) * w_attr_sim_bootstrap

#Infrastructure attributes

i_combined_sim_df = alpha * similarity_df + (1 - alpha) * i_attr_sim_df
i_combined_sim_boot = alpha * similarity_bootstrap + (1 - alpha) * i_attr_sim_bootstrap


In [ ]:
combined_sim_df

In [ ]:
#Only keep entries with high similarities, the rest is sparse

threshold = 0.8
combined_sim_df[combined_sim_df < threshold] = 0 #All points less than the threshold are 0
combined_sim_boot[combined_sim_boot < threshold] = 0
w_combined_sim_df[w_combined_sim_df < threshold] = 0
w_combined_sim_boot[w_combined_sim_boot < threshold] = 0
i_combined_sim_df[i_combined_sim_df < threshold] = 0
i_combined_sim_boot[i_combined_sim_boot < threshold] = 0

A_df = combined_sim_df.copy()
A_boot = combined_sim_boot.copy()
w_A_df = w_combined_sim_df.copy()
w_A_boot = w_combined_sim_boot.copy()
i_A_df = i_combined_sim_df.copy()
i_A_boot = i_combined_sim_boot.copy()

print(A_df.shape, A_boot.shape)
print(w_A_df.shape, w_A_boot.shape)
print(i_A_df.shape, i_A_boot.shape)

In [ ]:
A_df

In [ ]:
#Add self loops

from spektral.utils import add_self_loops

A_np1 = A_df.to_numpy()
A_with_loops1 = add_self_loops(A_np1)

A_np2 = A_boot.to_numpy()
A_with_loops2 = add_self_loops(A_np2)

w_A_np1 = w_A_df.to_numpy()
w_A_with_loops1 = add_self_loops(w_A_np1)

w_A_np2 = w_A_boot.to_numpy()
w_A_with_loops2 = add_self_loops(w_A_np2)

i_A_np1 = i_A_df.to_numpy()
i_A_with_loops1 = add_self_loops(i_A_np1)

i_A_np2 = i_A_boot.to_numpy()
i_A_with_loops2 = add_self_loops(i_A_np2)

print(i_A_with_loops1.shape)
print(i_A_with_loops2.shape)

In [ ]:
#Preprocessing the adjacency matrix
from spektral.utils import add_self_loops, normalized_adjacency, gcn_filter

# Normalize adjacency
A_with_loops1 = gcn_filter(A_with_loops1).astype("float32")
A_with_loops2 = gcn_filter(A_with_loops2).astype("float32")
w_A_with_loops1 = gcn_filter(w_A_with_loops1).astype("float32")
w_A_with_loops2 = gcn_filter(w_A_with_loops2).astype("float32")
i_A_with_loops1 = gcn_filter(i_A_with_loops1).astype("float32")
i_A_with_loops2 = gcn_filter(i_A_with_loops2).astype("float32")


from scipy.sparse import csr_matrix

# Convert the dense numpy array to a CSR sparse matrix
A_sparse1 = csr_matrix(A_with_loops1)
A_sparse2 = csr_matrix(A_with_loops2)
w_A_sparse1 = csr_matrix(w_A_with_loops1)
w_A_sparse2 = csr_matrix(w_A_with_loops2)
i_A_sparse1 = csr_matrix(i_A_with_loops1)
i_A_sparse2 = csr_matrix(i_A_with_loops2)

print(A_sparse1.shape)
print(A_sparse2.shape)
print(w_A_sparse1.shape)
print(w_A_sparse2.shape)
print(i_A_sparse1.shape)
print(i_A_sparse2.shape)

In [ ]:
# Generate the degree matrix
# The degree matrix is a diagonal matrix where the diagonal elements are the sum of the corresponding rows of the adjacency matrix.

D1 = np.diag(np.sum(A_with_loops1, axis=1))
print(D1.shape)
D2 = np.diag(np.sum(A_with_loops2, axis=1))
print(D2.shape)

In [ ]:
#Visualization weighted adjacency matrix

import matplotlib.pyplot as plt
import seaborn as sns

sns.heatmap(combined_sim_df.iloc[:50, :50], annot=True, cmap="Blues") #Change for boot and df
plt.title("Weighted Adjacency Matrix")
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

C = A_with_loops1[100:120, 100:120] #Change for boot and df
G = nx.from_numpy_array(C)
nx.draw(G, with_labels=False, node_color='lightblue')
plt.title("Graph structure from adjacency matrix")
plt.show()

Input and output matrices

In [ ]:
y1 = output1
y2 = output2

features_com1 = np.hstack([attributes1])#, spatial1])
features_com2 = np.hstack([attributes2])#, spatial2])

weather_com1 = np.hstack([weather1])
weather_com2 = np.hstack([weather2])

infra_com1 = np.hstack([infra1])
infra_com2 = np.hstack([infra2])

X1 = features_com1 #CHANGE FOR ABLATION
X2 = features_com2 #CHANGE FOR ABLATION

X1_w = weather_com1
X2_w = weather_com2

X1_i = infra_com1
X2_i = infra_com2

print("Original df shapes:", y1.shape, X1.shape, X1_w.shape, X1_i.shape)
print("Bootstrapped shapes:", y2.shape, X2.shape, X2_w.shape, X2_i.shape)


Single-mode dataloader to create graphs

In [ ]:
from spektral.data import Graph, Dataset

#Create Spektral dataset

from spektral.data import Dataset, Graph

#A_sparse1 = Sparse A with all features for original df
#A_sparse2 = Sparse A with all features (bootstrap)
#w_A_sparse1 = Sparse A with weather features for original df
#w_A_sparse2 = Sparse A with weather features (bootstrap)
#i_A_sparse1 = Sparse A with infrastructure features for original df
#i_A_sparse2 = Sparse A with infrastructure features (bootstrap)

class NodeLevelDataset1(Dataset):
    def read(self):
        return [Graph(x=features_com1, a=A_sparse1.toarray(), y=y1)] #Graph for original df with ALL features

class NodeLevelDataset2(Dataset):
    def read(self):
        return [Graph(x=features_com2, a=A_sparse2.toarray(), y=y2)] #Graph for bootstrapped df with ALL features

class NodeLevelDataset3(Dataset):
    def read(self):
        return [Graph(x=weather_com1, a=w_A_sparse1.toarray(), y=y1)] #Graph based on WEATHER features for original df

class NodeLevelDataset4(Dataset):
    def read(self):
        return [Graph(x=weather_com2, a=w_A_sparse2.toarray(), y=y2)] #Graph based on WEATHER features for bootstrapped df

class NodeLevelDataset5(Dataset):
    def read(self):
        return [Graph(x=infra_com1, a=i_A_sparse1.toarray(), y=y1)] #Graph based on INFRA features for original df

class NodeLevelDataset6(Dataset):
    def read(self):
        return [Graph(x=infra_com2, a=i_A_sparse2.toarray(), y=y2)] #Graph based on INFRA features for bootstrap

dataset1 = NodeLevelDataset1()
dataset2 = NodeLevelDataset2()
dataset3 = NodeLevelDataset3()
dataset4 = NodeLevelDataset4()
dataset5 = NodeLevelDataset5()
dataset6 = NodeLevelDataset6()

graph_df_all = dataset1[0]
graph_boot_all= dataset2[0]
w_graph_df = dataset3[0]
w_graph_boot = dataset4[0]
i_graph_df = dataset5[0]
i_graph_boot = dataset6[0]


num_nodes1 = graph_df_all.n_nodes
num_nodes2 = graph_boot_all.n_nodes
w_num_nodes1 = w_graph_df.n_nodes
w_num_nodes2 = w_graph_boot.n_nodes
i_num_nodes1 = i_graph_df.n_nodes
i_num_nodes2 = i_graph_boot.n_nodes

Time-windowing original df + bootstrapped

In [ ]:
C = 5
y_one_hot1 = tf.keras.utils.to_categorical(y1, num_classes=C)
y_one_hot2 = tf.keras.utils.to_categorical(y2, num_classes=C)

print("Original df: X shape:", X1.shape, "A shape:", A_with_loops1.shape, "y shape:", y_one_hot1.shape)
print("Bootstrapped: X shape:", X2.shape, "A shape:", A_with_loops2.shape, "y shape:", y_one_hot2.shape)

In [ ]:
#Time windowing

#X1 = features_com1
#X2 = features_com2

#X1_w = weather_com1
#X2_w = weather_com2

#X1_i = infra_com1
#X2_i = infra_com2


#A_sparse1 = Sparse A with all features for original df
#A_sparse2 = Sparse A with all features (bootstrap)
#w_A_sparse1 = Sparse A with weather features for original df
#w_A_sparse2 = Sparse A with weather features (bootstrap)
#i_A_sparse1 = Sparse A with infrastructure features for original df
#i_A_sparse2 = Sparse A with infrastructure features (bootstrap)

import numpy as np

def create_time_windows_dynamic(A, X, y, window_size=int):
    """
    Create time windows with dynamic adjacency for each window.

    Parameters:
        A : np.ndarray (N, N) - full adjacency matrix
        X : np.ndarray (N, F) - feature matrix
        y : np.ndarray (N, C) - one-hot labels
        window_size : int - number of nodes per window

    Returns:
        A_windows : list of np.ndarray (window_size, window_size)
        X_windows : list of np.ndarray (window_size, F)
        y_targets : list of np.ndarray (C,)
    """
    N = X.shape[0]
    A_windows, X_windows, y_targets = [], [], []

    for start in range(0, N - window_size):
        end = start + window_size

        # dynamic adjacency for just this window
        node_indices = np.arange(start, end)
        A_win = A[np.ix_(node_indices, node_indices)]  # submatrix

        # features for nodes in the window
        X_win = X[start:end]

        # label of the next node (31st)
        y_target = y[end]

        A_windows.append(A_win)
        X_windows.append(X_win)
        y_targets.append(y_target)

    return A_windows, X_windows, y_targets


A_windows1, X_windows1, y_targets1 = create_time_windows_dynamic(A_with_loops1, X1, y_one_hot1, window_size=4)
A_windows2, X_windows2, y_targets2 = create_time_windows_dynamic(A_with_loops2, X2, y_one_hot2, window_size=30)
w_A_windows1, w_X_windows1, w_y_targets1 = create_time_windows_dynamic(w_A_sparse1, X1_w, y_one_hot1, window_size=4)
w_A_windows2, w_X_windows2, w_y_targets2 = create_time_windows_dynamic(w_A_sparse2, X2_w, y_one_hot2, window_size=30)
i_A_windows1, i_X_windows1, i_y_targets1 = create_time_windows_dynamic(i_A_sparse1, X1_i, y_one_hot1, window_size=4)
i_A_windows2, i_X_windows2, i_y_targets2 = create_time_windows_dynamic(i_A_sparse2, X2_i, y_one_hot2, window_size=30)

# Convert lists of sparse matrices to lists of dense arrays (safely check if they have toarray method)
A_windows1 = [win.toarray() if hasattr(win, 'toarray') else win for win in A_windows1]
A_windows2 = [win.toarray() if hasattr(win, 'toarray') else win for win in A_windows2]
w_A_windows1 = [win.toarray() if hasattr(win, 'toarray') else win for win in w_A_windows1]
w_A_windows2 = [win.toarray() if hasattr(win, 'toarray') else win for win in w_A_windows2]
i_A_windows1 = [win.toarray() if hasattr(win, 'toarray') else win for win in i_A_windows1]
i_A_windows2 = [win.toarray() if hasattr(win, 'toarray') else win for win in i_A_windows2]

#Convert to numpy arrays
A_windows1 = np.array(A_windows1)
X_windows1 = np.array(X_windows1)
y_targets1 = np.array(y_targets1)

A_windows2 = np.array(A_windows2)
X_windows2 = np.array(X_windows2)
y_targets2 = np.array(y_targets2)

w_A_windows1 = np.array(w_A_windows1)
w_X_windows1 = np.array(w_X_windows1)
w_y_targets1 = np.array(w_y_targets1)

w_A_windows2 = np.array(w_A_windows2)
w_X_windows2 = np.array(w_X_windows2)
w_y_targets2 = np.array(w_y_targets2)

i_A_windows1 = np.array(i_A_windows1)
i_X_windows1 = np.array(i_X_windows1)
i_y_targets1 = np.array(i_y_targets1)

i_A_windows2 = np.array(i_A_windows2)
i_X_windows2 = np.array(i_X_windows2)
i_y_targets2 = np.array(i_y_targets2)


print("Original time window batches")
print(len(A_windows1), len(X_windows1), len(y_targets1))  #
print(A_windows1[0].shape, X_windows1[0].shape, y_targets1[0].shape)
print(A_windows1.shape, X_windows1.shape, y_targets1.shape)

print("Weather - Original time window batches")
print(len(w_A_windows1), len(w_X_windows1), len(w_y_targets1))  #
print(w_A_windows1[0].shape, w_X_windows1[0].shape, w_y_targets1[0].shape)
print(w_A_windows1.shape, w_X_windows1.shape, w_y_targets1.shape)

print("Infra - Original time window batches")
print(len(i_A_windows1), len(i_X_windows1), len(i_y_targets1))  #
print(i_A_windows1[0].shape, i_X_windows1[0].shape, i_y_targets1[0].shape)
print(i_A_windows1.shape, i_X_windows1.shape, i_y_targets1.shape)

print("Bootstrapped time window batches")
print(len(A_windows2), len(X_windows2), len(y_targets2))  # 9970
print(A_windows2[0].shape, X_windows2[0].shape, y_targets2[0].shape)
print(A_windows2.shape, X_windows2.shape, y_targets2.shape)

print("Weather - Bootstrapped time window batches")
print(len(w_A_windows2), len(w_X_windows2), len(w_y_targets2))  # 9970
print(w_A_windows2[0].shape, w_X_windows2[0].shape, w_y_targets2[0].shape)
print(w_A_windows2.shape, w_X_windows2.shape, w_y_targets2.shape)

print("Infra - Bootstrapped time window batches")
print(len(i_A_windows2), len(i_X_windows2), len(i_y_targets2))  # 9970
print(i_A_windows2[0].shape, i_X_windows2[0].shape, i_y_targets2[0].shape)
print(i_A_windows2.shape, i_X_windows2.shape, i_y_targets2.shape)

In [ ]:
print(A_windows1.shape)
print(A_windows2.shape)
print(w_A_windows1.shape)
print(w_A_windows2.shape)
print(i_A_windows1.shape)
print(i_A_windows2.shape)

In [ ]:
#Normalize the time windows
from spektral.utils import normalized_adjacency

def normalize_windows(A_windows, force=True):
    if not force:
        return A_windows.astype(np.float32)
    W = A_windows.shape[1]
    out = np.empty_like(A_windows, dtype=np.float32)
    for i, Aw in enumerate(A_windows):
        # add self-loops and normalize (Spektral helper)
        out[i] = normalized_adjacency(Aw + np.eye(W))
    return out

# Set to True if your windows are NOT yet normalized
A_windows1 = normalize_windows(A_windows1, force=True)
A_windows2 = normalize_windows(A_windows2, force=True)
w_A_windows1 = normalize_windows(w_A_windows1, force=True)
w_A_windows2 = normalize_windows(w_A_windows2, force=True)
i_A_windows1 = normalize_windows(i_A_windows1, force=True)
i_A_windows2 = normalize_windows(i_A_windows2, force=True)

In [ ]:
#Creating training, val, and test indices for time windows to create the splits

train1 = df1.loc['2014-01-01':'2020-07-01']
val1 = df1.loc['2020-07-02':'2021-07-01']
test1 = df1.loc['2021-07-02':]

train2 = bootstrapped_df.loc['2014-01-01':'2020-07-01']
val2 = bootstrapped_df.loc['2020-07-02':'2021-07-01']
test2 = bootstrapped_df.loc['2021-07-02':]

#Receive indices for train, val, test (in row number)

no_train1 = len(train1)
no_val1 = len(val1)
no_test1 = len(test1)
train_val1 = no_train1 + no_val1
val_test1 = (train_val1 + no_test1) - 1

no_train2 = len(train2)
no_val2 = len(val2)
no_test2 = len(test2)
train_val2 = no_train2 + no_val2
val_test2 = (train_val2 + no_test2) - 1

w_no_train1 = len(train1)
w_no_val1 = len(val1)
w_no_test1 = len(test1)
w_train_val1 = no_train1 + no_val1
w_val_test1 = (train_val1 + no_test1) - 1

w_no_train2 = len(train2)
w_no_val2 = len(val2)
w_no_test2 = len(test2)
w_train_val2 = no_train2 + no_val2
w_val_test2 = (train_val2 + no_test2) - 1

i_no_train1 = len(train1)
i_no_val1 = len(val1)
i_no_test1 = len(test1)
i_train_val1 = no_train1 + no_val1
i_val_test1 = (train_val1 + no_test1) - 1

i_no_train2 = len(train2)
i_no_val2 = len(val2)
i_no_test2 = len(test2)
i_train_val2 = no_train2 + no_val2
i_val_test2 = (train_val2 + no_test2) - 1

print("Original df: ",no_train1, no_val1, no_test1, train_val1, val_test1)
print("Bootstrapped df: ",no_train2, no_val2, no_test2, train_val2, val_test2)
print("Weather original: ",w_no_train1, w_no_val1, w_no_test1, w_train_val1, w_val_test1)
print("Weather bootstrapped: ",w_no_train2, w_no_val2, w_no_test2, w_train_val2, w_val_test2)
print("Infra original:",i_no_train1, i_no_val1, i_no_test1, i_train_val1, i_val_test1)
print("Infra bootstrapped:",i_no_train2, i_no_val2, i_no_test2, i_train_val2, i_val_test2)

idx_train1 = np.arange(0, (no_train1-1))
idx_val1   = np.arange(no_train1, no_train1 + no_val1)
idx_test1  = np.arange(no_train1 + no_val1, num_nodes1)

idx_train2 = np.arange(0, (no_train2-1))
idx_val2   = np.arange(no_train2, no_train2 + no_val2)
idx_test2  = np.arange(no_train2 + no_val2, num_nodes2)

w_idx_train1 = np.arange(0, (w_no_train1-1))
w_idx_val1   = np.arange(w_no_train1, w_no_train1 + w_no_val1)
w_idx_test1  = np.arange(w_no_train1 + w_no_val1, w_num_nodes1)

w_idx_train2 = np.arange(0, (w_no_train2-1))
w_idx_val2   = np.arange(w_no_train2, w_no_train2 + w_no_val2)
w_idx_test2  = np.arange(w_no_train2 + w_no_val2, w_num_nodes2)

i_idx_train1 = np.arange(0, (i_no_train1-1))
i_idx_val1   = np.arange(i_no_train1, i_no_train1 + i_no_val1)
i_idx_test1  = np.arange(i_no_train1 + i_no_val1, i_num_nodes1)

i_idx_train2 = np.arange(0, (i_no_train2-1))
i_idx_val2   = np.arange(i_no_train2, i_no_train2 + i_no_val2)
i_idx_test2  = np.arange(i_no_train2 + i_no_val2, i_num_nodes2)


In [ ]:
#Indices split

# Suppose X_windows, A_windows, y_targets are already created
#B = X_windows.shape[0]

# Indices for split
#train_end = int(0.7 * B)
#val_end = int(0.8 * B)

# Split windows while keeping all dimensions
X_train1 = X_windows1[:no_train1]
A_train1 = A_windows1[:no_train1]
y_train1 = y_targets1[:no_train1]

X_val1 = X_windows1[no_train1:train_val1]
A_val1 = A_windows1[no_train1:train_val1]
y_val1 = y_targets1[no_train1:train_val1]

X_test1 = X_windows1[train_val1:]
A_test1 = A_windows1[train_val1:]
y_test1 = y_targets1[train_val1:]

print("Original df Train shapes:", X_train1.shape, A_train1.shape, y_train1.shape)
print("Original df Val shapes:", X_val1.shape, A_val1.shape, y_val1.shape)
print("Original df Test shapes:", X_test1.shape, A_test1.shape, y_test1.shape)

X_train2 = X_windows2[:no_train2]
A_train2 = A_windows2[:no_train2]
y_train2 = y_targets2[:no_train2]

X_val2 = X_windows2[no_train2:train_val2]
A_val2 = A_windows2[no_train2:train_val2]
y_val2 = y_targets2[no_train2:train_val2]

X_test2 = X_windows2[train_val2:]
A_test2 = A_windows2[train_val2:]
y_test2 = y_targets2[train_val2:]

print("Bootstrapped df Train shapes:", X_train2.shape, A_train2.shape, y_train2.shape)
print("Bootstrapped df Val shapes:", X_val2.shape, A_val2.shape, y_val2.shape)
print("Bootstrapped df Test shapes:", X_test2.shape, A_test2.shape, y_test2.shape)
print("------------------------------------------")

w_X_train1 = w_X_windows1[:w_no_train1]
w_A_train1 = w_A_windows1[:w_no_train1]
w_y_train1 = w_y_targets1[:w_no_train1]

w_X_val1 = w_X_windows1[w_no_train1:w_train_val1]
w_A_val1 = w_A_windows1[w_no_train1:w_train_val1]
w_y_val1 = w_y_targets1[w_no_train1:w_train_val1]

w_X_test1 = w_X_windows1[w_train_val1:]
w_A_test1 = w_A_windows1[w_train_val1:]
w_y_test1 = w_y_targets1[w_train_val1:]

print("Weather - Original df Train shapes:", w_X_train1.shape, w_A_train1.shape, w_y_train1.shape)
print("Weather - Original df Val shapes:", w_X_val1.shape, w_A_val1.shape, w_y_val1.shape)
print("Weather - Original df Test shapes:", w_X_test1.shape, w_A_test1.shape, w_y_test1.shape)

w_X_train2 = w_X_windows2[:w_no_train2]
w_A_train2 = w_A_windows2[:w_no_train2]
w_y_train2 = w_y_targets2[:w_no_train2]

w_X_val2 = w_X_windows2[w_no_train2:w_train_val2]
w_A_val2 = A_windows2[w_no_train2:w_train_val2]
w_y_val2 = w_y_targets2[w_no_train2:w_train_val2]

w_X_test2 = w_X_windows2[w_train_val2:]
w_A_test2 = A_windows2[w_train_val2:]
w_y_test2 = w_y_targets2[w_train_val2:]

print("Weather - Bootstrapped df Train shapes:", w_X_train2.shape, w_A_train2.shape, w_y_train2.shape)
print("Weather - Bootstrapped df Val shapes:", w_X_val2.shape, w_A_val2.shape, w_y_val2.shape)
print("Weather - Bootstrapped df Test shapes:", w_X_test2.shape, w_A_test2.shape, w_y_test2.shape)
print("------------------------------------------")

i_X_train1 = i_X_windows1[:i_no_train1]
i_A_train1 = i_A_windows1[:i_no_train1]
i_y_train1 = i_y_targets1[:i_no_train1]

i_X_val1 = i_X_windows1[i_no_train1:i_train_val1]
i_A_val1 = i_A_windows1[i_no_train1:i_train_val1]
i_y_val1 = i_y_targets1[i_no_train1:i_train_val1]

i_X_test1 = i_X_windows1[i_train_val1:]
i_A_test1 = i_A_windows1[i_train_val1:]
i_y_test1 = i_y_targets1[i_train_val1:]

print("Infra - Original df Train shapes:", i_X_train1.shape, i_A_train1.shape, i_y_train1.shape)
print("Infra - Original df Val shapes:", i_X_val1.shape, i_A_val1.shape, i_y_val1.shape)
print("Infra - Original df Test shapes:", i_X_test1.shape, i_A_test1.shape, i_y_test1.shape)

i_X_train2 = i_X_windows2[:i_no_train2]
i_A_train2 = i_A_windows2[:i_no_train2]
i_y_train2 = i_y_targets2[:i_no_train2]

i_X_val2 = i_X_windows2[i_no_train2:i_train_val2]
i_A_val2 = i_A_windows2[i_no_train2:i_train_val2]
i_y_val2 = i_y_targets2[i_no_train2:i_train_val2]

i_X_test2 = i_X_windows2[i_train_val2:]
i_A_test2 = i_A_windows2[i_train_val2:]
i_y_test2 = i_y_targets2[i_train_val2:]

print("Infra - Bootstrapped df Train shapes:", i_X_train2.shape, i_A_train2.shape, i_y_train2.shape)
print("Infra - Bootstrapped df Val shapes:", i_X_val2.shape, i_A_val2.shape, i_y_val2.shape)
print("Infra - Bootstrapped df Test shapes:", i_X_test2.shape, i_A_test2.shape, i_y_test2.shape)




GAT-BLSTM

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Bidirectional, LSTM, Dropout, Dense, LeakyReLU, TimeDistributed
from spektral.layers import GATConv
from tensorflow.keras.optimizers import Adam


def build_gat_blstm(window_size=4, n_features=6, n_classes=5, #CHANGE WINDOW SIZE FOR BOOT
                    gat_channels1=16, gat_channels2=64,
                    attn_heads1=3, attn_heads2=2, dropout=0.35491190814353074,
                    lstm_units1=16, lstm_units2=64, l2_reg=0.008080657923512494): #CHANGE NUMBER OF FEATURES WITH OTHER SUBSETS
    """
    Model architecture:
    - Input: Node features (X) with shape (batch, N, F) and adjacency (A) with shape (batch, N, N).
    - GAT Block: Two GAT layers with kernel regularizer and LeakyReLU.
    - BLSTM Block: Two stacked Bidirectional LSTMs across time windows.
    - Dropout: 0.5 applied after BLSTM layers.
    - Dense: Softmax output for classification into 5 classes.
    """

    # (a) Inputs
    X_in = Input(shape=(window_size, n_features), name="X_in")
    A_in = Input(shape=(window_size, window_size), name="A_in")

    # (b) Two GAT layers
    gat1 = GATConv(
        channels=gat_channels1,
        attn_heads=attn_heads1,
        concat=True,
        activation=None,
        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)
    )([X_in, A_in])
    gat1 = LeakyReLU()(gat1)

    gat2 = GATConv(
        channels=gat_channels2,
        attn_heads=attn_heads2,
        concat=False,
        activation=None,
        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)
    )([gat1, A_in])
    gat2 = LeakyReLU()(gat2)

    # (c) Two BLSTM layers process temporal dependencies
    blstm1 = Bidirectional(LSTM(lstm_units1, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))(gat2)
    blstm1 = LeakyReLU()(blstm1)

    blstm2 = Bidirectional(LSTM(lstm_units2, return_sequences=False))(blstm1)
    blstm2 = LeakyReLU()(blstm2)

    # (d) Dropout and classification output
    drop = Dropout(dropout)(blstm2)
    out = Dense(n_classes, activation="softmax")(drop)

    return Model(inputs=[X_in, A_in], outputs=out)


In [ ]:
#Compilation

# Instantiate the corrected GAT-BLSTM model
gat_blstm_model = build_gat_blstm(
    window_size=i_X_windows1.shape[1], #CHANGE
    n_features=i_X_windows1.shape[2], #CHANGE
    n_classes=i_y_targets1.shape[1] #CHANGE
)

# Compile the model
gat_blstm_model.compile(
    optimizer=Adam(learning_rate=0.001156112303740331), #CHANGE
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

gat_blstm_model.summary()

# Define early stopping
early_stop_gat = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)


In [ ]:
import optuna
import tensorflow as tf

# Bayesian Optimization of hyperparameters (Optuna Search) for GAT-BLSTM

def objective_gat_blstm(trial):
    # --- Define search space ---
    gat_channels1 = trial.suggest_categorical('gat_channels1', [8, 16, 32, 64])
    gat_channels2 = trial.suggest_categorical('gat_channels2', [8, 16, 32, 64])
    attn_heads1 = trial.suggest_categorical('attn_heads1', [1, 2, 3])
    attn_heads2 = trial.suggest_categorical('attn_heads2', [1, 2, 3])
    lstm_units1 = trial.suggest_categorical('lstm_units1', [8, 16, 32, 64])
    lstm_units2 = trial.suggest_categorical('lstm_units2', [8, 16, 32, 64])
    dropout = trial.suggest_float('dropout', 0.1, 0.7)
    l2_reg = trial.suggest_float('l2_reg', 1e-4, 1e-1, log=True)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    # Build model using selected hyperparameters
    model = build_gat_blstm(
        window_size=i_X_windows1.shape[1], #CHANGE
        n_features=i_X_windows1.shape[2], #CHANGE
        n_classes=i_y_targets1.shape[1], #CHANGE
        gat_channels1=gat_channels1,
        gat_channels2=gat_channels2,
        attn_heads1=attn_heads1,
        attn_heads2=attn_heads2,
        lstm_units1=lstm_units1,
        lstm_units2=lstm_units2,
        l2_reg=l2_reg,
        dropout = dropout
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Fit on training data with early stopping based on val_loss
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    history = model.fit(
        [i_X_train1, i_A_train1], i_y_train1, #CHANGE
        validation_data=([i_X_val1, i_A_val1], i_y_val1), #CHANGE
        batch_size=64,
        epochs=20,
        verbose=0,
        callbacks=[early_stop]
    )

    # Return the best (minimum) validation loss achieved in this trial
    val_loss = min(history.history['val_loss'])
    val_accuracy = max(history.history['val_accuracy'])

    trial.set_user_attr('accuracy', val_accuracy)
    print(f"Trial {trial.number} -> val_loss: {val_loss:.4f}, val_accuracy: {val_accuracy:.4f}")

    return val_loss

# --- Run Bayesian Optimization Study to MINIMIZE loss ---
study_gat_blstm = optuna.create_study(direction='minimize')
study_gat_blstm.optimize(objective_gat_blstm, n_trials=20) # Adjust n_trials as needed

print("Optimization finished.")
print("Best trial (lowest loss):", study_gat_blstm.best_trial.params)
print("Best validation loss:", study_gat_blstm.best_value)
print("Best validation accuracy:", study_gat_blstm.best_trial.user_attrs['accuracy'])

In [ ]:
#Best parameters

#Original df
#Best trial (lowest loss): {'gat_channels1': 64, 'gat_channels2': 32, 'attn_heads1': 3, 'attn_heads2': 2, 'lstm_units1': 32, 'lstm_units2': 32, 'dropout': 0.6670369719336273, 'l2_reg': 0.0005992299576835011, 'learning_rate': 0.0012861565875564027}
#Best validation loss: 1.2165640592575073
#Best validation accuracy: 0.5169946551322937

#Bootstrapped df

#Original - w
#Best trial (lowest loss): {'gat_channels1': 64, 'gat_channels2': 32, 'attn_heads1': 3, 'attn_heads2': 3, 'lstm_units1': 8, 'lstm_units2': 64, 'dropout': 0.31336035867868445, 'l2_reg': 0.0003410677156741509, 'learning_rate': 0.0016304785571576612}
#Best validation loss: 1.2489179372787476
#Best validation accuracy: 0.5080500841140747

#Bootstrapped - w

#Original - i
#Best trial (lowest loss): {'gat_channels1': 16, 'gat_channels2': 64, 'attn_heads1': 3, 'attn_heads2': 2, 'lstm_units1': 16, 'lstm_units2': 64, 'dropout': 0.35491190814353074, 'l2_reg': 0.008080657923512494, 'learning_rate': 0.001156112303740331}
#Best validation loss: 1.3882654905319214
#Best validation accuracy: 0.4812164604663849

#Bootstrapped - i

In [ ]:
# Train the model

print("Training GAT-BLSTM model...")
history_gat_blstm = gat_blstm_model.fit(
    [i_X_train1, i_A_train1], i_y_train1, #CHANGE
    batch_size=64,
    epochs=100,
    validation_data=([i_X_val1, i_A_val1], i_y_val1), #CHANG
    verbose=1,
    callbacks=[early_stop_gat]
)

# Evaluate the model on the test set
print("\nEvaluating GAT-BLSTM model on Test Set...")
test_loss, test_acc = gat_blstm_model.evaluate([i_X_test1, i_A_test1], i_y_test1, verbose=1) #CHANGE
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")


In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history_gat_blstm.history['accuracy'], label='Train Accuracy')
plt.plot(history_gat_blstm.history['val_accuracy'], label='Val Accuracy')
plt.title('GAT-BLSTM Model Accuracy - Original Dataset (Full feature set)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history_gat_blstm.history['loss'], label='Train Loss')
plt.plot(history_gat_blstm.history['val_loss'], label='Val Loss')
plt.title('GAT-BLSTM Model Loss - Original Dataset (Full feature set)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Convert history to DataFrame
gatblstm_df_traininglog = pd.DataFrame(history_gat_blstm.history)

# Save to CSV
gatblstm_df_traininglog.to_csv("/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/gatblstm_df1_i_traininglog.csv", index=False) #CHANGE ACCORDING TO FULL FEATURE SET, W, or I

gat_blstm_model.save(filepath='/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/Ablation_models/gatblstm_df1_i.keras', include_optimizer=True)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
import seaborn as sns

# -----------------------------
# 4) Predictions
# -----------------------------

y_pred_probs = gat_blstm_model.predict([i_X_test1, w_A_test1])          # CHANGE softmax probabilities
y_pred = np.argmax(y_pred_probs, axis=1)                # predicted classes
y_true = np.argmax(i_y_test1, axis=1)                      # CHANGE true classes

class_names = list(label_enc1_mapping.keys())
# -----------------------------
# 5) Classification report
# -----------------------------
print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=2, target_names=class_names, zero_division=0))

#Balanced Accuracy
balanced_acc = balanced_accuracy_score(y_true, y_pred)
print(f"Balanced Accuracy on Test Set: {balanced_acc:.4f}")

#Weighted F1 Score
w_f1_score = f1_score(y_true, y_pred, average='weighted', zero_division=0)
print(f"Weighted F1 Score on Test Set: {w_f1_score:.4f}")
# -----------------------------
# 6) Confusion matrix
# -----------------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix GAT-BLSTM - Original dataset (IP features)")
plt.show()

In [ ]:
# ==========================================
# 3. GAT-BLSTM Model Evaluation
# ==========================================

n_iterations = 10
sample_size = 4*len(i_y_test1) #CHANGE

print("\nEvaluating GAT-BLSTM...")
metrics_list_gat_blstm = []
for i in range(n_iterations):
    boot_idx = resample(np.arange(len(i_y_test1)), n_samples=sample_size, random_state=42 + i, replace=True) #CHANGE
    X_test_boot = i_X_test1[boot_idx] #CHANGE
    A_test_boot = i_A_test1[boot_idx] #CHANGE
    y_true_boot = np.argmax(i_y_test1[boot_idx], axis=1) #CHANGE

    # Predict using the trained GAT-BLSTM model ('gat_blstm_model' variable)
    y_pred_probs_boot = gat_blstm_model.predict([X_test_boot, A_test_boot], verbose=0)
    y_pred_boot = np.argmax(y_pred_probs_boot, axis=1)

    precision = precision_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    recall = recall_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    f1 = f1_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_true_boot, y_pred_boot)
    balanced_acc = balanced_accuracy_score(y_true_boot, y_pred_boot)

    precision_per_class = precision_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))
    recall_per_class = recall_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))

    iter_metrics = {
        'iteration': i + 1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }
    for idx, c_name in enumerate(c_names):
        iter_metrics[f'precision_{c_name}'] = precision_per_class[idx]
        iter_metrics[f'recall_{c_name}'] = recall_per_class[idx]

    metrics_list_gat_blstm.append(iter_metrics)

bootstrap_metrics_gat_blstm_df = pd.DataFrame(metrics_list_gat_blstm)
display(bootstrap_metrics_gat_blstm_df)

mean_metrics_gat_blstm = bootstrap_metrics_gat_blstm_df.drop(columns=['iteration']).mean()
std_metrics_gat_blstm = bootstrap_metrics_gat_blstm_df.drop(columns=['iteration']).std()
summary_gat_blstm_df = pd.DataFrame({'Mean': mean_metrics_gat_blstm, 'Standard Deviation': std_metrics_gat_blstm})
print("\nMean and Standard Deviation of Evaluation Metrics GAT-BLSTM (10 iterations):")
display(summary_gat_blstm_df)



In [ ]:
y_pred_probs = pd.DataFrame(y_pred_probs, columns=class_names)
#y_pred.to_csv('drive/MyDrive/PhD data analysis/Paper1/testprob_bootstrap_gcnblstm.csv', header=True, index=True)
y_pred_probs.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/pred_df1_i_gatblstm.csv', header=True, index=True)
y_pred_probs.shape
summary_gat_blstm_df.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/gatblstm_df1_i_summary_metrics.csv', index=True)

GCN-BLSTM training

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Bidirectional, LSTM, Dropout, Dense, LeakyReLU, TimeDistributed
from spektral.layers import GCNConv



def build_gcn_blstm(window_size=4, n_features=6, n_classes=5,
                    gcn_units1=8, gcn_units2=32, dropout=0.456859756018133,
                    lstm_units1=8, lstm_units2=16, l2_reg=0.00011134159677493278): #CHANGE NUMBER OF FEATURES WITH OTHER SUBSETS
    """
    Model architecture:
    - Input: Node features (X) with shape (batch, N, F) and adjacency (A) with shape (batch, N, N).
    - GCN Block: Two GCN layers with kernel regularizer and LeakyReLU.
    - BLSTM Block: Two stacked Bidirectional LSTMs across time windows.
    - Dropout: 0.5 applied after BLSTM layers.
    - Dense: Softmax output for classification into 5 classes.
    """

    # (a) Inputs
    X_in = Input(shape=(window_size, n_features), name="X_in")
    A_in = Input(shape=(window_size, window_size), name="A_in")

    # (b) Two GCN layers
    gcn1 = GCNConv(
        gcn_units1,
        activation=None,
        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)
    )([X_in, A_in])
    gcn1 = LeakyReLU()(gcn1)

    gcn2 = GCNConv(
        gcn_units2,
        activation=None,
        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)
    )([gcn1, A_in])
    gcn2 = LeakyReLU()(gcn2)

    # (c) Two BLSTM layers process temporal dependencies
    blstm1 = Bidirectional(LSTM(lstm_units1, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))(gcn2)
    blstm1 = LeakyReLU()(blstm1)

    blstm2 = Bidirectional(LSTM(lstm_units2, return_sequences=False))(blstm1)
    blstm2 = LeakyReLU()(blstm2)

    # (d) Dropout and classification output
    drop = Dropout(dropout)(blstm2)
    out = Dense(n_classes, activation="softmax")(drop)

    return Model(inputs=[X_in, A_in], outputs=out)

In [ ]:
import optuna
import tensorflow as tf

#Bayesian Optimization of hyperparameters (Optuna Search)

def objective(trial):
    # --- Define search space ---
    gcn_units1 = trial.suggest_categorical('gcn_units1', [8, 16, 32, 64])
    gcn_units2 = trial.suggest_categorical('gcn_units2', [8, 16, 32, 64])
    lstm_units1 = trial.suggest_categorical('lstm_units1', [8, 16, 32, 64])
    lstm_units2 = trial.suggest_categorical('lstm_units2', [8, 16, 32, 64])
    dropout = trial.suggest_float('dropout', 0.1, 0.7)
    l2_reg = trial.suggest_float('l2_reg', 1e-4, 1e-1, log=True)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    # Build model using selected hyperparameters
    model = build_gcn_blstm(
        window_size=i_X_windows1.shape[1], #CHANGE
        n_features=i_X_windows1.shape[2], #CHANGE
        n_classes=i_y_targets1.shape[1], #CHANGE
        gcn_units1=gcn_units1,
        gcn_units2=gcn_units2,
        lstm_units1=lstm_units1,
        lstm_units2=lstm_units2,
        l2_reg=l2_reg,
        dropout=dropout
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Fit on training data with early stopping based on val_loss
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    history = model.fit(
        [i_X_train1, i_A_train1], i_y_train1, #CHANGE
        validation_data=([i_X_val1, i_A_val1], i_y_val1), #CHANGE
        batch_size=64,
        epochs=50,
        verbose=0,
        callbacks=[early_stop]
    )

    # Return the best (minimum) validation loss achieved in this trial
    val_loss = min(history.history['val_loss'])
    val_accuracy = max(history.history['val_accuracy'])

    trial.set_user_attr('accuracy', val_accuracy)
    print(f"Trial {trial.number} -> val_loss: {val_loss:.4f}, val_accuracy: {val_accuracy:.4f}")

    return val_loss

# --- Run Bayesian Optimization Study to MINIMIZE loss ---
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

print("Optimization finished.")
print("Best trial (lowest loss):", study.best_trial.params)
print("Best validation loss:", study.best_value)
print("Best validation accuracy:", study.best_trial.user_attrs['accuracy'])

In [ ]:
#Compile

model = build_gcn_blstm(window_size=i_X_windows1.shape[1], n_features=i_X_windows1.shape[2], n_classes=i_y_targets1.shape[1]) #CHANGE

model.compile(optimizer=Adam(learning_rate=0.0021951952746825714),  #CHANGE
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

In [ ]:
#Best parameters

#Original df
#Best trial (lowest loss): {'gcn_units1': 16, 'gcn_units2': 8, 'lstm_units1': 16, 'lstm_units2': 64, 'dropout': 0.3423143429849724, 'l2_reg': 0.00034989384768220686, 'learning_rate': 0.004373459575310068}
#Best validation loss: 1.1963735818862915
#Best validation accuracy: 0.5205724239349365

#Bootstrapped df

#Original - w
#Best trial (lowest loss): {'gcn_units1': 16, 'gcn_units2': 8, 'lstm_units1': 32, 'lstm_units2': 8, 'dropout': 0.41605599052721726, 'l2_reg': 0.0006039582068094363, 'learning_rate': 0.001294395011273408}
#Best validation loss: 1.1918762922286987
#Best validation accuracy: 0.5152057409286499

#Bootstrapped - w

#Original - i

#Bootstrapped - i
#Best trial (lowest loss): {'gcn_units1': 8, 'gcn_units2': 32, 'lstm_units1': 8, 'lstm_units2': 16, 'dropout': 0.456859756018133, 'l2_reg': 0.00011134159677493278, 'learning_rate': 0.0021951952746825714}
#Best validation loss: 1.3830738067626953
#Best validation accuracy: 0.4830053746700287

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

# Train the model
history = model.fit([i_X_train1, i_A_train1], i_y_train1, #CHANGE
batch_size=64,
epochs=100,
validation_data=([i_X_val1, i_A_val1], i_y_val1), #CHANGE
verbose=1, callbacks=[early_stop])


# Test the model
test_loss, test_acc = model.evaluate([i_X_test1, i_A_test1], i_y_test1, verbose=1) #CHANGE
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('GCN-BLSTM Model Accuracy - Original Dataset (Full feature set)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('GCN-BLSTM Model Loss - Original Dataset (Full feature set)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()


GCN-BLSTM testing + evaluation metrics

In [ ]:
# Convert history to DataFrame
gcnblstm_df_traininglog = pd.DataFrame(history.history)

# Save to CSV
gcnblstm_df_traininglog.to_csv("/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/gcnblstm_df1_i_traininglog.csv", index=False) #CHANGE ACCORDING TO FULL FEATURE SET, W, or I

model.save(filepath='/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/Ablation_models/gcnblstm_df1_i.keras', include_optimizer=True)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
import seaborn as sns

# -----------------------------
# 4) Predictions
# -----------------------------

y_pred_probs = model.predict([i_X_test1, i_A_test1])          # CHANGE softmax probabilities
y_pred = np.argmax(y_pred_probs, axis=1)                # predicted classes
y_true = np.argmax(i_y_test1, axis=1)                      # CHANGE true classes

class_names = list(label_enc1_mapping.keys())
# -----------------------------
# 5) Classification report
# -----------------------------
print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=2, target_names=class_names, zero_division=0))

#Balanced Accuracy
balanced_acc = balanced_accuracy_score(y_true, y_pred)
print(f"Balanced Accuracy on Test Set: {balanced_acc:.4f}")

#Weighted F1 Score
w_f1_score = f1_score(y_true, y_pred, average='weighted', zero_division=0)
print(f"Weighted F1 Score on Test Set: {w_f1_score:.4f}")
# -----------------------------
# 6) Confusion matrix
# -----------------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix GCN-BLSTM - Original dataset (IP features)")
plt.show()

In [ ]:
# ==========================================
# 2. GCN-BLSTM Model Evaluation
# ==========================================

n_iterations = 10
sample_size = 4*len(i_y_test1) #CHANGE


print("\nEvaluating GCN-BLSTM...")
metrics_list_gcn_blstm = []
for i in range(n_iterations):
    boot_idx = resample(np.arange(len(i_y_test1)), n_samples=sample_size, random_state=42 + i, replace=True) #CHANGE
    X_test_boot = i_X_test1[boot_idx] #CHANGE
    A_test_boot = i_A_test1[boot_idx] #CHANGE
    y_true_boot = np.argmax(i_y_test1[boot_idx], axis=1) #CHANGE

    # Predict using the trained GCN-BLSTM model ('model' variable)
    y_pred_probs_boot = model.predict([X_test_boot, A_test_boot], verbose=0)
    y_pred_boot = np.argmax(y_pred_probs_boot, axis=1)

    precision = precision_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    recall = recall_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    f1 = f1_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_true_boot, y_pred_boot)
    balanced_acc = balanced_accuracy_score(y_true_boot, y_pred_boot)

    precision_per_class = precision_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))
    recall_per_class = recall_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))

    iter_metrics = {
        'iteration': i + 1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }
    for idx, c_name in enumerate(c_names):
        iter_metrics[f'precision_{c_name}'] = precision_per_class[idx]
        iter_metrics[f'recall_{c_name}'] = recall_per_class[idx]

    metrics_list_gcn_blstm.append(iter_metrics)

bootstrap_metrics_gcn_blstm_df = pd.DataFrame(metrics_list_gcn_blstm)
display(bootstrap_metrics_gcn_blstm_df)

mean_metrics_gcn_blstm = bootstrap_metrics_gcn_blstm_df.drop(columns=['iteration']).mean()
std_metrics_gcn_blstm = bootstrap_metrics_gcn_blstm_df.drop(columns=['iteration']).std()
summary_gcn_blstm_df = pd.DataFrame({'Mean': mean_metrics_gcn_blstm, 'Standard Deviation': std_metrics_gcn_blstm})
print("\nMean and Standard Deviation of Evaluation Metrics GCN-BLSTM (10 iterations):")
display(summary_gcn_blstm_df)


In [ ]:
y_pred = pd.DataFrame(y_pred_probs, columns=class_names)
#y_pred.to_csv('drive/MyDrive/PhD data analysis/Paper1/testprob_bootstrap_gcnblstm.csv', header=True, index=True)
y_pred.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/pred_df1_i_gcnblstm.csv', header=True, index=True)
y_pred.shape
summary_gcn_blstm_df.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/gcnblstm_df1_i_summary_metrics.csv', index=True)

# GCN/GAT

Graph construction/loading for data splitting

In [ ]:
from spektral.data import SingleLoader

#num_nodes1 = graph_df_all.n_nodes
#num_nodes2 = graph_boot_all.n_nodes
#w_num_nodes1 = w_graph_df.n_nodes
#w_num_nodes2 = w_graph_boot.n_nodes
#i_num_nodes1 = i_graph_df.n_nodes
#i_num_nodes2 = i_graph_boot.n_nodes


#idx_train1 = np.arange(0, (no_train1-1))
#idx_val1   = np.arange(no_train1, no_train1 + no_val1)
#idx_test1  = np.arange(no_train1 + no_val1, num_nodes1)

#idx_train2 = np.arange(0, (no_train2-1))
#idx_val2   = np.arange(no_train2, no_train2 + no_val2)
#idx_test2  = np.arange(no_train2 + no_val2, num_nodes2)

#w_idx_train1 = np.arange(0, (w_no_train1-1))
#w_idx_val1   = np.arange(w_no_train1, w_no_train1 + w_no_val1)
#w_idx_test1  = np.arange(w_no_train1 + w_no_val1, w_num_nodes1)

#w_idx_train2 = np.arange(0, (w_no_train2-1))
#w_idx_val2   = np.arange(w_no_train2, w_no_train2 + w_no_val2)
#w_idx_test2  = np.arange(w_no_train2 + w_no_val2, w_num_nodes2)

#i_idx_train1 = np.arange(0, (i_no_train1-1))
#i_idx_val1   = np.arange(i_no_train1, i_no_train1 + i_no_val1)
#i_idx_test1  = np.arange(i_no_train1 + i_no_val1, i_num_nodes1)

#i_idx_train2 = np.arange(0, (i_no_train2-1))
#i_idx_val2   = np.arange(i_no_train2, i_no_train2 + i_no_val2)
#i_idx_test2  = np.arange(i_no_train2 + i_no_val2, i_num_nodes2)

# Create masks for splitting
train_mask_tf1 = tf.constant(np.isin(np.arange(num_nodes1), idx_train1), dtype=tf.float32)
val_mask_tf1 = tf.constant(np.isin(np.arange(num_nodes1), idx_val1), dtype=tf.float32)
test_mask_tf1  = tf.constant(np.isin(np.arange(num_nodes1), idx_test1), dtype=tf.float32)

train_mask_tf2 = tf.constant(np.isin(np.arange(num_nodes2), idx_train2), dtype=tf.float32)
val_mask_tf2 = tf.constant(np.isin(np.arange(num_nodes2), idx_val2), dtype=tf.float32)
test_mask_tf2  = tf.constant(np.isin(np.arange(num_nodes2), idx_test2), dtype=tf.float32)

w_train_mask_tf1 = tf.constant(np.isin(np.arange(w_num_nodes1), w_idx_train1), dtype=tf.float32)
w_val_mask_tf1 = tf.constant(np.isin(np.arange(w_num_nodes1), w_idx_val1), dtype=tf.float32)
w_test_mask_tf1  = tf.constant(np.isin(np.arange(w_num_nodes1), w_idx_test1), dtype=tf.float32)

w_train_mask_tf2 = tf.constant(np.isin(np.arange(w_num_nodes2), w_idx_train2), dtype=tf.float32)
w_val_mask_tf2 = tf.constant(np.isin(np.arange(w_num_nodes2), w_idx_val2), dtype=tf.float32)
w_test_mask_tf2  = tf.constant(np.isin(np.arange(w_num_nodes2), w_idx_test2), dtype=tf.float32)

i_train_mask_tf1 = tf.constant(np.isin(np.arange(i_num_nodes1), i_idx_train1), dtype=tf.float32)
i_val_mask_tf1 = tf.constant(np.isin(np.arange(i_num_nodes1), i_idx_val1), dtype=tf.float32)
i_test_mask_tf1  = tf.constant(np.isin(np.arange(i_num_nodes1), i_idx_test1), dtype=tf.float32)

i_train_mask_tf2 = tf.constant(np.isin(np.arange(i_num_nodes2), i_idx_train2), dtype=tf.float32)
i_val_mask_tf2 = tf.constant(np.isin(np.arange(i_num_nodes2), i_idx_val2), dtype=tf.float32)
i_test_mask_tf2  = tf.constant(np.isin(np.arange(i_num_nodes2), i_idx_test2), dtype=tf.float32)

# Create SingleLoader for each set (assuming full-batch training)
train_loader1 = SingleLoader(dataset1, sample_weights=train_mask_tf1)
val_loader1   = SingleLoader(dataset1, sample_weights=val_mask_tf1)
test_loader1  = SingleLoader(dataset1, sample_weights=test_mask_tf1)

train_loader2 = SingleLoader(dataset2, sample_weights=train_mask_tf2)
val_loader2   = SingleLoader(dataset2, sample_weights=val_mask_tf2)
test_loader2  = SingleLoader(dataset2, sample_weights=test_mask_tf2)

w_train_loader1 = SingleLoader(dataset3, sample_weights=w_train_mask_tf1)
w_val_loader1   = SingleLoader(dataset3, sample_weights=w_val_mask_tf1)
w_test_loader1  = SingleLoader(dataset3, sample_weights=w_test_mask_tf1)

w_train_loader2 = SingleLoader(dataset4, sample_weights=w_train_mask_tf2)
w_val_loader2   = SingleLoader(dataset4, sample_weights=w_val_mask_tf2)
w_test_loader2  = SingleLoader(dataset4, sample_weights=w_test_mask_tf2)

i_train_loader1 = SingleLoader(dataset5, sample_weights=i_train_mask_tf1)
i_val_loader1   = SingleLoader(dataset5, sample_weights=i_val_mask_tf1)
i_test_loader1  = SingleLoader(dataset5, sample_weights=i_test_mask_tf1)

i_train_loader_tf2 = SingleLoader(dataset5, sample_weights=i_train_mask_tf2)
i_val_loader_tf2   = SingleLoader(dataset5, sample_weights=i_val_mask_tf2)
i_test_loader_tf2  = SingleLoader(dataset5, sample_weights=i_test_mask_tf2)

# Proper masked loss (Sparse CE example; use CategoricalCE if you one-hot)
def masked_sparse_cce(y_true, y_pred, mask_tf):
    per_node = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)  # shape (N,)
    per_node = tf.cast(per_node, tf.float32) * mask_tf                          # zero out unmasked
    return tf.reduce_sum(per_node) / tf.reduce_sum(mask_tf)                     # average, not sum!

GCN training

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.layers import LeakyReLU
from spektral.layers import GCNConv
import tensorflow as tf



# Define the number of output classes
# C = dataset.n_labels # This is incorrect, dataset.n_labels seems to be the number of nodes
C = 5 # Corrected to the number of incident types

import tensorflow as tf

class GCN(tf.keras.Model):
    def __init__(self, channel1, channel2, n_classes, l2_reg=0.00011790213240045454, dropout_rate2=0.5038747874634564):
        super().__init__()
        self.channel1 = channel1
        self.channel2 = channel2  # Store parameters for get_config
        self.n_classes = n_classes
        self.l2_reg = l2_reg
   #     self.dropout_rate1 = dropout_rate1 # Keeping this to match original signature
        self.dropout_rate2 = dropout_rate2

        self.gcn1 = GCNConv(channel1,
                            activation=tf.keras.layers.LeakyReLU(negative_slope=0.2))#,
        #                    kernel_regularizer=tf.keras.regularizers.l2(l2_reg))
       # self.dropout1 = tf.keras.layers.Dropout(dropout_rate1)

        self.gcn2 = GCNConv(channel2,
                            activation=tf.keras.layers.LeakyReLU(negative_slope=0.2))#,
          #                  kernel_regularizer=tf.keras.regularizers.l2(l2_reg))
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate2)

       # Dense classifier head instead of GCNConv
        self.dense_out = tf.keras.layers.Dense(n_classes, activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg))

    def call(self, inputs, training=True):
        x, a = inputs

        x = self.gcn1([x, a])
      #  x = self.dropout1(x, training=training)

        x = self.gcn2([x, a])
        x = self.dropout2(x, training=training)

     # Final dense classifier
        x = self.dense_out(x)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "channel1": self.channel1,
            "channel2": self.channel2,
            "n_classes": self.n_classes,
            "l2_reg": self.l2_reg,
        #    "dropout_rate1": self.dropout_rate1,
            "dropout_rate2": self.dropout_rate2,
        })
        return config


model = GCN(channel1=64, channel2=64, n_classes=C)

#model.compile(
#    optimizer=Adam(learning_rate=1e-2),
#    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
#    weighted_metrics=["accuracy"],
#)

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0034436855930526596),
    loss=lambda y_true, y_pred: masked_sparse_cce(y_true, y_pred, i_train_mask_tf1), # CHANGE  Otherwise train_mask_tensor CHANGE WITH CLUSTER
    metrics=["accuracy"])
  #  jit_compile=False # Disable XLA compilation


early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

In [ ]:
import optuna
import tensorflow as tf

# Bayesian Optimization of hyperparameters (Optuna Search) for GCN

def objective_gcn(trial):
    # --- Define search space ---
    channel1 = trial.suggest_categorical('channel1', [8, 16, 32, 64])
    channel2 = trial.suggest_categorical('channel2', [8, 16, 32, 64])
    l2_reg = trial.suggest_float('l2_reg', 1e-4, 1e-1, log=True)
    dropout_rate2 = trial.suggest_float('dropout_rate2', 0.1, 0.7)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    # Build model using selected hyperparameters
    model = GCN(
        channel1=channel1,
        channel2=channel2,
        n_classes=5,  # C = 5 classes
        l2_reg=l2_reg,
        dropout_rate2=dropout_rate2
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=lambda y_true, y_pred: masked_sparse_cce(y_true, y_pred, i_train_mask_tf1), #CHANGE
        metrics=["accuracy"]
    )

    # Fit on training data with early stopping based on val_loss
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model.fit(
        i_train_loader1.load(),
        steps_per_epoch=i_train_loader1.steps_per_epoch, #CHANGE
        validation_data=i_val_loader1.load(), #CHANGE
        validation_steps=i_val_loader1.steps_per_epoch, #CHANGE
        epochs=20,
        verbose=0,
        callbacks=[early_stop]
    )

    # Return the best (minimum) validation loss achieved in this trial
    val_loss = min(history.history['val_loss'])
    val_accuracy = max(history.history['val_accuracy'])

    trial.set_user_attr('accuracy', val_accuracy)
    print(f"Trial {trial.number} -> val_loss: {val_loss:.4f}, val_accuracy: {val_accuracy:.4f}")

    return val_loss

# --- Run Bayesian Optimization Study to MINIMIZE loss ---
study_gcn = optuna.create_study(direction='minimize')
study_gcn.optimize(objective_gcn, n_trials=20)

print("Optimization finished.")
print("Best trial (lowest loss):", study_gcn.best_trial.params)
print("Best validation loss:", study_gcn.best_value)
print("Best validation accuracy:", study_gcn.best_trial.user_attrs['accuracy'])

In [ ]:
#Best parameters

#Original df
#Best trial (lowest loss): {'channel1': 16, 'channel2': 64, 'l2_reg': 0.00010022193551430476, 'dropout_rate2': 0.10973333423421638, 'learning_rate': 0.0090132361845873}
#Best validation loss: 0.1273932158946991
#Best validation accuracy: 0.5661227703094482

#Bootstrapped df

#Original - w
#Best trial (lowest loss): {'channel1': 32, 'channel2': 64, 'l2_reg': 0.0002599725548783481, 'dropout_rate2': 0.36789816422862853, 'learning_rate': 0.00442981889701694}
#Best validation loss: 0.13498926162719727
#Best validation accuracy: 0.5485186576843262

#Bootstrapped - w

#Original - i
#Best trial (lowest loss): {'channel1': 64, 'channel2': 64, 'l2_reg': 0.00011790213240045454, 'dropout_rate2': 0.5038747874634564, 'learning_rate': 0.0034436855930526596}
#Best validation loss: 0.15998294949531555
#Best validation accuracy: 0.45706310868263245

#Bootstrapped - i

In [ ]:
#graph_df_all = dataset1[0]
#w_graph_df = dataset3[0]
#i_graph_df = dataset5[0]

N = i_graph_df.n_nodes #CHANGE FOR EACH SUBSET
y_int = np.random.randint(0, 5, size=(N,))            # integer labels (0..4)

## Create SingleLoader for each set (assuming full-batch training)
#Load names

#train_loader1 = SingleLoader(dataset1, sample_weights=train_mask_tf1)
#val_loader1   = SingleLoader(dataset1, sample_weights=val_mask_tf1)
#test_loader1  = SingleLoader(dataset1, sample_weights=test_mask_tf1)
#train_loader2 = SingleLoader(dataset2, sample_weights=train_mask_tf2)
#val_loader2   = SingleLoader(dataset2, sample_weights=val_mask_tf2)
#test_loader2  = SingleLoader(dataset2, sample_weights=test_mask_tf2)
#w_train_loader1 = SingleLoader(dataset3, sample_weights=w_train_mask_tf1)
#w_val_loader1   = SingleLoader(dataset3, sample_weights=w_val_mask_tf1)
#w_test_loader1  = SingleLoader(dataset3, sample_weights=w_test_mask_tf1)
#w_train_loader2 = SingleLoader(dataset4, sample_weights=w_train_mask_tf2)
#w_val_loader2   = SingleLoader(dataset4, sample_weights=w_val_mask_tf2)
#w_test_loader2  = SingleLoader(dataset4, sample_weights=w_test_mask_tf2)
#i_train_loader_tf1 = SingleLoader(dataset5, sample_weights=i_train_mask_tf1)
#i_val_loader_tf1   = SingleLoader(dataset5, sample_weights=i_val_mask_tf1)
#i_test_loader_tf1  = SingleLoader(dataset5, sample_weights=i_test_mask_tf1)
#i_train_loader_tf2 = SingleLoader(dataset5, sample_weights=i_train_mask_tf2)
#i_val_loader_tf2   = SingleLoader(dataset5, sample_weights=i_val_mask_tf2)
#i_test_loader_tf2  = SingleLoader(dataset5, sample_weights=i_test_mask_tf2)

history = model.fit(
    i_train_loader1.load(),  #CHANGE
    steps_per_epoch= i_train_loader1.steps_per_epoch, #CHANGE
    validation_data= i_val_loader1.load(), #CHANGE
    validation_steps= i_val_loader1.steps_per_epoch, #CHANGE
    epochs=100,
    callbacks=[early_stop],
    verbose=2
)

no_epochs=len(history.history['val_loss'])

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('GCN Model Accuracy - Original dataset (Full feature set)') #CHANGE
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('GCN Model Loss - Original dataset (Full feature set)') #CHANGE
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Convert history to DataFrame
gcn_df_traininglog = pd.DataFrame(history.history) #CHANGE NAME

# Save to CSV
gcn_df_traininglog.to_csv("/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/gcn_df1_i_traininglog.csv", index=False) #CHANGE ACCORDING TO FULL FEATURE SET, W, or I

model.save(filepath='/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/Ablation_models/gcn_df1_w.keras', include_optimizer=True) #CHANGE NAME

In [ ]:
# Ensure X and A_with_loops are float32 for consistency with model layers


#X1 = full feature set
#X2 = full feature set boot
#X1_w = weather_com1
#X2_w = weather_com2
#X1_i = infra_com1
#X2_i = infra_com2

#A_with_loops1
#A_with_loops2
#w_A_with_loops1
#w_A_with_loops2
#i_A_with_loops1
#i_A_with_loops2


X1_tf = tf.cast(X1, tf.float32)
A1_tf = tf.cast(A_with_loops1, tf.float32)
X2_tf = tf.cast(X2, tf.float32)
A2_tf = tf.cast(A_with_loops2, tf.float32)
X1_w_tf = tf.cast(X1_w, tf.float32)
A1_w_tf = tf.cast(w_A_with_loops1, tf.float32)
X2_w_tf = tf.cast(X2_w, tf.float32)
A2_w_tf = tf.cast(w_A_with_loops2, tf.float32)
X1_i_tf = tf.cast(X1_i, tf.float32)
A1_i_tf = tf.cast(i_A_with_loops1, tf.float32)
X2_i_tf = tf.cast(X2_i, tf.float32)
A2_i_tf = tf.cast(i_A_with_loops2, tf.float32)

GCN testing + evaluation

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report,balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Get the actual class names from the label encoder mapping
class_names = list(label_enc1_mapping.keys())

# ----------------------------------------------------------------------
# 1) Run predictions for all nodes in the graph
# ----------------------------------------------------------------------

# Generate predictions. Using the model call directly can sometimes resolve
# unpacking issues in custom Spektral/Keras models during inference.

#Names X and A
#X1_tf = tf.cast(X1, tf.float32)
#A1_tf = tf.cast(A_with_loops1, tf.float32)
#X2_tf = tf.cast(X2, tf.float32)
#A2_tf = tf.cast(A_with_loops2, tf.float32)
#X1_w_tf = tf.cast(X1_w, tf.float32)
#A1_w_tf = tf.cast(w_A_with_loops1, tf.float32)
#X2_w_tf = tf.cast(X2_w, tf.float32)
#A2_w_tf = tf.cast(w_A_with_loops2, tf.float32)
#X1_i_tf = tf.cast(X1_i, tf.float32)
#A1_i_tf = tf.cast(i_A_with_loops1, tf.float32)
#X2_i_tf = tf.cast(X2_i, tf.float32)
#A2_i_tf = tf.cast(i_A_with_loops2, tf.float32)

y_pred_softmax = model([X1_i_tf, A1_i_tf], training=False).numpy() #CHANGE
y_pred_classes = np.argmax(y_pred_softmax, axis=1)

#Test indices names

#idx_test1  = np.arange(no_train1 + no_val1, num_nodes1)
#idx_test2  = np.arange(no_train2 + no_val2, num_nodes2)
#w_idx_test1  = np.arange(w_no_train1 + w_no_val1, w_num_nodes1)
#w_idx_test2  = np.arange(w_no_train2 + w_no_val2, w_num_nodes2)
#i_idx_test1  = np.arange(i_no_train1 + i_no_val1, i_num_nodes1)
#i_idx_test2  = np.arange(i_no_train2 + i_no_val2, i_num_nodes2)
# ----------------------------------------------------------------------
# 2) Extract the test set
# ----------------------------------------------------------------------
# y2 contains the ground truth integer labels for the bootstrapped set
y_true_test = y1[i_idx_test1]              #CHANGE
y_pred_test = y_pred_classes[i_idx_test1]  #CHANGE
y_prob_test = y_pred_softmax[i_idx_test1]  #CHANGE

# ----------------------------------------------------------------------
# 3) Confusion Matrix
# ----------------------------------------------------------------------
cm = confusion_matrix(y_true_test, y_pred_test)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix GCN - Original dataset (IP features)") #CHANGE
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------
# 4) Classification Report
# ----------------------------------------------------------------------
print("Classification Report on Test Set:")
print(classification_report(y_true_test, y_pred_test, target_names=class_names, zero_division=0))

#Balanced Accuracy
balanced_acc = balanced_accuracy_score(y_true_test, y_pred_test)
print(f"Balanced Accuracy on Test Set: {balanced_acc:.4f}")

#Weighted F1 Score
w_f1_score = f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0)
print(f"Weighted F1 Score on Test Set: {w_f1_score:.4f}")

# ----------------------------------------------------------------------


In [ ]:
#Bootstrapped evaluation metrics

from sklearn.utils import resample
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, balanced_accuracy_score
import pandas as pd
import numpy as np

# Initialize lists to store metrics
metrics_list_gcn = []

n_iterations = 10
sample_size = 4*len(i_idx_test1) #CHANGE

# Fallback for class names
try:
    c_names = class_names
except NameError:
    c_names = [f"Class {i}" for i in range(5)]

# Using indices for resampling from the test subset
for i in range(n_iterations):
    # Bootstrap the indices of the test set
    boot_idx = resample(i_idx_test1, n_samples=sample_size, random_state=42 + i, replace=True) #CHANGE

    # Predict using the trained GCN model
    # Note: For GCN, we need to pass the features and adjacency matrix.
    # Since the prediction on the whole graph is already available, we can just resample from those predictions.
    y_true_boot = y1[boot_idx] #CHANGE
    y_pred_boot = y_pred_classes[boot_idx]

    # Calculate general metrics
    precision = precision_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    recall = recall_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    f1 = f1_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_true_boot, y_pred_boot)
    balanced_acc = balanced_accuracy_score(y_true_boot, y_pred_boot)

    # Calculate per-class metrics
    precision_per_class = precision_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))
    recall_per_class = recall_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))

    # Build iteration dictionary
    iter_metrics = {
        'iteration': i + 1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }

    # Add per-class metrics to the dictionary
    for idx, c_name in enumerate(c_names):
        iter_metrics[f'precision_{c_name}'] = precision_per_class[idx]
        iter_metrics[f'recall_{c_name}'] = recall_per_class[idx]

    # Append to list
    metrics_list_gcn.append(iter_metrics)

# Create a DataFrame to view the metrics
bootstrap_metrics_gcn_df = pd.DataFrame(metrics_list_gcn)
display(bootstrap_metrics_gcn_df)

# Report the mean score of all evaluation metrics with their standard deviations
mean_metrics_gcn = bootstrap_metrics_gcn_df.drop(columns=['iteration']).mean()
std_metrics_gcn = bootstrap_metrics_gcn_df.drop(columns=['iteration']).std()

summary_gcn_df = pd.DataFrame({
    'Mean': mean_metrics_gcn,
    'Standard Deviation': std_metrics_gcn
})

print("\nMean and Standard Deviation of Evaluation Metrics (10 iterations):")
display(summary_gcn_df)


In [ ]:
y_prob_pd = pd.DataFrame(y_prob_test, columns=class_names)
#y_prob_pd.to_csv('drive/MyDrive/PhD data analysis/Paper1/testprob_bootstrap_gcn.csv', header=True, index=True)
y_prob_pd.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/pred_df1_i_gcn.csv', header=True, index=True)
y_prob_test.shape

summary_gcn_df.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/summary_df1_i_gcn_metrics.csv', header=True, index=True)

GAT

In [ ]:
#One hot encoding of targets

y_one_hot1 = tf.keras.utils.to_categorical(y1, num_classes=5)
print(y_one_hot1.shape)
y_one_hot2 = tf.keras.utils.to_categorical(y2, num_classes=5)
print(y_one_hot2.shape)

In [ ]:
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from spektral.layers import GATConv
from spektral.data import Dataset, Graph
from spektral.data.loaders import SingleLoader

#Names X and A
#X1_tf = tf.cast(X1, tf.float32)
#A1_tf = tf.cast(A_with_loops1, tf.float32)
#X2_tf = tf.cast(X2, tf.float32)
#A2_tf = tf.cast(A_with_loops2, tf.float32)
#X1_w_tf = tf.cast(X1_w, tf.float32)
#A1_w_tf = tf.cast(w_A_with_loops1, tf.float32)
#X2_w_tf = tf.cast(X2_w, tf.float32)
#A2_w_tf = tf.cast(w_A_with_loops2, tf.float32)
#X1_i_tf = tf.cast(X1_i, tf.float32)
#A1_i_tf = tf.cast(i_A_with_loops1, tf.float32)
#X2_i_tf = tf.cast(X2_i, tf.float32)
#A2_i_tf = tf.cast(i_A_with_loops2, tf.float32)

F=4 #CHANGE with subset
n_classes = 5   # output classes
l2_reg = 0.0003559581948469762


X_in = Input(shape=(F,), name='X_in')
A_in = Input(shape=(None,), sparse=True, name='A_in')

x = GATConv(
    channels=16,
    attn_heads=2,
    concat=True,
    activation='relu',
    #kernel_regularizer=l2(l2_reg),
    bias_regularizer=l2(l2_reg),
)([X_in, A_in])
#x = Dropout(0.5)(x)

x = GATConv(
    channels=8,
    attn_heads=4,
    concat=False,
    activation='relu',
    kernel_regularizer=l2(l2_reg),
   # bias_regularizer=l2(l2_reg),
)([x, A_in])
x = Dropout(0.21102037471008914)(x)


out = Dense(
    n_classes,
    activation='softmax',
    kernel_regularizer=l2(l2_reg),
    bias_regularizer=l2(l2_reg),
)(x)

model_gat = Model(inputs=[X_in, A_in], outputs=out)
model_gat.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.009815809088614048),
   loss='categorical_crossentropy',
    weighted_metrics=['accuracy'],
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

#model_gat.compile(
#    optimizer=tf.keras.optimizers.Adam(1e-2),
#    loss=lambda y_true, y_pred: masked_sparse_cce(y_true, y_pred, train_mask_tf), #Otherwise train_mask_tensor
#    metrics=["accuracy"]) #ASK CHATGPT HOW TO ADD DEFINED ACC METRIC

model_gat.summary()

In [ ]:
import optuna
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from spektral.layers import GATConv

def objective_gat(trial):
    # --- Define search space ---
    channels1 = trial.suggest_categorical('channels1', [8, 16, 32, 64])
    channels2 = trial.suggest_categorical('channels2', [8, 16, 32, 64])
    attn_heads1 = trial.suggest_categorical('attn_heads1', [1, 2, 4])
    attn_heads2 = trial.suggest_categorical('attn_heads2', [1, 2, 4])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.7)
    l2_reg = trial.suggest_float('l2_reg', 1e-4, 1e-1, log=True)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    F = 4 #CHANGE with subset
    n_classes = 5

    X_in = Input(shape=(F,), name='X_in')
    A_in = Input(shape=(None,), sparse=True, name='A_in')

    x = GATConv(
        channels=channels1,
        attn_heads=attn_heads1,
        concat=True,
        activation='relu',
        bias_regularizer=l2(l2_reg),
    )([X_in, A_in])

    x = GATConv(
        channels=channels2,
        attn_heads=attn_heads2,
        concat=False,
        activation='relu',
        kernel_regularizer=l2(l2_reg),
    )([x, A_in])
    x = Dropout(dropout_rate)(x)

    out = Dense(
        n_classes,
        activation='softmax',
        kernel_regularizer=l2(l2_reg),
        bias_regularizer=l2(l2_reg),
    )(x)

    model_gat = Model(inputs=[X_in, A_in], outputs=out)
    model_gat.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Fit on training data with early stopping based on val_loss
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model_gat.fit(
        i_train_loader1.load(), #CHANGE
        steps_per_epoch=i_train_loader1.steps_per_epoch, #CHANGE
        validation_data=i_val_loader1.load(), #CHANGE
        validation_steps=i_val_loader1.steps_per_epoch, #CHANGE
        epochs=20,
        verbose=0,
        callbacks=[early_stop]
    )

    # Return the best (minimum) validation loss achieved in this trial
    val_loss = min(history.history['val_loss'])
    val_accuracy = max(history.history['accuracy'] if 'val_accuracy' not in history.history else history.history['val_accuracy'])

    trial.set_user_attr('accuracy', val_accuracy)
    print(f"Trial {trial.number} -> val_loss: {val_loss:.4f}, val_accuracy: {val_accuracy:.4f}")

    return val_loss

# --- Run Bayesian Optimization Study to MINIMIZE loss ---
study_gat = optuna.create_study(direction='minimize')
study_gat.optimize(objective_gat, n_trials=20) # Adjust n_trials as needed

print("Optimization finished.")
print("Best trial (lowest loss):", study_gat.best_trial.params)
print("Best validation loss:", study_gat.best_value)
print("Best validation accuracy:", study_gat.best_trial.user_attrs['accuracy'])

In [ ]:
#Best parameters

#full features
#Best trial (lowest loss): {'channels1': 32, 'channels2': 32, 'attn_heads1': 2, 'attn_heads2': 4, 'dropout_rate': 0.40006469208861795, 'l2_reg': 0.0008321702761460888, 'learning_rate': 0.005414179784675206}
#Best validation loss: 0.15236592292785645
#Best validation accuracy: 0.5942464470863342

#Weather features
#Best trial (lowest loss): {'channels1': 64, 'channels2': 16, 'attn_heads1': 2, 'attn_heads2': 4, 'dropout_rate': 0.2271496646750485, 'l2_reg': 0.00010017165371384705, 'learning_rate': 0.005527174877681036}
#Best validation loss: 0.13527996838092804
#Best validation accuracy: 0.563761293888092

#IP features
#Best trial (lowest loss): {'channels1': 16, 'channels2': 8, 'attn_heads1': 2, 'attn_heads2': 4, 'dropout_rate': 0.21102037471008914, 'l2_reg': 0.0003559581948469762, 'learning_rate': 0.009815809088614048}
#Best validation loss: 0.17363114655017853
#Best validation accuracy: 0.45706310868263245

In [ ]:
#Train the GAT

model_gat.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    weighted_metrics=['accuracy'],
)

history = model_gat.fit(
    i_train_loader1.load(),      #CHANGE            # generator
    steps_per_epoch=i_train_loader1.steps_per_epoch, #CHANGE
    validation_data=i_val_loader1.load(),  #CHANGE# generator
    validation_steps=i_val_loader1.steps_per_epoch, #CHANGE
    epochs=100,
    verbose=2,
    callbacks=[early_stop]
)

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('GAT Model Accuracy - Original dataset (Full feature set)') #CHANGE
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('GAT Model Loss - Original dataset (Full feature set)') #CHANGE
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Convert history to DataFrame
gat_df_traininglog = pd.DataFrame(history.history) #CHANGE NAME

# Save to CSV
gat_df_traininglog.to_csv("/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/gat_df1_i_traininglog.csv", index=False) #CHANGE ACCORDING TO FULL FEATURE SET, W, or I

model.save(filepath='/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/Ablation_models/gat_i_df1.keras', include_optimizer=True) #CHANGE NAME

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Test the model
test_loss, test_acc = model_gat.evaluate(i_test_loader1.load(), steps=i_test_loader1.steps_per_epoch) #CHANGE
print(f'Test accuracy: {test_acc}')

# Predict on the entire graph
y_pred_probs = model_gat.predict(i_test_loader1.load(), steps=i_test_loader1.steps_per_epoch) #CHANGE
if len(y_pred_probs.shape) == 3:
    y_pred_probs = y_pred_probs[0]

y_pred_classes = np.argmax(y_pred_probs, axis=1)

# Extract the test set predictions and true labels using test indices
y_true_test = y1[i_idx_test1] #CHANGE
y_pred_test = y_pred_classes[i_idx_test1] #CHANGE

class_names = list(label_enc1_mapping.keys())

# -----------------------------
# Classification report
# -----------------------------
print("\nClassification Report:")
print(classification_report(y_true_test, y_pred_test, target_names=class_names, zero_division=0))

# Balanced Accuracy
balanced_acc = balanced_accuracy_score(y_true_test, y_pred_test)
print(f"Balanced Accuracy on Test Set: {balanced_acc:.4f}")

# Weighted F1 Score
w_f1_score = f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0)
print(f"Weighted F1 Score on Test Set: {w_f1_score:.4f}")

# -----------------------------
# Confusion matrix
# -----------------------------
cm = confusion_matrix(y_true_test, y_pred_test)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix GAT - Original dataset (IP features)")
#plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.utils import resample
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, balanced_accuracy_score
import pandas as pd
import numpy as np

n_iterations = 10
sample_size = 4*len(i_idx_test1) #CHANGE

# Ensure class_names is available
try:
    c_names = class_names
except NameError:
    c_names = [f"Class {i}" for i in range(5)]


print("Evaluating GAT...")
metrics_list_gat = []
for i in range(n_iterations):
    # Bootstrap the indices of the test set
    boot_idx = resample(i_idx_test1, n_samples=sample_size, random_state=42 + i, replace=True) #CHANGE
    y_true_boot = y1[boot_idx] #CHANGE
    # y_pred_classes is assumed to be available from the prior GAT prediction cell
    y_pred_boot = y_pred_classes[boot_idx]

    precision = precision_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    recall = recall_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    f1 = f1_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_true_boot, y_pred_boot)
    balanced_acc = balanced_accuracy_score(y_true_boot, y_pred_boot)

    precision_per_class = precision_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))
    recall_per_class = recall_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))

    iter_metrics = {
        'iteration': i + 1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }
    for idx, c_name in enumerate(c_names):
        iter_metrics[f'precision_{c_name}'] = precision_per_class[idx]
        iter_metrics[f'recall_{c_name}'] = recall_per_class[idx]

    metrics_list_gat.append(iter_metrics)

bootstrap_metrics_gat_df = pd.DataFrame(metrics_list_gat)
display(bootstrap_metrics_gat_df)

mean_metrics_gat = bootstrap_metrics_gat_df.drop(columns=['iteration']).mean()
std_metrics_gat = bootstrap_metrics_gat_df.drop(columns=['iteration']).std()
summary_gat_df = pd.DataFrame({'Mean': mean_metrics_gat, 'Standard Deviation': std_metrics_gat})
print("\nMean and Standard Deviation of Evaluation Metrics GAT (10 iterations):")
display(summary_gat_df)




In [ ]:

y_pred_probs = pd.DataFrame(y_prob_test, columns=class_names)
#y_prob_pd.to_csv('drive/MyDrive/PhD data analysis/Paper1/testprob_bootstrap_gcn.csv', header=True, index=True)
y_pred_probs.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/pred_df1_i_gat.csv', header=True, index=True)
y_pred_probs.shape
summary_gat_df.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/gat_df1_i_summary_metrics.csv', index=True)

# BLSTM/LSTM

Time-windowing using time-series generator (original + bootstrapped)

In [ ]:
#Data splitting original data

train1 = df1.loc['2014-01-01':'2020-07-01']
val1 = df1.loc['2020-07-02':'2021-07-01']
test1 = df1.loc['2021-07-02':]

print("Train set shape:", train1.shape)
print("Validation set shape:", val1.shape)
print("Test set shape:", test1.shape)

In [ ]:
#Splitting attributes

weather = ['max_temp','sum_prec (1h)']#.to_numpy().tolist()
infra = ['low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']
features = ['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']
spatial = ['x_coor', 'y_coor', 'z_coor']#.to_numpy().tolist()
coords = ['lat', 'lon']
output = ['incident_type']#tolist()

attributes1 = df1[['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']]

X_train_w1 = train1[weather]
X_train_i1 = train1[infra]
X_train1 = train1[features]
y_train1 = train1[output]
y_train1 = y_train1.values.flatten()

X_val_w1 = val1[weather]
X_val_i1 = val1[infra]
X_val1 = val1[features]
y_val1 = val1[output]
y_val1 = y_val1.values.flatten()

X_test_w1 = test1[weather]
X_test_i1 = test1[infra]
X_test1 = test1[features]
y_test1 = test1[output]
y_test1 = y_test1.values.flatten()

print("X train weather, infra, total, and y train shapes:", X_train_w1.shape, X_train_i1.shape, X_train1.shape, y_train1.shape)
print("X val weather, infra, total, and y val shapes:", X_val_w1.shape, X_val_i1.shape, X_val1.shape, y_val1.shape)
print("X test weather, infra, total, and y test shapes:", X_test_w1.shape, X_test_i1.shape, X_test1.shape, y_test1.shape)

In [ ]:
#Data splitting bootstrapping

train2 = bootstrapped_df.loc['2014-01-01':'2020-07-01']
val2 = bootstrapped_df.loc['2020-07-02':'2021-07-01']
test2 = bootstrapped_df.loc['2021-07-02':]

print("Train set shape:", train2.shape)
print("Validation set shape:", val2.shape)
print("Test set shape:", test2.shape)

In [ ]:
#Splitting attributes across train, val, test

weather = ['max_temp','sum_prec (1h)']#.to_numpy().tolist()
infra = ['low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']
features = ['max_temp','sum_prec (1h)', 'low_veg_%',
         'mid_high_veg_%',
         'pavement_%',
          'water_%']
spatial = ['x_coor', 'y_coor', 'z_coor']#.to_numpy().tolist()
coords = ['lat', 'lon']
output = ['incident_type']#tolist()

X_train_w2 = train2[weather]
X_train_i2 = train2[infra]
X_train2 = train2[features]
y_train2 = train2[output]
y_train2 = y_train2.values.flatten()

X_val_w2 = val2[weather]
X_val_i2 = val2[infra]
X_val2 = val2[features]
y_val2 = val2[output]
y_val2 = y_val2.values.flatten()

X_test_w2 = test2[weather]
X_test_i2 = test2[infra]
X_test2 = test2[features]
y_test2 = test2[output]
y_test2 = y_test2.values.flatten()

print("X train weather, infra, total, and y train shapes:", X_train_w2.shape, X_train_i2.shape, X_train2.shape, y_train2.shape)
print("X val weather, infra, total, and y val shapes:", X_val_w2.shape, X_val_i2.shape, X_val2.shape, y_val2.shape)
print("X test weather, infra, total, and y test shapes:", X_test_w2.shape, X_test_i2.shape, X_test2.shape, y_test2.shape)

In [ ]:
#Time-series generator

import keras
import tensorflow as tf

#generator11
#generator12
#generator13 Test
#generator21
#generator22
#generator23 Test bootstrap
#generator_w_11
#generator_w_12
#generator_w_13 Test original w
#generator_i_11
#generator_i_12
#generator_i_13 Test original i
#generator_w_21
#generator_w_22
#generator_w_23 Test boot w
#generator_i_21
#generator_i_22
#generator_i_23 Test boot i


generator11 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train1,
                                                                y_train1,
                                                                length=4, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator12 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val1,
                                                                y_val1,
                                                                length=4,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator13 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test1,
                                                                y_test1,
                                                                length=4,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator_w_11 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train_w1,
                                                                y_train1,
                                                                length=4, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator_w_12 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val_w1,
                                                                y_val1,
                                                                length=4,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator_w_13 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test_w1,
                                                                y_test1,
                                                                length=4,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator_i_11 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train_i1,
                                                                y_train1,
                                                                length=4, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator_i_12 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val_i1,
                                                                y_val1,
                                                                length=4,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator_i_13 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test_i1,
                                                                y_test1,
                                                                length=4,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator21 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train2,
                                                                y_train2,
                                                                length=30, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator22 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val2,
                                                                y_val2,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator23 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test2,
                                                                y_test2,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator_w_21 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train_w2,
                                                                y_train2,
                                                                length=30, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator_w_22 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val_w2,
                                                                y_val2,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator_w_23 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test_w2,
                                                                y_test2,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator_i_21 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train_i2,
                                                                y_train2,
                                                                length=30, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator_i_22 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val_i2,
                                                                y_val2,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator_i_23 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test_i2,
                                                                y_test2,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)
X_test = np.array(X_test)
y_test = np.array(y_test)


print("No. of train samples by generator - original data:", len(generator11))#, "Training data shape:", X_train.shape, y_train.shape)
print("No. of val samples by generator - bootstrapped:", len(generator12))#, "Validation data shape:", X_val.shape, y_val.shape)
print("No of. test samples by generator - red cluster :", len(generator13))#, "Test data shape:", X_test.shape, y_test.shape)


print(f"Training batches: {len(generator11)}, Validation batches: {len(generator12)}")
print(f"One batch x shape: {generator11[0][0].shape}, y shape: {generator11[0][1].shape}")

LSTM training

In [ ]:
#Model building POTENTIALLY ADD AUTO-ENCODER architecture for feature extraction
#Add incident duration as an output

#-CLUSTER 3 in PAPER - Cluster 1 = Red region (778) - train1,val1,test1
#-CLUSTER 1 IN PAPER - Cluster 2 = Blue region (1184) - train2,val2,test2
#-CLUSTER 2 in PAPER - Cluster 3 = Green region (1256) - train3,val3,test3
#-CLUSTER 4 in PAPER - Cluster 4 = Grey region (1193) - train4,val4,test4
#-CLUSTER 4 in PAPER - Cluster 5 = Purple region (351) - train5,val5,test5

from tensorflow import keras
from keras.layers import Dense, LSTM, Bidirectional

def build_lstm(lstm_units1, lstm_units2, dropout_rate):
    model = keras.models.Sequential()

    # First Layer
    model.add(keras.layers.LSTM(lstm_units1, return_sequences=True, input_shape=(4, 6))) # 5 timestamps, 6 features (CHANGE FOR EACH SUBSET)

    # Second Layer
    model.add(keras.layers.LSTM(lstm_units2, return_sequences=False))
    model.add(keras.layers.LeakyReLU(negative_slope=0.1))

    # Dropout Layer
    model.add(keras.layers.Dropout(dropout_rate))

    # Final Output Layer
    model.add(keras.layers.Dense(5, activation='softmax'))

    return model

# Instantiate the model by calling the function with your desired hyperparameters
model = build_lstm(lstm_units1=16, lstm_units2=16, dropout_rate=0.4238106445149247)
model.summary()


##model = keras.models.Sequential()

# First Layer
##model.add(keras.layers.LSTM(32, return_sequences=True, input_shape=(30,6))) #30 is timestamps, 6 is no. of features (CHANGE FOR EACH SUBSET)

# Second Layer
##model.add(keras.layers.LSTM(32, return_sequences=False))#, activation="relu"))
##model.add(keras.layers.LeakyReLU(alpha=0.1))

# 3rd Layer (Dense)
#model.add(keras.layers.Dense(32))#, activation="relu"))
#model.add(keras.layers.LeakyReLU(alpha=0.1))

# 4th Layer (Dropout)
##model.add(keras.layers.Dropout(0.5))

# Final Output Layer
##odel.add(keras.layers.Dense(5, activation='softmax'))

##model.summary()

callback = keras.callbacks.EarlyStopping(monitor='val_loss',
                                              patience=3)
model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                optimizer=tf.keras.optimizers.Adamax(learning_rate=0.006,
                                                     beta_1=0.9,
                                                     beta_2=0.999),
                metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

In [ ]:
#Bayesian optimization of hyperparameters

import optuna
import tensorflow as tf
from tensorflow import keras


def objective_lstm(trial):
    # --- Define search space ---
    lstm_units1 = trial.suggest_categorical('lstm_units1', [8, 16, 32, 64])
    lstm_units2 = trial.suggest_categorical('lstm_units2', [8, 16, 32, 64])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.5)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    # Build model
    model = build_lstm(lstm_units1, lstm_units2, dropout_rate)

    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        optimizer=tf.keras.optimizers.Adamax(learning_rate=lr, beta_1=0.9, beta_2=0.999),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
    )

    # Early stopping to prevent overfitting and speed up search
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Fit the model using generators
    history = model.fit(
        generator_i_11, #CHANGE
        validation_data=generator_i_12, #CHANGE
        epochs=20,
        verbose=0,
        callbacks=[early_stop]
    )

    # Return the best validation loss achieved in this trial
    val_loss = min(history.history['val_loss'])
    # With SparseCategoricalAccuracy, metric name is val_sparse_categorical_accuracy
    val_accuracy = max(history.history['val_sparse_categorical_accuracy'])

    trial.set_user_attr('accuracy', val_accuracy)
    print(f"Trial {trial.number} -> val_loss: {val_loss:.4f}, val_accuracy: {val_accuracy:.4f}")

    return val_loss

# --- Run Bayesian Optimization Study ---
study_lstm = optuna.create_study(direction='minimize')
study_lstm.optimize(objective_lstm, n_trials=20) # Adjust n_trials as needed based on compute limits

print("Optimization finished.")
print("Best trial (lowest loss):", study_lstm.best_trial.params)
print("Best validation loss:", study_lstm.best_value)
print("Best validation accuracy:", study_lstm.best_trial.user_attrs['accuracy'])

In [ ]:
#Best parameters

#Original df
#Best trial (lowest loss): {'lstm_units1': 8, 'lstm_units2': 64, 'dropout_rate': 0.41445037853859434, 'learning_rate': 0.009713680473875897}
#Best validation loss: 1.1635074615478516
#Best validation accuracy: 0.5231910943984985

#Bootstrapped df

#Original - w
#Best trial (lowest loss): {'lstm_units1': 16, 'lstm_units2': 16, 'dropout_rate': 0.2946151870263789, 'learning_rate': 0.0042133970304861455}
#Best validation loss: 1.1637895107269287
#Best validation accuracy: 0.5250464081764221

#Bootstrapped - w

#Original - i
#Best trial (lowest loss): {'lstm_units1': 16, 'lstm_units2': 16, 'dropout_rate': 0.4238106445149247, 'learning_rate': 0.00642761991964907}
#Best validation loss: 1.3523476123809814
#Best validation accuracy: 0.4953617751598358

#Bootstrapped - i

In [ ]:
#Training

#generator11
#generator12
#generator13 Test
#generator21
#generator22
#generator23 Test bootstrap
#generator_w_11
#generator_w_12
#generator_w_13 Test original w
#generator_i_11
#generator_i_12
#generator_i_13 Test original i
#generator_w_21
#generator_w_22
#generator_w_23 Test boot w
#generator_i_21
#generator_i_22
#generator_i_23 Test boot i

training = model.fit(generator_i_11, epochs=100, batch_size=64, validation_data=generator_i_12, #CHANGE
                     callbacks=[callback],verbose=1)

#training = model.fit(x61, y_train6, epochs=30,
#                     batch_size=32, validation_data=(x52, y_val5),
#                     callbacks=[callback])
no_epochs=len(training.history['val_loss'])

In [ ]:
#Plotting
no_epochs=len(training.history['loss'])
acc = model.history.history['sparse_categorical_accuracy']
val_acc = model.history.history['val_sparse_categorical_accuracy']
loss = model.history.history['loss']
val_loss = model.history.history['val_loss']
epochs_range = range(no_epochs)

In [ ]:
plt.figure(figsize=(14, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.xlabel('Number of epochs')
plt.ylabel('Accuracy')
plt.title('Sparse Categorical Accuracy (Training and validation)')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.xlabel('Number of epochs')
plt.ylabel('Loss')
plt.title('Sparse Categorical Cross Entropy (Training and validation)')
plt.show()

In [ ]:
#Save log of training and validation accuracy scores

# Convert history to DataFrame
lstm_df_traininglog = pd.DataFrame(training.history) #CHANGE NAME

# Save to CSV
lstm_df_traininglog.to_csv("/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/lstm_df1_i_traininglog.csv", index=False) #CHANGE ACCORDING TO FULL FEATURE SET, W, or I

model.save(filepath='/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/Ablation_models/lstm_df1_i.keras', include_optimizer=True) #CHANGE NAME

LSTM testing + evaluation metrics

In [ ]:
#Testing

predictions = model.predict(generator_i_13) #CHANGE per subset
y_pred = np.argmax(predictions, axis=1)


In [ ]:
#Generator true labels
test_gen = generator_i_13 #CHANGE per subset

y_true = []
for i in range(len(test_gen)):
    y_batch = test_gen[i][1]
    # If one-hot encoded:
    if y_batch.ndim > 1:
        y_true.extend(np.argmax(y_batch, axis=1))
    else:
        y_true.extend(y_batch)
y_true = np.array(y_true)

print(len(y_true))
print(len(y_pred))

In [ ]:
#Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, multilabel_confusion_matrix, balanced_accuracy_score, f1_score

# Compute confusion matrix and report
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix LSTM - Original dataset (Infrastructure Prox. features)")
#plt.xticks(rotation=45, ha='right')
#plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
#Classification report
# Get unique labels from y_test
unique_labels = np.unique(y_true)

# Filter target_names to match the unique labels
filtered_target_names = [label_enc21.classes_[i] for i in unique_labels]

# Now, generate the classification report with the filtered target names
report = classification_report(y_true, y_pred, target_names=filtered_target_names, labels=unique_labels)
print(report)

#Balanced Accuracy
balanced_acc = balanced_accuracy_score(y_true, y_pred)
print(f"Balanced Accuracy on Test Set: {balanced_acc:.4f}")

#Weighted F1 Score
w_f1_score = f1_score(y_true, y_pred, average='weighted')
print(f"Weighted F1 Score on Test Set: {w_f1_score:.4f}")

In [ ]:
loss, accuracy = model.evaluate(generator_i_13) #Change for each cluster
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.utils import resample
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, balanced_accuracy_score
import pandas as pd
import numpy as np

n_iterations = 10
sample_size = 4*len(y_true)

# Ensure class_names is available
try:
    c_names = class_names
except NameError:
    c_names = [f"Class {i}" for i in range(5)]

print("Evaluating LSTM...")
metrics_list_lstm = []
for i in range(n_iterations):
    # Bootstrap the indices of the test set predictions
    boot_idx = resample(np.arange(len(y_true)), n_samples=sample_size, random_state=42 + i, replace=True)
    y_true_boot = y_true[boot_idx]
    y_pred_boot = y_pred[boot_idx]

    precision = precision_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    recall = recall_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    f1 = f1_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_true_boot, y_pred_boot)
    balanced_acc = balanced_accuracy_score(y_true_boot, y_pred_boot)

    precision_per_class = precision_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))
    recall_per_class = recall_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))

    iter_metrics = {
        'iteration': i + 1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }
    for idx, c_name in enumerate(c_names):
        iter_metrics[f'precision_{c_name}'] = precision_per_class[idx]
        iter_metrics[f'recall_{c_name}'] = recall_per_class[idx]

    metrics_list_lstm.append(iter_metrics)

bootstrap_metrics_lstm_df = pd.DataFrame(metrics_list_lstm)
display(bootstrap_metrics_lstm_df)

mean_metrics_lstm = bootstrap_metrics_lstm_df.drop(columns=['iteration']).mean()
std_metrics_lstm = bootstrap_metrics_lstm_df.drop(columns=['iteration']).std()
summary_lstm_df = pd.DataFrame({'Mean': mean_metrics_lstm, 'Standard Deviation': std_metrics_lstm})

print("\nMean and Standard Deviation of Evaluation Metrics LSTM (10 iterations):")
display(summary_lstm_df)


In [ ]:
predictions[:5]
df_pred = pd.DataFrame(predictions)
df_pred.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/pred_df1_i_lstm.csv', header=True, index=True)

summary_lstm_df.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/lstm_df1_i_summary_metrics.csv', index=True)

BLSTM training

In [ ]:
#BiLSTM model

#Best trial (lowest loss): {'lstm_units1': 16, 'lstm_units2': 16, 'dropout_rate': 0.2788706367091317, 'learning_rate': 0.009603466579872287}

from tensorflow import keras
from keras.layers import Dense, LSTM, Bidirectional

def build_blstm(blstm_units1, blstm_units2, dropout_rate):
    model1 = keras.models.Sequential()

    # First Layer
    model1.add(keras.layers.Bidirectional(LSTM(blstm_units1, return_sequences=True), input_shape=(4,6))) # 5 timestamps, 6 features (CHANGE FOR EACH SUBSET)

    # Second Layer
    model1.add(keras.layers.Bidirectional(LSTM(blstm_units2, return_sequences=False)))#, activation="relu"))
    model1.add(keras.layers.LeakyReLU(alpha=0.1))

    # Dropout Layer
    model1.add(keras.layers.Dropout(dropout_rate))

    # Final Output Layer
    model1.add(keras.layers.Dense(5, activation='softmax'))

    return model1

# Instantiate the model by calling the function with your desired hyperparameters
model1 = build_blstm(blstm_units1=16, blstm_units2=16, dropout_rate=0.2788706367091317)

# Compile the model
model1.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adamax(learning_rate=0.01, beta_1=0.9, beta_2=0.999),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

# Print the model summary
model1.summary()
##model1 = keras.models.Sequential()

# First Layer
#model1.add(keras.layers.Bidirectional(LSTM(16, return_sequences=True), input_shape=(X_train.shape[1],1))) #For generator, shape is (sequence length of generator, total amount of variables)
##model1.add(keras.layers.Bidirectional(LSTM(32, return_sequences=True), input_shape=(30,6)))

# Second Layer
##model1.add(keras.layers.Bidirectional(LSTM(32, return_sequences=False)))#, activation="relu"))
##model1.add(keras.layers.LeakyReLU(alpha=0.1))

# 3rd Layer (Dense)
#model.add(keras.layers.Dense(32, activation="relu"))

# 4th Layer (Dropout)
##model1.add(keras.layers.Dropout(0.5))

# Final Output Layer
##model1.add(keras.layers.Dense(5, activation='softmax'))

##model1.summary()
callback = keras.callbacks.EarlyStopping(monitor='val_loss',
                                              patience=5)
##model1.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(),
##                optimizer=tf.keras.optimizers.Adamax(learning_rate=0.001,
##                                                     beta_1=0.9,
##                                                     beta_2=0.999),
##                metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])


In [ ]:
import optuna
import tensorflow as tf
from tensorflow import keras


def objective_lstm(trial):
    # --- Define search space ---
    blstm_units1 = trial.suggest_categorical('lstm_units1', [8, 16, 32, 64])
    blstm_units2 = trial.suggest_categorical('lstm_units2', [8, 16, 32, 64])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.5)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    # Build model
    model1 = build_blstm(blstm_units1, blstm_units2, dropout_rate)

    model1.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        optimizer=tf.keras.optimizers.Adamax(learning_rate=lr, beta_1=0.9, beta_2=0.999),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
    )

    # Early stopping to prevent overfitting and speed up search
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Fit the model using generators
    history = model1.fit(
        generator_w_11, #CHANGE
        validation_data=generator_w_12, #CHANGE
        epochs=20,
        verbose=0,
        callbacks=[early_stop]
    )

    # Return the best validation loss achieved in this trial
    val_loss = min(history.history['val_loss'])
    val_accuracy = max(history.history['val_sparse_categorical_accuracy'])

    trial.set_user_attr('accuracy', val_accuracy)
    print(f"Trial {trial.number} -> val_loss: {val_loss:.4f}, val_accuracy: {val_accuracy:.4f}")

    return val_loss

# --- Run Bayesian Optimization Study ---
study_lstm = optuna.create_study(direction='minimize')
study_lstm.optimize(objective_lstm, n_trials=20) # Adjust n_trials as needed based on compute limits

print("Optimization finished.")
print("Best trial (lowest loss):", study_lstm.best_trial.params)
print("Best validation loss:", study_lstm.best_value)
print("Best validation accuracy:", study_lstm.best_trial.user_attrs['accuracy'])

In [ ]:
#Best parameters

#Original df
#Best trial (lowest loss): {'lstm_units1': 16, 'lstm_units2': 16, 'dropout_rate': 0.2788706367091317, 'learning_rate': 0.009603466579872287}
#Best validation loss: 1.167597770690918
#Best validation accuracy: 0.5287569761276245

#Bootstrapped df

#Original - w
#Best trial (lowest loss): {'lstm_units1': 32, 'lstm_units2': 8, 'dropout_rate': 0.3275655718685642, 'learning_rate': 0.009945372738606562}
#Best validation loss: 1.1710132360458374
#Best validation accuracy: 0.5287569761276245

#Bootstrapped - w

#Original - i

#Bootstrapped - i
#Best trial (lowest loss): {'lstm_units1': 64, 'lstm_units2': 64, 'dropout_rate': 0.26915334859127205, 'learning_rate': 0.0017313266473460629}
#Best validation loss: 1.342475175857544
#Best validation accuracy: 0.4953617751598358

In [ ]:
# Training

#generator11
#generator12
#generator13 Test
#generator21
#generator22
#generator23 Test bootstrap
#generator_w_11
#generator_w_12
#generator_w_13 Test original w
#generator_i_11
#generator_i_12
#generator_i_13 Test original i
#generator_w_21
#generator_w_22
#generator_w_23 Test boot w
#generator_i_21
#generator_i_22
#generator_i_23 Test boot i

training = model1.fit(generator11, epochs=100, batch_size=64, validation_data=generator12,
                     callbacks=[callback],verbose=1)

#training = model.fit(x61, y_train6, epochs=30,
#                     batch_size=32, validation_data=(x52, y_val5),
#                     callbacks=[callback])
no_epochs=len(training.history['val_loss'])

In [ ]:
#Plotting

acc = model1.history.history['sparse_categorical_accuracy']
val_acc = model1.history.history['val_sparse_categorical_accuracy']
loss = model1.history.history['loss']
val_loss = model1.history.history['val_loss']
epochs_range = range(no_epochs)

In [ ]:
plt.figure(figsize=(14, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.xlabel('Number of epochs')
plt.ylabel('Accuracy')
plt.title('BLSTM Model Accuracy - Original dataset (Full features)') #CHANGE

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.xlabel('Number of epochs')
plt.ylabel('Loss')
plt.title('BLSTM Model Loss - Original dataset (Full features)')
plt.show()

In [ ]:
#Save log of training and validation accuracy scores

# Convert history to DataFrame
blstm_df_traininglog = pd.DataFrame(training.history) #CHANGE NAME

# Save to CSV
blstm_df_traininglog.to_csv("/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/blstm_df1_traininglog.csv", index=False) #CHANGE ACCORDING TO FULL FEATURE SET, W, or I

model1.save(filepath='/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/Ablation_models/blstm_df1.keras', include_optimizer=True) #CHANGE NAME

BLSTM testing + evaluation metrics

In [ ]:
#Testing

predictions = model1.predict(generator13) #CHANGE WITH EACH CLUSTER to generator13 - 53
y_pred = np.argmax(predictions, axis=1)


In [ ]:
#Generator true labels
test_gen = generator13 #CHANGE WITH EACH CLUSTER to generator13 - 53

y_true = []
for i in range(len(test_gen)):
    y_batch = test_gen[i][1]
    # If one-hot encoded:
    if y_batch.ndim > 1:
        y_true.extend(np.argmax(y_batch, axis=1))
    else:
        y_true.extend(y_batch)
y_true = np.array(y_true)

print(len(y_true))
print(len(y_pred))

In [ ]:
#Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, multilabel_confusion_matrix, balanced_accuracy_score, f1_score

# Compute confusion matrix and report
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix BLSTM - Original dataset (Full features)")
#plt.xticks(rotation=45, ha='right')
#plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
#Classification report
# Get unique labels from y_test
unique_labels = np.unique(y_true)

# Filter target_names to match the unique labels
filtered_target_names = [label_enc21.classes_[i] for i in unique_labels]

# Now, generate the classification report with the filtered target names
report = classification_report(y_true, y_pred, target_names=filtered_target_names, labels=unique_labels)
print(report)

#Balanced Accuracy
balanced_acc = balanced_accuracy_score(y_true, y_pred)
print(f"Balanced Accuracy on Test Set: {balanced_acc:.4f}")

#Weighted F1 Score
w_f1_score = f1_score(y_true, y_pred, average='weighted')
print(f"Weighted F1 Score on Test Set: {w_f1_score:.4f}")

In [ ]:
loss, accuracy = model1.evaluate(generator13) #CHANGE W CLUSTER
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.utils import resample
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, balanced_accuracy_score
import pandas as pd
import numpy as np

n_iterations = 10
sample_size = 4*len(y_true)

# Ensure class_names is available
try:
    c_names = class_names
except NameError:
    c_names = [f"Class {i}" for i in range(5)]

print("Evaluating BLSTM...")
metrics_list_blstm = []

# Safely extract 1D predictions for the BLSTM
y_pred_blstm = np.argmax(predictions, axis=1) if 'predictions' in locals() else np.array(y_pred)
if isinstance(y_pred_blstm, pd.DataFrame) or y_pred_blstm.ndim > 1:
    y_pred_blstm = np.argmax(np.array(y_pred_blstm), axis=1)

for i in range(n_iterations):
    # Bootstrap the indices of the test set predictions
    boot_idx = resample(np.arange(len(y_true)), n_samples=sample_size, random_state=42 + i, replace=True)
    y_true_boot = np.array(y_true)[boot_idx]
    y_pred_boot = y_pred_blstm[boot_idx]

    precision = precision_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    recall = recall_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    f1 = f1_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_true_boot, y_pred_boot)
    balanced_acc = balanced_accuracy_score(y_true_boot, y_pred_boot)

    precision_per_class = precision_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))
    recall_per_class = recall_score(y_true_boot, y_pred_boot, average=None, zero_division=0, labels=range(len(c_names)))

    iter_metrics = {
        'iteration': i + 1,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }
    for idx, c_name in enumerate(c_names):
        iter_metrics[f'precision_{c_name}'] = precision_per_class[idx]
        iter_metrics[f'recall_{c_name}'] = recall_per_class[idx]

    metrics_list_blstm.append(iter_metrics)

bootstrap_metrics_blstm_df = pd.DataFrame(metrics_list_blstm)
display(bootstrap_metrics_blstm_df)

mean_metrics_blstm = bootstrap_metrics_blstm_df.drop(columns=['iteration']).mean()
std_metrics_blstm = bootstrap_metrics_blstm_df.drop(columns=['iteration']).std()
summary_blstm_df = pd.DataFrame({'Mean': mean_metrics_blstm, 'Standard Deviation': std_metrics_blstm})

print("\nMean and Standard Deviation of Evaluation Metrics BLSTM (10 iterations):")
display(summary_blstm_df)



In [ ]:
predictions[:5]
df_pred = pd.DataFrame(predictions)
df_pred.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/pred_df1_blstm.csv', header=True, index=True)

# Save the summary DataFrame to CSV
summary_blstm_df.to_csv('/content/drive/MyDrive/PhD data analysis/Paper1/Revision_major1_DS4T/Ablation/blstm_df1_summary_metrics.csv', index=True)

# ***OLD CODE***

# GCN + GCN-BLSTM

Creating and splitting the dataset (train, test, val) https://chatgpt.com/c/68a47592-8164-8331-bee2-e84ce20a5a2b

Model building GCN

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.layers import LeakyReLU
from spektral.layers import GCNConv
import tensorflow as tf


# Define the number of output classes
# C = dataset.n_labels # This is incorrect, dataset.n_labels seems to be the number of nodes
C = 5 # Corrected to the number of incident types

import tensorflow as tf

class GCN(tf.keras.Model):
    def __init__(self, channels, n_classes, l2_reg=0.001, dropout_rate1=0.5, dropout_rate2=0.5, dropout_rate3=0.5):
        super().__init__()
        self.gcn1 = GCNConv(channels,
                            activation=tf.keras.layers.LeakyReLU(negative_slope=0.2))#,
        #                    kernel_regularizer=tf.keras.regularizers.l2(l2_reg))
       # self.dropout1 = tf.keras.layers.Dropout(dropout_rate1)

        self.gcn2 = GCNConv(channels,
                            activation=tf.keras.layers.LeakyReLU(negative_slope=0.2))#,
          #                  kernel_regularizer=tf.keras.regularizers.l2(l2_reg))
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate2)

        self.dense_out = tf.keras.layers.Dense(n_classes, activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg))

      #  self.gcn3 = GCNConv(channels,
      #                      activation=tf.keras.layers.LeakyReLU(negative_slope=0.2))#,
       #                     kernel_regularizer=tf.keras.regularizers.l2(l2_reg))

        #self.gcn3 = GCNConv(n_classes,
      #                      activation="softmax",
       #                     kernel_regularizer=tf.keras.regularizers.l2(l2_reg))
       # self.dropout3 = tf.keras.layers.Dropout(dropout_rate3)

       # Dense classifier head instead of GCNConv
        #self.dense_out = tf.keras.layers.Dense(n_classes, activation="softmax",
        #    kernel_regularizer=tf.keras.regularizers.l2(l2_reg))

    def call(self, inputs, training=True):
        x, a = inputs

        x = self.gcn1([x, a])
      #  x = self.dropout1(x, training=training)

        x = self.gcn2([x, a])
        x = self.dropout2(x, training=training)

      #  x = self.gcn3([x, a])
     #   x = self.dropout3(x, training=training)

     # Final dense classifier
        x = self.dense_out(x)

        return x


#train_mask_tensor = tf.constant(np.isin(np.arange(N), idx_train), dtype=tf.float32)
#val_mask_tensor   = tf.constant(np.isin(np.arange(N), idx_val), dtype=tf.float32)

model = GCN(channels=32, n_classes=C)

#model.compile(
#    optimizer=Adam(learning_rate=1e-2),
#    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
#    weighted_metrics=["accuracy"],
#)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-2),
    loss=lambda y_true, y_pred: masked_sparse_cce(y_true, y_pred, train_mask_tf1), #Otherwise train_mask_tensor CHANGE WITH CLUSTER
    metrics=["accuracy"])
  #  jit_compile=False # Disable XLA compilation


early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

In [ ]:
N = dataset1.n_nodes #CHANGE
y_int = np.random.randint(0, 5, size=(N,))            # integer labels (0..4)


history = model.fit(
    train_loader1.load(),
    steps_per_epoch=train_loader1.steps_per_epoch,
    validation_data=val_loader1.load(),
    validation_steps=val_loader1.steps_per_epoch,
    epochs=100,
 #   callbacks=[early_stop],
    verbose=2
)

no_epochs=len(history.history['val_loss'])

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('GCN Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('GCN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Convert history to DataFrame
gcn_training_log1 = pd.DataFrame(history.history)

# Save to CSV
gcn_training_log1.to_csv("/content/drive/MyDrive/PhD data analysis/Paper1/gcn_training_log1.csv", index=False)

In [ ]:
model.save(filepath='/content/drive/MyDrive/PhD data analysis/Paper1/Models/gcn_cluster1.keras', include_optimizer=True)

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Get the actual class names from the label encoder mapping
class_names = list(label_enc21_mapping.keys()) #['Infrastructure damage', 'Other', 'Road ponding', 'Roadside fire', 'Slippery pavement']

# ----------------------------------------------------------------------
# 1) Run predictions for all nodes in the graph
# ----------------------------------------------------------------------
# Pass the entire dataset as a single batch for prediction
y_pred_softmax = model.predict([X1, A_with_loops1], batch_size=num_nodes1, verbose=0)   # shape (N, C), probabilities CHANGE PER CLUSTER
y_pred_classes = np.argmax(y_pred_softmax, axis=1)  # predicted class ids

# ----------------------------------------------------------------------
# 2) Extract the test set
# ----------------------------------------------------------------------
y_true_test = y1.iloc[idx_test1]            # true labels for test nodes using iloc
y_pred_test = y_pred_classes[idx_test1]   # predicted labels for test nodes
y_prob_test = y_pred_softmax[idx_test1]   # softmax probabilities for test nodes

# ----------------------------------------------------------------------
# 3) Confusion Matrix
# ----------------------------------------------------------------------
cm = confusion_matrix(y_true_test, y_pred_test)

plt.figure(figsize=(8,6)) # Increased figure size for better readability
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Cluster 1 - Confusion Matrix on Test Set")
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
plt.yticks(rotation=0) # Keep y-axis labels horizontal
plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()

# ----------------------------------------------------------------------
# 4) Classification Report
# ----------------------------------------------------------------------
print("Classification Report on Test Set:")
print(classification_report(y_true_test, y_pred_test, target_names=class_names))

# ----------------------------------------------------------------------
# 5) Visualize Softmax Probability Distribution
# ----------------------------------------------------------------------
# Example: plot distribution of predicted probabilities for the first 500 test nodes
plt.figure(figsize=(16,8)) # Increased figure size
for c in range(5):
   # plt.plot(y_prob_test[:500, c], label=class_names[c], alpha=0.7) # Use class names for legend
     plt.plot(y_prob_test[:500, c], label=class_names[c], alpha=0.7)


plt.title("Softmax Probability Distribution (first 500 test nodes)")
plt.xlabel("Test Node Index (subset)")
plt.ylabel("Predicted Probability")
plt.legend()
plt.show()

# Alternative: histogram of max predicted probabilities (confidence)
plt.figure(figsize=(6,4))
plt.hist(np.max(y_prob_test, axis=1), bins=20, color="skyblue", edgecolor="k")
plt.title("Model Confidence on Test Nodes")
plt.xlabel("Max Softmax Probability")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Convert history to DataFrame
gcn_training_track1 = pd.DataFrame(history.history)

# Save to CSV
gcn_training_track1.to_csv("/content/drive/MyDrive/PhD data analysis/Paper1/training_log.csv", index=False)

In [ ]:
# Evaluate the model specifically on the test subset by providing the test data directly
# Use the integer-encoded labels for the test set for this evaluation
y_true_test_int = y.iloc[idx_test1]

loss, accuracy = model.evaluate(
    [X[idx_test1], A_with_loops[np.ix_(idx_test1, idx_test1)]], # Use test subset of features and corresponding adjacency block
    y_true_test_int,                         # Use integer-encoded true labels for the test subset
    batch_size=len(idx_test1),              # Use the size of the test set as batch size
    verbose=0
)

print(f"Test Loss (evaluated on test subset): {loss:.4f}")
print(f"Test Accuracy (evaluated on test subset): {accuracy:.4f}")

Hyperparameter tuning

In [ ]:
!pip install optuna

In [ ]:
import optuna
import tensorflow as tf

def objective(trial):
    # --- Define search space ---
    gcn_units = trial.suggest_categorical('gcn_units', [16, 32, 64])
    lstm_units = trial.suggest_categorical('lstm_units', [16, 32, 64])
    l2_reg = trial.suggest_float('l2_reg', 1e-4, 1e-1, log=True)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    # Build model using selected hyperparameters
    model = build_gcn_blstm(
        window_size=X_train3.shape[1],
        n_features=X_train3.shape[2],
        n_classes=y_train3.shape[1],
        gcn_units=gcn_units,
        lstm_units=lstm_units,
        l2_reg=l2_reg
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Fit on training data with early stopping based on val_loss
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    history = model.fit(
        [X_train3, A_train3], y_train3,
        validation_data=([X_val3, A_val3], y_val3),
        batch_size=64,
        epochs=20,
        verbose=0,
        callbacks=[early_stop]
    )

    # Return the best (minimum) validation loss achieved in this trial
    val_loss = min(history.history['val_loss'])
    return val_loss

# --- Run Bayesian Optimization Study to MINIMIZE loss ---
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

print("Optimization finished.")
print("Best trial (lowest loss):", study.best_trial.params)
print("Best validation loss:", study.best_value)

In [ ]:
#Cluster 1

#Best trial (lowest loss): {'gcn_units': 32, 'lstm_units': 64, 'l2_reg': 0.00010001720739326539, 'learning_rate': 0.001628918048984873}
#Best validation loss: 1.2743843793869019

#Cluster 2
#Best trial (lowest loss): {'gcn_units': 16, 'lstm_units': 16, 'l2_reg': 0.0001624378445893729, 'learning_rate': 0.0012181902971185544}
#Best validation loss: 0.9538324475288391

#Cluster 3
#Best trial (lowest loss): {'gcn_units': 16, 'lstm_units': 16, 'l2_reg': 0.0017204290314213227, 'learning_rate': 0.009741450716059627}
#Best validation loss: 0.8025434613227844

#Cluster 4
#Best trial (lowest loss: Best validation loss: 1.2573060989379883): {'gcn_units': 32, 'lstm_units': 64, 'l2_reg': 0.0006184923563950077, 'learning_rate': 0.009303507796142264}

#Cluster 5
#Best trial (lowest loss): {'gcn_units': 32, 'lstm_units': 16, 'l2_reg': 0.0009461881837617413, 'learning_rate': 0.007935901510526294}
#Best validation loss: 0.8092283010482788

### Final Model Training with Optuna Results
Now we re-build the model using the best found hyperparameters and evaluate on the test set.

In [ ]:
best_params = study.best_params
print(f"Training final model with params: {best_params}")

final_model = build_gcn_blstm(
    window_size=X_train1.shape[1],
    n_features=X_train1.shape[2],
    n_classes=y_train1.shape[1],
    gcn_units=best_params['gcn_units'],
    lstm_units=best_params['lstm_units'],
    l2_reg=best_params['l2_reg']
)

final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['learning_rate']),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = final_model.fit(
    [X_train1, A_train1], y_train1,
    validation_data=([X_val1, A_val1], y_val1),
    batch_size=64,
    epochs=10,
    verbose=1
)

test_loss, test_acc = final_model.evaluate([X_test1, A_test1], y_test1)
print(f"Final Test Accuracy with Bayesian Optimized Model: {test_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('GCN-BLSTM Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('GCN-BLSTM Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Convert history to DataFrame
gcn_training_log1 = pd.DataFrame(history.history)

# Save to CSV
gcn_training_log1.to_csv("/content/drive/MyDrive/PhD data analysis/Paper1/gcnblstm_training_log1.csv", index=False)

model.save(filepath='/content/drive/MyDrive/PhD data analysis/Paper1/Models/gcnblstm1.keras', include_optimizer=True)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# -----------------------------
# 4) Predictions
# -----------------------------

y_pred_probs = model.predict([X_test1, A_test1])          # softmax probabilities
y_pred = np.argmax(y_pred_probs, axis=1)                # predicted classes
y_true = np.argmax(y_test1, axis=1)                      # true classes

class_names = list(label_enc21_mapping.keys())
# -----------------------------
# 5) Classification report
# -----------------------------
print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=2, target_names=class_names))

# -----------------------------
# 6) Confusion matrix
# -----------------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Cluster 1")
plt.show()

# -----------------------------
# 7) Inspect softmax probabilities
# -----------------------------
print("\nSoftmax probabilities (first 5 test samples):")
print(y_pred_probs[:5])

# 8) Visualize Softmax Probability Distribution
# ----------------------------------------------------------------------
# Example: plot distribution of predicted probabilities for the first 500 test nodes
plt.figure(figsize=(16,8)) # Increased figure size
for c in range(C):
   # plt.plot(y_prob_test[:500, c], label=class_names[c], alpha=0.7) # Use class names for legend
     plt.plot(y_pred_probs[0:2300, c], label=class_names[c], alpha=0.7)


plt.title("Softmax Probability Distribution over all nodes")
plt.xlabel("Test Node Index (subset)")
plt.ylabel("Predicted Probability")
plt.legend()
plt.show()

# Alternative: histogram of max predicted probabilities (confidence)
plt.figure(figsize=(6,4))
plt.hist(np.max(y_pred_probs, axis=1), bins=20, color="skyblue", edgecolor="k")
plt.title("Model Confidence on Test Nodes")
plt.xlabel("Max Softmax Probability")
plt.ylabel("Frequency")
plt.show()

In [ ]:
y_prob_pd = pd.DataFrame(y_pred_probs, columns=class_names)
y_prob_pd.to_csv('drive/MyDrive/PhD data analysis/Paper1/test1_gcnblstm.csv', header=True, index=True)


y_prob_pd.shape

In [ ]:
from collections import Counter

counts = Counter(y_pred)

print(counts)

In [ ]:

unique, counts = np.unique(y_pred, return_counts=True)
print(dict(zip(unique, counts)))

In [ ]:
unique, counts = np.unique(y_true, return_counts=True)
print(dict(zip(unique, counts)))

In [ ]:
y_pred[800:1000]

# Statistical significance testing for test performance

In [ ]:
#Friedman test


# Accuracy scores: rows = clusters, columns = models
# Shape: (n_clusters, n_models)
accuracies = np.array([
    [0.53, 0.56, 0.37, 0.53],
    [0.48, 0.48, 0.37, 0.53],
    [0.55, 0.49, 0.52, 0.59],
    [0.47, 0.49, 0.37, 0.49],
    [0.53, 0.53, 0.56, 0.62],
])

model_names = ["LSTM", "BLSTM", "GCN", "GCN_BLSTM"]

from scipy.stats import friedmanchisquare

# Transpose so each argument = one model
stat, p_value = friedmanchisquare(
    accuracies[:, 0],
    accuracies[:, 1],
    accuracies[:, 2],
    accuracies[:, 3]
)

print(f"Friedman statistic = {stat:.3f}")
print(f"p-value = {p_value:.4f}")

In [ ]:
#Nemenyi-Friedman

!pip install scikit-posthocs

In [ ]:
import scikit_posthocs as sp

nemenyi = sp.posthoc_nemenyi_friedman(accuracies)
print(nemenyi)

In [ ]:
#PostHoc (no difference, so we report effect size)

import numpy as np
from scipy.stats import rankdata

n, k = accuracies.shape  # n=clusters, k=models

ranks = np.array([rankdata(row) for row in accuracies])
mean_ranks = ranks.mean(axis=0)

W = (12 * n * np.sum((mean_ranks - (k + 1)/2)**2)) / (n**2 * (k**2 - 1))
print("Kendall's W:", W)

In [ ]:
ranks

Time-windowing

In [ ]:
import keras
import tensorflow as tf


generator11 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train1,
                                                                y_train1,
                                                                length=30, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator12 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val1,
                                                                y_val1,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator13 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test1,
                                                                y_test1,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator21 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train2,
                                                                y_train2,
                                                                length=30, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator22 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val2,
                                                                y_val2,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator23 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test2,
                                                                y_test2,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator31 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train3,
                                                                y_train3,
                                                                length=30, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator32 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val3,
                                                                y_val3,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator33 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test3,
                                                                y_test3,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator41 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train4,
                                                                y_train4,
                                                                length=30, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator42 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val4,
                                                                y_val4,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator43 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test4,
                                                                y_test4,
                                                                length=30,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator51 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_train5,
                                                                y_train5,
                                                                length=15, #Length of the output sequences (in number of timesteps).
                                                                stride=1, #Period between successive individual timesteps within sequences. For rate r, timesteps data[i], data[i-r], ... data[i - length] are used for create a sample sequence.
                                                                sampling_rate=1, #Period between successive output sequences. For stride s, consecutive output samples would be centered around data[i], data[i+s], data[i+2*s], etc.
                                                                batch_size=1)

generator52 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_val5,
                                                                y_val5,
                                                                length=15,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

generator53 = tf.keras.preprocessing.sequence.TimeseriesGenerator(X_test5,
                                                                y_test5,
                                                                length=15,
                                                                stride=1,
                                                                sampling_rate=1,
                                                                batch_size=1)

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)
X_test = np.array(X_test)
y_test = np.array(y_test)


print("No. of train samples by generator - red cluster:", len(generator11))#, "Training data shape:", X_train.shape, y_train.shape)
print("No. of val samples by generator - red cluster:", len(generator12))#, "Validation data shape:", X_val.shape, y_val.shape)
print("No of. test samples by generator - red cluster :", len(generator13))#, "Test data shape:", X_test.shape, y_test.shape)
print("No. of train samples by generator - blue cluster:", len(generator21))#, "Training data shape:", X_train.shape, y_train.shape)
print("No. of val samples by generator - blue cluster:", len(generator22))#, "Validation data shape:", X_val.shape, y_val.shape)
print("No of. test samples by generator - blue cluster :", len(generator23))#, "Test data shape:", X_test.shape, y_test.shape)
print("No. of train samples by generator - green cluster:", len(generator31))#, "Training data shape:", X_train.shape, y_train.shape)
print("No. of val samples by generator - green cluster:", len(generator32))#, "Validation data shape:", X_val.shape, y_val.shape)
print("No of. test samples by generator - green cluster :", len(generator33))#, "Test data shape:", X_test.shape, y_test.shape)
print("No. of train samples by generator - grey cluster:", len(generator41))#, "Training data shape:", X_train.shape, y_train.shape)
print("No. of val samples by generator - grey cluster:", len(generator42))#, "Validation data shape:", X_val.shape, y_val.shape)
print("No of. test samples by generator - grey cluster :", len(generator43))#, "Test data shape:", X_test.shape, y_test.shape)
print("No. of train samples by generator - purple cluster:", len(generator51))#, "Training data shape:", X_train.shape, y_train.shape)
print("No. of val samples by generator - purple cluster:", len(generator52))#, "Validation data shape:", X_val.shape, y_val.shape)
print("No of. test samples by generator - purple cluster :", len(generator53))#, "Test data shape:", X_test.shape, y_test.shape)

print(f"Training batches: {len(generator11)}, Validation batches: {len(generator12)}")
print(f"One batch x shape: {generator11[0][0].shape}, y shape: {generator11[0][1].shape}")

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)
X_test = np.array(X_test)
y_test = np.array(y_test)


**LSTM Model building**

In [ ]:
#Model building POTENTIALLY ADD AUTO-ENCODER architecture for feature extraction
#Add incident duration as an output

#-CLUSTER 3 in PAPER - Cluster 1 = Red region (778) - train1,val1,test1
#-CLUSTER 1 IN PAPER - Cluster 2 = Blue region (1184) - train2,val2,test2
#-CLUSTER 2 in PAPER - Cluster 3 = Green region (1256) - train3,val3,test3
#-CLUSTER 4 in PAPER - Cluster 4 = Grey region (1193) - train4,val4,test4
#-CLUSTER 4 in PAPER - Cluster 5 = Purple region (351) - train5,val5,test5

from tensorflow import keras
from keras.layers import Dense, LSTM, Bidirectional

model = keras.models.Sequential()

# First Layer
model.add(keras.layers.LSTM(32, return_sequences=True, input_shape=(30,6))) #30 is timestamps, 6 is no. of features (CHANGE TO 15 FOR LIMBURG)

# Second Layer
model.add(keras.layers.LSTM(32, return_sequences=False))#, activation="relu"))
model.add(keras.layers.LeakyReLU(alpha=0.1))

# 3rd Layer (Dense)
#model.add(keras.layers.Dense(32))#, activation="relu"))
#model.add(keras.layers.LeakyReLU(alpha=0.1))

# 4th Layer (Dropout)
model.add(keras.layers.Dropout(0.5))

# Final Output Layer
model.add(keras.layers.Dense(3, activation='softmax'))

model.summary()

callback = keras.callbacks.EarlyStopping(monitor='val_loss',
                                              patience=3)
model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                optimizer=tf.keras.optimizers.Adamax(learning_rate=0.001,
                                                     beta_1=0.9,
                                                     beta_2=0.999),
                metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

training = model.fit(generator21, epochs=30, batch_size=32, validation_data=generator22,
                     callbacks=[callback],verbose=1)

#training = model.fit(x61, y_train6, epochs=30,
#                     batch_size=32, validation_data=(x52, y_val5),
#                     callbacks=[callback])
no_epochs=len(training.history['val_loss'])

In [ ]:
#Plotting
no_epochs=len(training.history['loss'])
acc = model.history.history['sparse_categorical_accuracy']
val_acc = model.history.history['val_sparse_categorical_accuracy']
loss = model.history.history['loss']
val_loss = model.history.history['val_loss']
epochs_range = range(no_epochs)

In [ ]:
plt.figure(figsize=(14, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.xlabel('Number of epochs')
plt.ylabel('Accuracy')
plt.title('Sparse Categorical Accuracy (Training and validation)')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.xlabel('Number of epochs')
plt.ylabel('Loss')
plt.title('Sparse Categorical Cross Entropy (Training and validation)')
plt.show()

In [ ]:
#Testing

predictions = model.predict(generator23) #CHANGE WITH CLUSTER
y_pred = np.argmax(predictions, axis=1)


In [ ]:
predictions[:5]
df_pred = pd.DataFrame(predictions)
df_pred.to_csv('drive/MyDrive/PhD data analysis/Paper1/df_pred.csv', header=True, index=True)

In [ ]:
#Generator true labels
test_gen = generator23 #CHANGE WITH EACH CLUSTER to generator13 - 53

y_true = []
for i in range(len(test_gen)):
    y_batch = test_gen[i][1]
    # If one-hot encoded:
    if y_batch.ndim > 1:
        y_true.extend(np.argmax(y_batch, axis=1))
    else:
        y_true.extend(y_batch)
y_true = np.array(y_true)

print(len(y_true))
print(len(y_pred))

In [ ]:
#Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, multilabel_confusion_matrix

# Compute confusion matrix and report
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred)

ConfusionMatrixDisplay.from_predictions(
   y_true, y_pred, display_labels=label_enc21.classes_, xticks_rotation='vertical')
plt.title('Confusion Matrix - Cluster 5 (Purple region)') #CHANGE THIS ACCORDING TO CLUSTER
print("Confusion Matrix:\n", cm)


In [ ]:
#Classification report
# Get unique labels from y_test
unique_labels = np.unique(y_true)

# Filter target_names to match the unique labels
filtered_target_names = [label_enc21.classes_[i] for i in unique_labels]

# Now, generate the classification report with the filtered target names
report = classification_report(y_true, y_pred, target_names=filtered_target_names, labels=unique_labels)
print(report)

In [ ]:
loss, accuracy = model.evaluate(generator23) #Change for each cluster
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

Mapping LSTM results

In [ ]:
#Generator true labels
test_gen = generator23 #CHANGE WITH EACH CLUSTER to generator13 - 53

y_true = []
for i in range(len(test_gen)):
    y_batch = test_gen[i][1]
    # If one-hot encoded:
    if y_batch.ndim > 1:
        y_true.extend(np.argmax(y_batch, axis=1))
    else:
        y_true.extend(y_batch)
y_true = np.array(y_true)

print(len(y_true))
print(len(y_pred))

In [ ]:
# Let's iterate and see indices of the test datasets

gen = generator23 #CHANGE WITH EACH CLUSTER CHANGE from generator13 - 53
look_back = 30 #15 for cluster 5
indices = []
for i in range(len(gen)):
    X, y = gen[i]
    target_index = i + look_back
    print(f"Generator index {i} => target index in data: {target_index}, target value={y.flatten()[0]}")
    indices.append(target_index)

In [ ]:
#Coordinates of y targets

test_set = test2.copy() #CHANGE FOR EVERY CLUSTER TO TEST1 - TEST5
test_set = test_set[['lat', 'lon', 'incident_type', 'duration_min']]
windowed_test = test_set.iloc[indices]

print("Test_set shape:", test_set.shape, "Windowed_test shape", windowed_test.shape)

In [ ]:
test_set.shape

In [ ]:
windowed_test.shape

In [ ]:
#Mapping true + predicted labels

true_map = folium.Map(location=(51.903, 5.187))

In [ ]:
true_test = gpd.GeoDataFrame(windowed_test, geometry=gpd.points_from_xy(windowed_test.lat, windowed_test.lon), crs="EPSG:28992")
true_list = [[point.xy[0][0], point.xy[1][0]] for point in true_test.geometry]

In [ ]:
label_enc21_mapping = dict(zip(label_enc21.classes_, label_enc21.transform(label_enc21.classes_))) #Change to label_enc11 for previous categories
print(label_enc21_mapping)

In [ ]:
true_test['incident_type'].unique() #{'Other': np.int64(0), 'Road ponding/slippery pavement': np.int64(1), 'Roadside fire': np.int64(2)}

i = 0
for coordinates in true_list:
    if true_test.incident_type[i] == 0: #'Other'
        type_color = "black"
    elif true_test.incident_type[i] == 1: #'Road ponding/slippery pavement'
        type_color = "blue"
    elif true_test.incident_type[i] == 2: #Roadside fire
        type_color = "orange"
   # elif true_test.incident_type[i] == 3: #Slippery pavement
   #     type_color = "green"
   # elif true_test.incident_type[i] == 4: #Roadside fires
   #     type_color = "orange"
   # elif true_test.incident_type[i] == 5: #Slippery pavement
    #    type_color = "darkred"
    #elif df1.incident_type[i] == 'Gladheid':
    #    type_color = "green"
    #else:
    #    type_color = "black"


    true_map.add_child(
        folium.Marker(
            location=coordinates,
            #icon=folium.Icon(color="%s" % type_color),
            popup= "Incident duration: "
            + str(true_test.duration_min[i])
            + "<br>"
            + "Incident type: "
            + str(true_test.incident_type[i]),
            icon=folium.Icon(color="%s" % type_color),

        )
    )
    i = i + 1

true_map

true_map

In [ ]:
#Adding the predicted labels

pred_test = true_test.copy()
pred_test['test_label'] = y_pred

In [ ]:
pred_map = folium.Map(location=(51.903, 5.187))
pred_list = true_list.copy()

In [ ]:
pred_test['test_label'].unique() #{'Other': np.int64(0), 'Road ponding/slippery pavement': np.int64(1), 'Roadside fire': np.int64(2)}

i = 0
for coordinates in pred_list:
    if pred_test.test_label[i] == 0: #'Other'
        type_color = "black"
    elif pred_test.test_label[i] == 1: #'Road ponding/slippery pavement'
        type_color = "blue"
    elif pred_test.test_label[i] == 2: #'Roadside fires'
        type_color = "orange"
#    elif pred_test.test_label[i] == 3: #'Slippery pavement'
  #      type_color = "green"
   # elif pred_test.test_label[i] == 4: #Roadside fires
    #    type_color = "orange"
  #  elif pred_test.test_label[i] == 5: #Slippery pavement
   #     type_color = "darkred"
    #elif df1.incident_type[i] == 'Gladheid':
    #    type_color = "green"
    #else:
    #    type_color = "black"


    pred_map.add_child(
        folium.Marker(
            location=coordinates,
            #icon=folium.Icon(color="%s" % type_color),
            popup= "Incident type: "
            + str(pred_test.test_label[i]),
            icon=folium.Icon(color="%s" % type_color),

        )
    )
    i = i + 1

pred_map

pred_map

In [ ]:
#Saving the maps
true_map.save("LSTM_TrueURC_cluster4.html")
pred_map.save("LSTM_PredURC_cluster4.html")

BLSTM model

In [ ]:
#BiLSTM model

#-CLUSTER 1 IN PAPER - Cluster 2 = Blue region (1184) - train2,val2,test2
#-CLUSTER 2 in PAPER - Cluster 3 = Green region (1256) - train3,val3,test3
#-CLUSTER 3 in PAPER - Cluster 1 = Red region (778) - train1,val1,test1
#-CLUSTER 4 in PAPER - Cluster 4 = Grey region (1193) - train4,val4,test4
#-CLUSTER 4 in PAPER - Cluster 5 = Purple region (351) - train5,val5,test5

from tensorflow import keras
from keras.layers import Dense, LSTM, Bidirectional

model1 = keras.models.Sequential()

# First Layer
#model1.add(keras.layers.Bidirectional(LSTM(16, return_sequences=True), input_shape=(X_train.shape[1],1))) #For generator, shape is (sequence length of generator, total amount of variables)
model1.add(keras.layers.Bidirectional(LSTM(32, return_sequences=True), input_shape=(30,6)))

# Second Layer
model1.add(keras.layers.Bidirectional(LSTM(32, return_sequences=False)))#, activation="relu"))
model1.add(keras.layers.LeakyReLU(alpha=0.1))

# 3rd Layer (Dense)
#model.add(keras.layers.Dense(32, activation="relu"))

# 4th Layer (Dropout)
model1.add(keras.layers.Dropout(0.5))

# Final Output Layer
model1.add(keras.layers.Dense(3, activation='softmax'))

model1.summary()
callback = keras.callbacks.EarlyStopping(monitor='val_loss',
                                              patience=2)
model1.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                optimizer=tf.keras.optimizers.Adamax(learning_rate=0.001,
                                                     beta_1=0.9,
                                                     beta_2=0.999),
                metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

training = model1.fit(generator41, epochs=30, batch_size=32, validation_data=generator42,
                     callbacks=[callback],verbose=1)

#training = model.fit(x61, y_train6, epochs=30,
#                     batch_size=32, validation_data=(x52, y_val5),
#                     callbacks=[callback])
no_epochs=len(training.history['val_loss'])

In [ ]:
#Plotting

acc = model1.history.history['sparse_categorical_accuracy']
val_acc = model1.history.history['val_sparse_categorical_accuracy']
loss = model1.history.history['loss']
val_loss = model1.history.history['val_loss']
epochs_range = range(no_epochs)

In [ ]:
plt.figure(figsize=(14, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.xlabel('Number of epochs')
plt.ylabel('Accuracy')
plt.title('BLSTM Sparse Categorical Accuracy (Training and validation)')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.xlabel('Number of epochs')
plt.ylabel('Loss')
plt.title('BLSTM Sparse Categorical Cross Entropy (Training and validation)')
plt.show()

In [ ]:
#Testing

predictions = model1.predict(generator43) #CHANGE WITH EACH CLUSTER to generator13 - 53
y_pred = np.argmax(predictions, axis=1)


In [ ]:
#Saving the probabilities
predictions[:5]
df_pred1 = pd.DataFrame(predictions)
df_pred1.to_csv('drive/MyDrive/PhD data analysis/Paper1/df_pred1.csv', header=True, index=True)

In [ ]:
#Generator true labels
test_gen = generator43 #CHANGE WITH EACH CLUSTER to generator13 - 53

y_true = []
for i in range(len(test_gen)):
    y_batch = test_gen[i][1]
    # If one-hot encoded:
    if y_batch.ndim > 1:
        y_true.extend(np.argmax(y_batch, axis=1))
    else:
        y_true.extend(y_batch)
y_true = np.array(y_true)

print(len(y_true))
print(len(y_pred))

In [ ]:
#Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, multilabel_confusion_matrix

# Compute confusion matrix and report
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred)

ConfusionMatrixDisplay.from_predictions(
   y_true, y_pred, display_labels=label_enc21.classes_, xticks_rotation='vertical')
plt.title('Confusion Matrix BLSTM - Cluster 5 (Purple region)') #CHANGE THIS ACCORDING TO CLUSTER
print("Confusion Matrix:\n", cm)


In [ ]:
#Classification report
# Get unique labels from y_test
unique_labels = np.unique(y_true)

# Filter target_names to match the unique labels
filtered_target_names = [label_enc21.classes_[i] for i in unique_labels]

# Now, generate the classification report with the filtered target names
report = classification_report(y_true, y_pred, target_names=filtered_target_names, labels=unique_labels)
print(report)

In [ ]:
loss, accuracy = model1.evaluate(generator53) #CHANGE W CLUSTER
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

Mapping BLSTM results

In [ ]:
#Generator true labels
test_gen = generator53 #CHANGE WITH EACH CLUSTER to generator13 - 53

y_true = []
for i in range(len(test_gen)):
    y_batch = test_gen[i][1]
    # If one-hot encoded:
    if y_batch.ndim > 1:
        y_true.extend(np.argmax(y_batch, axis=1))
    else:
        y_true.extend(y_batch)
y_true = np.array(y_true)

print(len(y_true))
print(len(y_pred))

In [ ]:
# Let's iterate and see indices of the test datasets

gen = generator23 #CHANGE WITH EACH CLUSTER CHANGE from generator13 - 53
look_back = 30 #15 for cluster 5
indices = []
for i in range(len(gen)):
    X, y = gen[i]
    target_index = i + look_back
    print(f"Generator index {i} => target index in data: {target_index}, target value={y.flatten()[0]}")
    indices.append(target_index)

In [ ]:
test5.iloc[-10:]

In [ ]:
#Coordinates of y targets

test_set = test5.copy() #CHANGE FOR EVERY CLUSTER TO TEST1 - TEST5
test_set = test_set[['lat', 'lon', 'incident_type', 'duration_min']]
windowed_test = test_set.iloc[indices]

print("Test_set shape:", test_set.shape, "Windowed_test shape", windowed_test.shape)

In [ ]:
x_cluster2 = test2.iloc[30:]
x_cluster2.to_csv('drive/MyDrive/PhD data analysis/Paper1/x_cluster2.csv', header=True, index=True)

In [ ]:
test_set.shape

In [ ]:
windowed_test.shape

In [ ]:
#Mapping true + predicted labels

true_map = folium.Map(location=(51.903, 5.187))

In [ ]:
true_test = gpd.GeoDataFrame(windowed_test, geometry=gpd.points_from_xy(windowed_test.lat, windowed_test.lon), crs="EPSG:28992")
true_list = [[point.xy[0][0], point.xy[1][0]] for point in true_test.geometry]

In [ ]:
label_enc21_mapping = dict(zip(label_enc21.classes_, label_enc21.transform(label_enc21.classes_))) #Change to label_enc11 for previous categories
print(label_enc21_mapping)

In [ ]:
true_test['incident_type'].unique() #{'Other': np.int64(0), 'Road ponding/slippery pavement': np.int64(1), 'Roadside fire': np.int64(2)}
i = 0
for coordinates in true_list:
    if true_test.incident_type[i] == 0: #'Other'
        type_color = "black"
    elif true_test.incident_type[i] == 1: #'Road ponding/slippery pavement'
        type_color = "blue"
    elif true_test.incident_type[i] == 2: #Roadside fire
        type_color = "orange"
  #  elif true_test.incident_type[i] == 3: #Slippery pavement
  #      type_color = "green"
   # elif true_test.incident_type[i] == 4: #Roadside fires
   #     type_color = "orange"
   # elif true_test.incident_type[i] == 5: #Slippery pavement
    #    type_color = "darkred"
    #elif df1.incident_type[i] == 'Gladheid':
    #    type_color = "green"
    #else:
    #    type_color = "black"


    true_map.add_child(
        folium.Marker(
            location=coordinates,
            #icon=folium.Icon(color="%s" % type_color),
            popup= "Incident duration: "
            + str(true_test.duration_min[i])
            + "<br>"
            + "Incident type: "
            + str(true_test.incident_type[i]),
            icon=folium.Icon(color="%s" % type_color),

        )
    )
    i = i + 1

true_map

true_map

In [ ]:
#Adding the predicted labels

pred_test = true_test.copy()
pred_test['test_label'] = y_pred

In [ ]:
pred_map = folium.Map(location=(51.903, 5.187))
pred_list = true_list.copy()

In [ ]:
pred_test['test_label'].unique() #{'Other': np.int64(0), 'Road ponding/slippery pavement': np.int64(1), 'Roadside fire': np.int64(2)}

i = 0
for coordinates in pred_list:
    if pred_test.test_label[i] == 0: #'Other'
        type_color = "black"
    elif pred_test.test_label[i] == 1: #'Road ponding/slippery pavement'
        type_color = "blue"
    elif pred_test.test_label[i] == 2: #'Roadside fires'
        type_color = "orange"
 #   elif pred_test.test_label[i] == 3: #'Slippery pavement'
  #      type_color = "green"
   # elif pred_test.test_label[i] == 4: #Roadside fires
    #    type_color = "orange"
  #  elif pred_test.test_label[i] == 5: #Slippery pavement
   #     type_color = "darkred"
    #elif df1.incident_type[i] == 'Gladheid':
    #    type_color = "green"
    #else:
    #    type_color = "black"


    pred_map.add_child(
        folium.Marker(
            location=coordinates,
            #icon=folium.Icon(color="%s" % type_color),
            popup= "Incident type: "
            + str(pred_test.test_label[i]),
            icon=folium.Icon(color="%s" % type_color),

        )
    )
    i = i + 1

pred_map

pred_map

In [ ]:
#Saving the maps
true_map.save("TrueURC_cluster1.html")
pred_map.save("PredURC_cluster1.html")

Entire dataset preprocessing

In [ ]:
b_df = bootstrap_spatiotemporal_naive(df1, n_samples=10000) #Change to 50000 later
b_df[0:5]
#print(b_df.shape)

In [ ]:
#Sort according to time index
b_df = b_df.sort_values(['start_date'], ascending=[True])
b_df = b_df.set_index('start_date')
print(b_df.head(-10))

In [ ]:
print(b_df.head(-10))
print(b_df.shape)

In [ ]:
incidents = b_df['incident_type'].value_counts()
print(incidents)

In [ ]:
#Standardization of infrastructure and temperature data

from sklearn import preprocessing

infra_col = ['low_veg_%',
               'mid_high_veg_%',
               'pavement_%',
               'water_%']
spatial_col = ['x_coor', 'y_coor', 'z_coor']
temp = ['max_temp']

scaler = preprocessing.MinMaxScaler()
b_df[infra_col] = scaler.fit_transform(b_df[infra_col])
b_df[temp] = scaler.fit_transform(b_df[temp])

#Log 1-p scaling of precipitation data

sum1 = b_df['sum_prec (1h)']
logsum1 = np.log1p(sum1)
b_df['sum_prec (1h)'] = logsum1


In [ ]:
b_df.head()

In [ ]:
#Data splitting

attributes = b_df[['max_temp',
               'sum_prec (1h)',
               'low_veg_%',
               'mid_high_veg_%',
               'pavement_%',
               'water_%']]#.to_numpy().tolist()
spatial = b_df[['x_coor', 'y_coor', 'z_coor']]#.to_numpy().tolist()
coords = b_df[['lat', 'lon']]
output = b_df['incident_type']#tolist()

coords1 = coords[0:2000]
attributes1 = attributes[0:2000]
output1 = output[0:2000]

Weighted adjacency matrix (incl. spatial + contextual information)

In [ ]:
#Calculate spatial distance using Haversine #LOAD THEM FOR NEXT TIME

#!pip install haversine
#from haversine import haversine, Unit

#n = len(coords)
#spatial_dist = np.zeros((n, n))

#for i in range(n):
#    for j in range(n):
#        if i != j:
#            spatial_dist[i, j] = haversine(tuple(coords.iloc[i]), tuple(coords.iloc[j]), unit=Unit.KILOMETERS)

#spatial_dist2 = pd.DataFrame(spatial_dist)
#spatial_dist2.to_csv('drive/MyDrive/PhD data analysis/Paper1/spatial_dist_20000.csv', header=True, index=True)

In [ ]:
#Save coordinates

#spatial_dist2 = pd.DataFrame(spatial_dist)
#spatial_dist2.to_csv('drive/MyDrive/PhD data analysis/Paper1/spatial_dist_50000.csv', header=True, index=True)

In [ ]:
#Load the bootstrapped spatial matrix

spatial_dist = pd.read_csv("/content/drive/MyDrive/PhD data analysis/Paper1/spatial_dist_10000.csv") #Change accordingly, depending on which dataset to use
spatial_dist = spatial_dist.drop(columns=['Unnamed: 0'])
#spatial_sim = pd.read_csv("/content/drive/MyDrive/PhD data analysis/Paper1/spatial_sim_30000.csv") #Change accordingly, depending on which dataset to use
#spatial_sim = spatial_sim.drop(columns=['Unnamed: 0'])

In [ ]:
spatial_dist.shape

In [ ]:
#Convert to similarity (Exponential decay)

spatial_sim = np.exp(-spatial_dist / spatial_dist.std())
spatial_sim

In [ ]:
#spatial_sim.to_csv('drive/MyDrive/PhD data analysis/Paper1/spatial_sim_30000.csv', header=True, index=True)

In [ ]:
spatial_sim.shape

In [ ]:
#A = spatial_sim.copy()

In [ ]:
# @title Default title text
#Attribute similarity using Cosine similarity

from sklearn.metrics.pairwise import cosine_similarity

attr_sim = cosine_similarity(attributes)

In [ ]:
attr_sim[0]
attr_sim[attr_sim<0] #No negative values, so no opposities, only similarities (good news)

In [ ]:
attr_sim.shape

In [ ]:
attr_sim[0]

In [ ]:
#Combine spatial + attribute similarities - Weighted sum or product (tunable hyperparameter alpha):

alpha = 0.5  # weight for spatial vs attribute
combined_sim = alpha * spatial_sim + (1 - alpha) * attr_sim

In [ ]:
#combined_sim[1:10]

In [ ]:
#Weighted adjacency matrix

#threshold = 0.6
#combined_sim[combined_sim < threshold] = 0 #All points less than the threshold are 0
A = combined_sim.copy()

In [ ]:
A.shape

In [ ]:
A[1:10]

In [ ]:
#Add self loops

from spektral.utils import add_self_loops

A_np = A.to_numpy()
A_with_loops = add_self_loops(A_np)

print(A_with_loops.shape)

In [ ]:
A_with_loops[1:10]

In [ ]:
#Preprocessing the adjacency matrix
from spektral.utils import add_self_loops, normalized_adjacency, gcn_filter

# Normalize adjacency
A_with_loops = gcn_filter(A_with_loops).astype("float32")

from scipy.sparse import csr_matrix

# Convert the dense numpy array to a CSR sparse matrix
A_sparse = csr_matrix(A_with_loops)

print(A_sparse.shape)
#print(A_sparse)

In [ ]:
#Create Spektral dataset

from spektral.data import Dataset, Graph

features_com = np.hstack([attributes, spatial])
#features_com_np = features_com.to_numpy()

graph = Graph(x=features_com, a=A_sparse, y=output) #Separate graph

class SingleGraphDataset(Dataset):
    def read(self):
        return [Graph(x=features_com, a=A_sparse, y=output)]

In [ ]:
dataset = SingleGraphDataset()
graph = dataset[0]

In [ ]:
graph

In [ ]:
dataset

In [ ]:

train = b_df.loc['2014-01-01':'2020-01-01']
val = b_df.loc['2020-01-01':'2021-01-01']
test = b_df.loc['2021-01-01':]

#Receive indices for train, val, test (in row number)

no_train = len(train)
no_val = len(val)
no_test = len(test)

N = len(features_com)
n_train = int(0.7 * N)   # 7000
n_val   = int(0.1 * N)   # 1000
n_test  = N - n_train - n_val  # 2000

idx_train = np.arange(0, n_train)
idx_val   = np.arange(n_train, n_train + n_val)
idx_test  = np.arange(n_train + n_val, N)

idx_train1 = np.arange(0, (no_train-1))
idx_val1   = np.arange(no_train, no_train + no_val)
idx_test1  = np.arange(no_train + no_val, N)

In [ ]:
train_incidents = train['incident_type'].value_counts()
print("Train incidents", train_incidents)
validation_incidents = val['incident_type'].value_counts()
print("Validation incidents", validation_incidents)
test_incidents = test['incident_type'].value_counts()
print("Test incidents", test_incidents)

In [ ]:
from spektral.data import SingleLoader

# Create masks for splitting
train_mask_tf = tf.constant(np.isin(np.arange(N), idx_train1), dtype=tf.float32)
val_mask_tf   = tf.constant(np.isin(np.arange(N), idx_val1), dtype=tf.float32)
test_mask_tf  = tf.constant(np.isin(np.arange(N), idx_test1), dtype=tf.float32)

# Create SingleLoader for each set (assuming full-batch training)
train_loader = SingleLoader(dataset, sample_weights=train_mask_tf)
val_loader   = SingleLoader(dataset, sample_weights=val_mask_tf)
test_loader  = SingleLoader(dataset, sample_weights=test_mask_tf)

# Proper masked loss (Sparse CE example; use CategoricalCE if you one-hot)
def masked_sparse_cce(y_true, y_pred, mask_tf):
    per_node = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)  # shape (N,)
    per_node = tf.cast(per_node, tf.float32) * mask_tf                          # zero out unmasked
    return tf.reduce_sum(per_node) / tf.reduce_sum(mask_tf)                     # average, not sum!

DisJointLoader mode

In [ ]:
#1 Split the data according to timestamp

train = b_df.loc['2014-01-01':'2020-01-01']
val = b_df.loc['2020-01-01':'2021-01-01']
test = b_df.loc['2021-01-01':]

#Receive indices for train, val, test (in row number)

print(len(train))
print(len(val))
print(len(test))

In [ ]:
from spektral.data import Graph, Dataset

#Create Spektral dataset

from spektral.data import Dataset, Graph

features_com = np.hstack([attributes, spatial])
class TrainingData(Dataset):
    def read(self):
        return [Graph(x=features_com, a=A_sparse, y=output)]

dataset = TrainingData()
graph = dataset[0]
num_nodes = len(features_com)

In [ ]:
dataset

In [ ]:
N = len(features_com)
n_train = int(0.7 * N)   # 7000
n_val   = int(0.1 * N)   # 1000
n_test  = N - n_train - n_val  # 2000

idx_train = np.arange(0, n_train)
idx_val   = np.arange(n_train, n_train + n_val)
idx_test  = np.arange(n_train + n_val, N)

In [ ]:
from spektral.data import DisjointLoader

DJloader = DisjointLoader(dataset, batch_size=128)

In [ ]:
batch = DJloader.__next__()
inputs, target = batch

In [ ]:
x, a, i = inputs

In [ ]:
print(x.shape)
print(a.shape)
print(i.shape)
print(target.shape)

In [ ]:
from spektral.utils import normalized_adjacency

#def masked_sparse_cce(y_true, y_pred, mask):
    # y_true: (N,)
    # y_pred: (N, C)
#    mask = tf.cast(mask, tf.float32)
#    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
#    loss *= mask
 #   return tf.reduce_sum(loss) / tf.reduce_sum(mask)


# Proper masked loss (Sparse CE example; use CategoricalCE if you one-hot)
#def masked_sparse_cce(y_true, y_pred, mask_tf):
#    per_node = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)  # shape (N,)
#    per_node = tf.cast(per_node, tf.float32) * mask_tf                          # zero out unmasked
 #   return tf.reduce_sum(per_node) / tf.reduce_sum(mask_tf)                     # average, not sum!

# Task
Generate a choropleth map of the Netherlands using the data in `df_1`, with the color gradient representing the precipitation rate.

## Download geographical data

### Subtask:
Download a shapefile or GeoJSON of the Netherlands' administrative boundaries.


In [ ]:
import requests
import json
import pandas as pd
import plotly.express as px
import geopandas as gpd

# GeoJSON link for Netherlands provinces
geojso_url = "https://cartomap.github.io/nl/wgs84/provincie_2023.geojson"

resp = requests.get(geojso_url)
resp.raise_for_status() # Ensure the request was successful
geojson = resp.json()

# Load the GeoJSON into a GeoDataFrame
provinces_gdf = gpd.GeoDataFrame.from_features(geojson["features"])

# Ensure the coordinate reference system (CRS) is set for spatial operations
# Assuming WGS84 (EPSG:4326) for the GeoJSON. You might need to verify this.
provinces_gdf.crs = "EPSG:4326"

# Convert df_1 to a GeoDataFrame
# Ensure 'start_date' is not the index, or reset it if needed for lat/lon access
# Assuming df_1 already has 'lat' and 'lon' columns
incidents_gdf = gpd.GeoDataFrame(
    df_1.copy().reset_index() if 'start_date' in df_1.index.names else df_1.copy(),
    geometry=gpd.points_from_xy(df_1.lon, df_1.lat),
    crs="EPSG:4326"
)

# Perform a spatial join to assign each incident to a province
# 'sjoin' performs a spatial join between two GeoDataFrames
incidents_with_provinces = gpd.sjoin(incidents_gdf, provinces_gdf, how="left", predicate='within')

# Aggregate precipitation data by province (e.g., mean sum_prec (1h))
# Filter out incidents not within any province (where 'statnaam' is NaN)
precipitation_by_province = incidents_with_provinces.dropna(subset=['statnaam']).groupby('statnaam')['sum_prec (1h)'].mean().reset_index()
precipitation_by_province.rename(columns={'statnaam': 'province', 'sum_prec (1h)': 'avg_precipitation'}, inplace=True)

# Plot the choropleth map
fig = px.choropleth(
    precipitation_by_province,
    geojson=geojson,
    color="avg_precipitation",
    locations="province",
    featureidkey="properties.statnaam",
    color_continuous_scale="Viridis", # Changed color scale for precipitation
    labels={"avg_precipitation": "Average 1h Precipitation (mm)"}
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_layout(
    title_text="Average 1h Precipitation per Province — Netherlands",
    margin={"r":0, "t":40, "l":0, "b":0}
)

fig.show()
